# Six-Mechanism Harmful-Rhetoric Analysis

## Configuration

In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter
import itertools
import logging
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm
warnings.filterwarnings('ignore')
logging.getLogger('pgmpy').setLevel(logging.WARNING)
PROJECT_DIR = Path('/harmful-speech')
ANNOTATION_DIR = PROJECT_DIR / 'harmful_speech_annotations'
SOURCE_SHARD_DIR = Path('/df/broken_up_df')
OUTPUT_DIR = PROJECT_DIR / 'paper_outputs_six_mechanisms'
QWEN_PATH = PROJECT_DIR / 'llm_validation_results/qwen3_14b_predictions.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
OUTER_FOLDS = 5
EDGE_INNER_FOLDS = 3
EDGE_FULL_FOLDS = 5
STRUCTURE_SAMPLE_N = 300000
BACKBONE_CUM_WEIGHT = 0.95
BN_CONSENSUS_MIN_FREQ = 0.8
TEMPORAL_MIN_POSTS = 10
TEMPORAL_NULL_DRAWS = 1000
MECHANISMS = ['boundary_construction', 'threat_construction', 'scapegoating', 'negative_evaluation', 'dehumanization', 'action_orientation']
MECH_COLS = [f'mech_{m}' for m in MECHANISMS]
SHORT = {'mech_boundary_construction': 'Boundary', 'mech_threat_construction': 'Threat', 'mech_scapegoating': 'Scapegoating', 'mech_negative_evaluation': 'Negative evaluation', 'mech_dehumanization': 'Dehumanization', 'mech_action_orientation': 'Action orientation'}
FULL_LABEL = {'mech_boundary_construction': 'Boundary construction', 'mech_threat_construction': 'Threat construction', 'mech_scapegoating': 'Scapegoating', 'mech_negative_evaluation': 'Negative evaluation', 'mech_dehumanization': 'Dehumanization', 'mech_action_orientation': 'Action orientation'}
PASS_REQUIRED = {'group_narrative': ['doc_id', 'platform', 'target_surface', 'boundary_construction', 'threat_construction', 'scapegoating', 'confidence'], 'degrading_language': ['doc_id', 'platform', 'target_surface', 'negative_evaluation', 'dehumanization', 'confidence'], 'harmful_action': ['doc_id', 'platform', 'target_surface', 'action_orientation', 'confidence']}
SOURCE_METADATA_COLUMNS = ['id', 'platform', 'claim', 'claim_index', 'timestamp', 'cluster', 'orig_row']
print('Annotation directory:', ANNOTATION_DIR)
print('Source shard directory:', SOURCE_SHARD_DIR)
print('Output directory:', OUTPUT_DIR)


## Load Annotations

In [ ]:
def shard_from_filename(path, pass_name):
    prefix = pass_name + '__'
    stem = path.stem
    return stem[len(prefix):] if stem.startswith(prefix) else stem

def read_pass(pass_name):
    paths = sorted(ANNOTATION_DIR.glob(f'{pass_name}__*.tsv'))
    if not paths:
        raise FileNotFoundError(f'No TSVs found for {pass_name}')
    frames = []
    for path in paths:
        try:
            df = pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False, on_bad_lines='warn')
        except pd.errors.EmptyDataError:
            continue
        required = PASS_REQUIRED[pass_name]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f'{path.name} missing {missing}')
        df = df[required].copy()
        df['_shard'] = shard_from_filename(path, pass_name)
        df['doc_id'] = df['doc_id'].astype(str).str.strip()
        df['platform'] = df['platform'].astype(str).str.strip()
        df['_doc_occurrence'] = df.groupby(['_shard', 'doc_id'], sort=False).cumcount().astype(int)
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    print(f'{pass_name:20s}: {len(out):,} rows')
    return out
pass_frames = {name: read_pass(name) for name in PASS_REQUIRED}
progress_rows = []
for pass_name, df in pass_frames.items():
    for shard, g in df.groupby('_shard'):
        progress_rows.append({'pass': pass_name, 'shard': shard, 'rows_saved': len(g)})
progress = pd.DataFrame(progress_rows)
display(progress.pivot(index='shard', columns='pass', values='rows_saved').fillna(0).astype(int))


## Build Analysis Dataset

In [ ]:
MECH_COLS = ['boundary_construction', 'threat_construction', 'scapegoating', 'negative_evaluation', 'dehumanization', 'action_orientation']
for col in MECH_COLS:
    annotated_claims[col] = pd.to_numeric(annotated_claims[col], errors='raise').astype(np.int8)
    assert annotated_claims[col].isin([0, 1]).all()
print('Complete three-pass rows:', f'{len(annotated_claims):,}')

def source_path(shard):
    import re
    match = re.fullmatch('part_(\\d+)', str(shard).strip())
    if not match:
        raise ValueError(f'Unexpected shard: {shard!r}')
    part = match.group(1)
    return SOURCE_SHARD_DIR / f'filtered_clustered_df_classified_part_{part}.parquet'
source_frames = []
for shard in sorted(annotated_claims['_shard'].unique()):
    path = source_path(shard)
    print(f'Loading metadata for {shard}:', path)
    if not path.exists():
        raise FileNotFoundError(path)
    src = pd.read_parquet(path, columns=SOURCE_METADATA_COLUMNS).copy()
    src['_shard'] = str(shard)
    src['doc_id'] = src['id'].astype(str).str.strip()
    src['platform_source'] = src['platform'].astype(str).str.strip()
    src['_doc_occurrence'] = src.groupby(['_shard', 'doc_id'], sort=False).cumcount().astype(int)
    source_frames.append(src[['_shard', 'doc_id', '_doc_occurrence', 'platform_source', 'claim', 'claim_index', 'timestamp', 'cluster', 'orig_row']])
source_metadata = pd.concat(source_frames, ignore_index=True)
annotated = annotated_claims.merge(source_metadata, on=JOIN_KEYS, how='left', validate='one_to_one', indicator=True)
matched = annotated['_merge'].eq('both')
platform_match = annotated['platform'] == annotated['platform_source']
print('\nMetadata recovery:', f'{matched.mean():.3%}')
print('Platform agreement among matched rows:', f'{platform_match.loc[matched].mean():.3%}')
annotated = annotated.loc[matched & platform_match].drop(columns='_merge').copy()
print('Rows after metadata validation:', f'{len(annotated):,}')


In [ ]:
RAW_MECHANISMS = ['boundary_construction', 'threat_construction', 'scapegoating', 'negative_evaluation', 'dehumanization', 'action_orientation']
CANONICAL_MECH_COLS = [f'mech_{m}' for m in RAW_MECHANISMS]
print('Mechanism-like columns currently in annotated:')
print([c for c in annotated.columns if any((m in c for m in RAW_MECHANISMS))])
rename_map = {}
for m in RAW_MECHANISMS:
    canonical = f'mech_{m}'
    possible_names = [canonical, m, f'mech_mech_{m}']
    found = [c for c in possible_names if c in annotated.columns]
    if len(found) == 0:
        raise KeyError(f'Could not find column for {m}. Current matching columns: {[c for c in annotated.columns if m in c]}')
    source_col = found[0]
    if source_col != canonical:
        rename_map[source_col] = canonical
if rename_map:
    print('\nRenaming:')
    for old, new in rename_map.items():
        print(f'  {old} -> {new}')
    annotated = annotated.rename(columns=rename_map)
missing = [c for c in CANONICAL_MECH_COLS if c not in annotated.columns]
if missing:
    raise KeyError(f'Still missing mechanism columns: {missing}')
print('\nCanonical mechanism columns ready:')
print(CANONICAL_MECH_COLS)
analysis_cols = ['_shard', 'doc_id', '_doc_occurrence', 'platform'] + CANONICAL_MECH_COLS
engagement_df = annotated[analysis_cols].merge(source_engagement, on=['_shard', 'doc_id', '_doc_occurrence'], how='left', validate='one_to_one')
print('\nEngagement dataframe rows:', f'{len(engagement_df):,}')


In [ ]:
print('Before original-post dedup:', f'{len(annotated):,}')
annotated = annotated.drop_duplicates(subset=['orig_row'], keep='first').reset_index(drop=True).copy()
assert len(annotated) == annotated['orig_row'].nunique()
X_bn = annotated[MECH_COLS].dropna().astype(np.int8).reset_index(drop=True)
print('Final analysis N:', f'{len(X_bn):,}')
print('Mechanisms:', len(MECH_COLS))
print('Possible undirected pairs:', math.comb(len(MECH_COLS), 2))


## Engagement Robustness

In [ ]:
import numpy as np
import pandas as pd
del tqdm
from tqdm import tqdm
from IPython.display import display
ENGAGEMENT_COLS = ['like_count', 'reply_count', 'repost_count', 'view_count']
MECHANISMS = ['mech_boundary_construction', 'mech_threat_construction', 'mech_scapegoating', 'mech_negative_evaluation', 'mech_dehumanization', 'mech_action_orientation']
MECH_LABELS = {'mech_boundary_construction': 'Boundary construction', 'mech_threat_construction': 'Threat construction', 'mech_scapegoating': 'Scapegoating', 'mech_negative_evaluation': 'Negative evaluation', 'mech_dehumanization': 'Dehumanization', 'mech_action_orientation': 'Action orientation'}
PLATFORM_LABELS = {'twitter': 'Twitter/X', 'tiktok': 'TikTok', 'truth_social': 'Truth Social'}
print('=' * 80)
print('RECOVERING ENGAGEMENT METADATA')
print('=' * 80)
engagement_frames = []
for shard in tqdm(sorted(annotated['_shard'].unique()), desc='Source shards'):
    path = source_path(shard)
    available_cols = pd.read_parquet(path).columns.tolist()
    metric_cols_here = [c for c in ENGAGEMENT_COLS if c in available_cols]
    src = pd.read_parquet(path, columns=['id'] + metric_cols_here).copy()
    src['_shard'] = shard
    src['doc_id'] = src['id'].astype(str).str.strip()
    src['_doc_occurrence'] = src.groupby(['_shard', 'doc_id'], sort=False).cumcount().astype(int)
    for c in ENGAGEMENT_COLS:
        if c not in src.columns:
            src[c] = np.nan
    engagement_frames.append(src[['_shard', 'doc_id', '_doc_occurrence'] + ENGAGEMENT_COLS])
source_engagement = pd.concat(engagement_frames, ignore_index=True)
analysis_cols = ['_shard', 'doc_id', '_doc_occurrence', 'platform'] + MECHANISMS
engagement_df = annotated[analysis_cols].merge(source_engagement, on=['_shard', 'doc_id', '_doc_occurrence'], how='left', validate='one_to_one')
print(f'\nAnalyzed rows recovered: {len(engagement_df):,}')
for col in ENGAGEMENT_COLS:
    engagement_df[col] = pd.to_numeric(engagement_df[col], errors='coerce')
    engagement_df.loc[engagement_df[col] < 0, col] = np.nan
availability_rows = []
for platform in sorted(engagement_df['platform'].dropna().unique()):
    platform_df = engagement_df[engagement_df['platform'] == platform]
    for metric in ENGAGEMENT_COLS:
        n_available = platform_df[metric].notna().sum()
        availability_rows.append({'Platform': PLATFORM_LABELS.get(platform, platform), 'Metric': metric, 'N': len(platform_df), 'Available': n_available, 'Available_pct': 100 * n_available / len(platform_df)})
availability_table = pd.DataFrame(availability_rows)
print('\n' + '=' * 80)
print('METADATA AVAILABILITY')
print('=' * 80)
display(availability_table.round(2))
print('\nCalculating platform-specific standardized engagement...')
for metric in tqdm(ENGAGEMENT_COLS, desc='Metrics'):
    log_col = f'log1p_{metric}'
    z_col = f'z_{metric}'
    engagement_df[log_col] = np.log1p(engagement_df[metric])
    engagement_df[z_col] = engagement_df.groupby('platform')[log_col].transform(lambda x: (x - x.mean()) / x.std(ddof=0) if x.notna().sum() > 1 and x.std(ddof=0) > 0 else np.nan)
print('\nCalculating within-platform engagement deciles...')
for metric in tqdm(ENGAGEMENT_COLS, desc='Deciles'):
    z_col = f'z_{metric}'
    decile_col = f'decile_{metric}'
    engagement_df[decile_col] = np.nan
    for platform in engagement_df['platform'].dropna().unique():
        mask = (engagement_df['platform'] == platform) & engagement_df[z_col].notna()
        x = engagement_df.loc[mask, z_col]
        if len(x) == 0:
            continue
        ranks = x.rank(method='first', pct=True)
        deciles = np.ceil(ranks * 10).clip(1, 10)
        engagement_df.loc[mask, decile_col] = deciles.astype(int)
coverage_rows = []
for mech in tqdm(MECHANISMS, desc='Mechanisms'):
    for metric in ENGAGEMENT_COLS:
        decile_col = f'decile_{metric}'
        z_col = f'z_{metric}'
        for platform in sorted(engagement_df['platform'].dropna().unique()):
            sub = engagement_df[(engagement_df['platform'] == platform) & engagement_df[decile_col].notna()]
            positive = sub[sub[mech] == 1]
            if len(positive) == 0:
                continue
            occupied = sorted(positive[decile_col].dropna().astype(int).unique())
            coverage_rows.append({'mechanism': MECH_LABELS[mech], 'metric': metric, 'platform': PLATFORM_LABELS.get(platform, platform), 'n_positive': len(positive), 'deciles_occupied': len(occupied), 'full_decile_coverage': len(occupied) == 10, 'lowest_decile': min(occupied) if occupied else np.nan, 'highest_decile': max(occupied) if occupied else np.nan, 'mean_z': positive[z_col].mean(), 'median_z': positive[z_col].median(), 'bottom_20_pct': 100 * (positive[decile_col] <= 2).mean(), 'middle_60_pct': 100 * positive[decile_col].between(3, 8).mean(), 'top_20_pct': 100 * (positive[decile_col] >= 9).mean()})
engagement_coverage = pd.DataFrame(coverage_rows)
print('\n' + '=' * 80)
print('ENGAGEMENT DISTRIBUTION COVERAGE')
print('=' * 80)
display(engagement_coverage.round(3))
coverage_summary = engagement_coverage.groupby(['mechanism', 'metric']).agg(platform_tests=('full_decile_coverage', 'size'), platforms_full_coverage=('full_decile_coverage', 'sum'), min_deciles_occupied=('deciles_occupied', 'min'), mean_z_min=('mean_z', 'min'), mean_z_max=('mean_z', 'max')).reset_index()
coverage_summary['all_platforms_full_coverage'] = coverage_summary['platform_tests'] == coverage_summary['platforms_full_coverage']
print('\n' + '=' * 80)
print('COMPACT COVERAGE SUMMARY')
print('=' * 80)
display(coverage_summary.round(3))
decile_prevalence_rows = []
for mech in tqdm(MECHANISMS, desc='Decile prevalence'):
    for metric in ENGAGEMENT_COLS:
        decile_col = f'decile_{metric}'
        for platform in sorted(engagement_df['platform'].dropna().unique()):
            sub = engagement_df[(engagement_df['platform'] == platform) & engagement_df[decile_col].notna()]
            if len(sub) == 0:
                continue
            rates = sub.groupby(decile_col)[mech].agg(['mean', 'sum', 'count']).reset_index()
            for _, row in rates.iterrows():
                decile_prevalence_rows.append({'mechanism': MECH_LABELS[mech], 'metric': metric, 'platform': PLATFORM_LABELS.get(platform, platform), 'decile': int(row[decile_col]), 'positive': int(row['sum']), 'n': int(row['count']), 'prevalence_pct': 100 * row['mean']})
decile_prevalence = pd.DataFrame(decile_prevalence_rows)
print('\n' + '=' * 80)
print('MECHANISM PREVALENCE BY ENGAGEMENT DECILE')
print('=' * 80)
display(decile_prevalence.round(3))
appendix_summary = engagement_coverage.groupby('mechanism').agg(tests=('full_decile_coverage', 'size'), full_decile_tests=('full_decile_coverage', 'sum'), min_deciles_occupied=('deciles_occupied', 'min'), mean_standardized_engagement_min=('mean_z', 'min'), mean_standardized_engagement_max=('mean_z', 'max')).reset_index()
appendix_summary['pct_platform_metric_combinations_full_coverage'] = 100 * appendix_summary['full_decile_tests'] / appendix_summary['tests']
print('\n' + '=' * 80)
print('APPENDIX-READY SUMMARY')
print('=' * 80)
display(appendix_summary.round(3))
engagement_coverage.to_csv(OUTPUT_DIR / 'engagement_distribution_coverage.csv', index=False)
decile_prevalence.to_csv(OUTPUT_DIR / 'mechanism_prevalence_by_engagement_decile.csv', index=False)
appendix_summary.to_csv(OUTPUT_DIR / 'engagement_coverage_appendix_summary.csv', index=False)
availability_table.to_csv(OUTPUT_DIR / 'engagement_metadata_availability.csv', index=False)
print('\nSaved engagement robustness outputs.')


In [ ]:
print('\nCalculating platform-specific standardized engagement...')
for metric in tqdm(ENGAGEMENT_COLS, desc='Metrics'):
    log_col = f'log1p_{metric}'
    z_col = f'z_{metric}'
    engagement_df[log_col] = np.log1p(engagement_df[metric])
    engagement_df[z_col] = engagement_df.groupby('platform')[log_col].transform(lambda x: (x - x.mean()) / x.std(ddof=0) if x.notna().sum() > 1 and x.std(ddof=0) > 0 else np.nan)
print('\nCalculating within-platform engagement deciles...')
for metric in tqdm(ENGAGEMENT_COLS, desc='Deciles'):
    z_col = f'z_{metric}'
    decile_col = f'decile_{metric}'
    engagement_df[decile_col] = np.nan
    for platform in engagement_df['platform'].dropna().unique():
        mask = (engagement_df['platform'] == platform) & engagement_df[z_col].notna()
        x = engagement_df.loc[mask, z_col]
        if len(x) == 0:
            continue
        ranks = x.rank(method='first', pct=True)
        deciles = np.ceil(ranks * 10).clip(1, 10)
        engagement_df.loc[mask, decile_col] = deciles.astype(int)
coverage_rows = []
for mech in tqdm(MECHANISMS, desc='Mechanisms'):
    for metric in ENGAGEMENT_COLS:
        decile_col = f'decile_{metric}'
        z_col = f'z_{metric}'
        for platform in sorted(engagement_df['platform'].dropna().unique()):
            sub = engagement_df[(engagement_df['platform'] == platform) & engagement_df[decile_col].notna()]
            positive = sub[sub[mech] == 1]
            if len(positive) == 0:
                continue
            occupied = sorted(positive[decile_col].dropna().astype(int).unique())
            coverage_rows.append({'mechanism': MECH_LABELS[mech], 'metric': metric, 'platform': PLATFORM_LABELS.get(platform, platform), 'n_positive': len(positive), 'deciles_occupied': len(occupied), 'full_decile_coverage': len(occupied) == 10, 'lowest_decile': min(occupied) if occupied else np.nan, 'highest_decile': max(occupied) if occupied else np.nan, 'mean_z': positive[z_col].mean(), 'median_z': positive[z_col].median(), 'bottom_20_pct': 100 * (positive[decile_col] <= 2).mean(), 'middle_60_pct': 100 * positive[decile_col].between(3, 8).mean(), 'top_20_pct': 100 * (positive[decile_col] >= 9).mean()})
engagement_coverage = pd.DataFrame(coverage_rows)
print('\n' + '=' * 80)
print('ENGAGEMENT DISTRIBUTION COVERAGE')
print('=' * 80)
display(engagement_coverage.round(3))
coverage_summary = engagement_coverage.groupby(['mechanism', 'metric']).agg(platform_tests=('full_decile_coverage', 'size'), platforms_full_coverage=('full_decile_coverage', 'sum'), min_deciles_occupied=('deciles_occupied', 'min'), mean_z_min=('mean_z', 'min'), mean_z_max=('mean_z', 'max')).reset_index()
coverage_summary['all_platforms_full_coverage'] = coverage_summary['platform_tests'] == coverage_summary['platforms_full_coverage']
print('\n' + '=' * 80)
print('COMPACT COVERAGE SUMMARY')
print('=' * 80)
display(coverage_summary.round(3))
decile_prevalence_rows = []
for mech in tqdm(MECHANISMS, desc='Decile prevalence'):
    for metric in ENGAGEMENT_COLS:
        decile_col = f'decile_{metric}'
        for platform in sorted(engagement_df['platform'].dropna().unique()):
            sub = engagement_df[(engagement_df['platform'] == platform) & engagement_df[decile_col].notna()]
            if len(sub) == 0:
                continue
            rates = sub.groupby(decile_col)[mech].agg(['mean', 'sum', 'count']).reset_index()
            for _, row in rates.iterrows():
                decile_prevalence_rows.append({'mechanism': MECH_LABELS[mech], 'metric': metric, 'platform': PLATFORM_LABELS.get(platform, platform), 'decile': int(row[decile_col]), 'positive': int(row['sum']), 'n': int(row['count']), 'prevalence_pct': 100 * row['mean']})
decile_prevalence = pd.DataFrame(decile_prevalence_rows)
print('\n' + '=' * 80)
print('MECHANISM PREVALENCE BY ENGAGEMENT DECILE')
print('=' * 80)
display(decile_prevalence.round(3))
appendix_summary = engagement_coverage.groupby('mechanism').agg(tests=('full_decile_coverage', 'size'), full_decile_tests=('full_decile_coverage', 'sum'), min_deciles_occupied=('deciles_occupied', 'min'), mean_standardized_engagement_min=('mean_z', 'min'), mean_standardized_engagement_max=('mean_z', 'max')).reset_index()
appendix_summary['pct_platform_metric_combinations_full_coverage'] = 100 * appendix_summary['full_decile_tests'] / appendix_summary['tests']
print('\n' + '=' * 80)
print('APPENDIX-READY SUMMARY')
print('=' * 80)
display(appendix_summary.round(3))
engagement_coverage.to_csv(OUTPUT_DIR / 'engagement_distribution_coverage.csv', index=False)
decile_prevalence.to_csv(OUTPUT_DIR / 'mechanism_prevalence_by_engagement_decile.csv', index=False)
appendix_summary.to_csv(OUTPUT_DIR / 'engagement_coverage_appendix_summary.csv', index=False)
availability_table.to_csv(OUTPUT_DIR / 'engagement_metadata_availability.csv', index=False)
print('\nSaved engagement robustness outputs.')


In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
JOIN_KEYS = ['_shard', 'doc_id', '_doc_occurrence']
print('=' * 80)
print('RECOVERING USERNAME')
print('=' * 80)
print('model_df rows:', f'{len(model_df):,}')
if 'username' in model_df.columns:
    print('username already present in model_df.')
else:
    username_parts = []
    shards_needed = model_df['_shard'].dropna().astype(str).unique().tolist()
    print('Shards needed:', shards_needed)
    for shard in tqdm(shards_needed, desc='Loading usernames'):
        path = source_path(shard)
        src = pd.read_parquet(path, columns=['id', 'username'])
        src = src.rename(columns={'id': 'doc_id'})
        src['_shard'] = shard
        src['_doc_occurrence'] = src.groupby('doc_id', dropna=False).cumcount()
        username_parts.append(src[['_shard', 'doc_id', '_doc_occurrence', 'username']])
    username_source = pd.concat(username_parts, ignore_index=True)
    print('\nSource username rows:', f'{len(username_source):,}')
    print('Duplicate source join keys:', username_source.duplicated(JOIN_KEYS).sum())
    n_before = len(model_df)
    model_df = model_df.merge(username_source, on=JOIN_KEYS, how='left', validate='one_to_one')
    n_after = len(model_df)
    print('\nRows before merge:', f'{n_before:,}')
    print('Rows after merge: ', f'{n_after:,}')
    assert n_before == n_after, 'Username merge changed row count.'
model_df['username'] = model_df['username'].astype('string')
model_df.loc[model_df['username'].isna() | (model_df['username'].str.strip() == ''), 'username'] = pd.NA
print('\n' + '=' * 80)
print('USERNAME RECOVERY SUMMARY')
print('=' * 80)
print('Rows with username:', f"{model_df['username'].notna().sum():,}")
print('Rows missing username:', f"{model_df['username'].isna().sum():,}")
print('Unique usernames:', f"{model_df['username'].nunique():,}")
print('Unique platform-specific users:', f"{model_df[['platform', 'username']].dropna().drop_duplicates().shape[0]:,}")
print('\nUsername ready.')


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from IPython.display import display
del tqdm
from tqdm import tqdm
MECH_COLS_ANALYSIS = ['mech_boundary_construction', 'mech_threat_construction', 'mech_scapegoating', 'mech_negative_evaluation', 'mech_dehumanization', 'mech_action_orientation']
MECH_LABELS = {'mech_boundary_construction': 'Boundary construction', 'mech_threat_construction': 'Threat construction', 'mech_scapegoating': 'Scapegoating', 'mech_negative_evaluation': 'Negative evaluation', 'mech_dehumanization': 'Dehumanization', 'mech_action_orientation': 'Action orientation'}
ENGAGEMENT_METRICS = ['like_count', 'reply_count', 'repost_count', 'view_count']
required_base_cols = ['platform', 'username', 'month', 'cluster'] + MECH_COLS_ANALYSIS
missing = [c for c in required_base_cols if c not in model_df.columns]
if missing:
    raise KeyError(f'model_df is missing required columns: {missing}')
print('=' * 80)
print('AUTHOR-ADJUSTED ENGAGEMENT ROBUSTNESS')
print('=' * 80)
print('Rows:', f'{len(model_df):,}')
model_df['username'] = model_df['username'].astype('string')
model_df.loc[model_df['username'].isna() | (model_df['username'].str.strip() == ''), 'username'] = pd.NA
print('Rows with username:', f"{model_df['username'].notna().sum():,}")
print('Unique platform-user identities:', f"{model_df[['platform', 'username']].dropna().drop_duplicates().shape[0]:,}")
for metric in ENGAGEMENT_METRICS:
    if metric not in model_df.columns:
        print(f'Missing raw metric: {metric}')
        continue
    model_df[metric] = pd.to_numeric(model_df[metric], errors='coerce').astype('float64')
    model_df.loc[model_df[metric] < 0, metric] = np.nan
    log_col = f'log_{metric}'
    model_df[log_col] = np.log1p(model_df[metric]).astype('float64')
for metric in ENGAGEMENT_METRICS:
    z_col = f'z_{metric}'
    if z_col not in model_df.columns:
        continue
    model_df[z_col] = pd.to_numeric(model_df[z_col], errors='coerce').astype('float64')
AUTHOR_KEYS = ['platform', 'username']
print('\n' + '=' * 80)
print('COMPUTING LEAVE-ONE-OUT AUTHOR BASELINES')
print('=' * 80)
for metric in tqdm(ENGAGEMENT_METRICS, desc='Author baselines'):
    log_col = f'log_{metric}'
    if log_col not in model_df.columns:
        continue
    loo_col = f'author_loo_{log_col}'
    n_col = f'author_n_{metric}'
    valid = model_df['username'].notna() & model_df[log_col].notna()
    temp = model_df.loc[valid, AUTHOR_KEYS + [log_col]].copy()
    grouped = temp.groupby(AUTHOR_KEYS, dropna=False)[log_col].agg(author_sum='sum', author_n='count').reset_index()
    grouped['author_sum'] = pd.to_numeric(grouped['author_sum'], errors='coerce').astype('float64')
    grouped['author_n'] = pd.to_numeric(grouped['author_n'], errors='coerce').astype('float64')
    lookup = model_df[AUTHOR_KEYS].merge(grouped, on=AUTHOR_KEYS, how='left', validate='many_to_one')
    author_sum = pd.to_numeric(lookup['author_sum'], errors='coerce').to_numpy(dtype=np.float64, na_value=np.nan)
    author_n = pd.to_numeric(lookup['author_n'], errors='coerce').to_numpy(dtype=np.float64, na_value=np.nan)
    current = pd.to_numeric(model_df[log_col], errors='coerce').to_numpy(dtype=np.float64, na_value=np.nan)
    loo = np.full(len(model_df), np.nan, dtype=np.float64)
    usable = np.isfinite(author_sum) & np.isfinite(author_n) & np.isfinite(current) & (author_n > 1)
    loo[usable] = (author_sum[usable] - current[usable]) / (author_n[usable] - 1.0)
    model_df[loo_col] = loo
    model_df[n_col] = author_n
    print(f'\n{metric}')
    print('  valid engagement rows:', f'{valid.sum():,}')
    print('  posts with LOO author baseline:', f'{np.isfinite(loo).sum():,}')
    print('  platform-users with >=2 valid posts:', f"{(grouped['author_n'] > 1).sum():,}")
for metric in ENGAGEMENT_METRICS:
    loo_col = f'author_loo_log_{metric}'
    z_loo_col = f'z_author_loo_log_{metric}'
    if loo_col not in model_df.columns:
        continue

    def zscore_safe(x):
        x = pd.to_numeric(x, errors='coerce').astype('float64')
        sd = x.std(ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(np.nan, index=x.index, dtype='float64')
        return ((x - x.mean()) / sd).astype('float64')
    model_df[z_loo_col] = model_df.groupby('platform')[loo_col].transform(zscore_safe)
for mech in MECH_COLS_ANALYSIS:
    model_df[mech] = pd.to_numeric(model_df[mech], errors='coerce').astype('float64')

def fit_engagement_model(d, outcome, extra_controls=None):
    if extra_controls is None:
        extra_controls = []
    X_mech = d[MECH_COLS_ANALYSIS].apply(pd.to_numeric, errors='coerce').astype('float64')
    X_platform = pd.get_dummies(d['platform'].astype(str), prefix='platform', drop_first=True, dtype=float)
    X_month = pd.get_dummies(d['month'].astype(str), prefix='month', drop_first=True, dtype=float)
    parts = [X_mech, X_platform, X_month]
    if extra_controls:
        X_extra = d[extra_controls].apply(pd.to_numeric, errors='coerce').astype('float64')
        parts.append(X_extra)
    X = pd.concat(parts, axis=1)
    X = sm.add_constant(X, has_constant='add')
    X = X.astype('float64')
    y = pd.to_numeric(d[outcome], errors='coerce').to_numpy(dtype=np.float64, na_value=np.nan)
    X_np = X.to_numpy(dtype=np.float64, na_value=np.nan)
    groups = d['cluster'].astype(str).to_numpy()
    if not np.isfinite(X_np).all():
        raise ValueError(f'Non-finite values remain in X for {outcome}')
    if not np.isfinite(y).all():
        raise ValueError(f'Non-finite values remain in y for {outcome}')
    model = sm.OLS(y, X_np).fit(cov_type='cluster', cov_kwds={'groups': groups})
    names = X.columns.tolist()
    params = pd.Series(model.params, index=names)
    ses = pd.Series(model.bse, index=names)
    pvals = pd.Series(model.pvalues, index=names)
    conf = pd.DataFrame(model.conf_int(), index=names, columns=['ci_low', 'ci_high'])
    return (model, params, ses, pvals, conf)
results = []
for metric in tqdm(ENGAGEMENT_METRICS, desc='Engagement models'):
    outcome = f'z_{metric}'
    author_control = f'z_author_loo_log_{metric}'
    if outcome not in model_df.columns:
        print(f'\nSkipping {metric}: {outcome} missing')
        continue
    if author_control not in model_df.columns:
        print(f'\nSkipping {metric}: {author_control} missing')
        continue
    base_required = [outcome, 'platform', 'month', 'cluster'] + MECH_COLS_ANALYSIS
    d_base = model_df[base_required].replace([np.inf, -np.inf], np.nan).dropna().copy()
    print('\n' + '=' * 80)
    print(metric)
    print('=' * 80)
    print('Base model N:', f'{len(d_base):,}')
    print('Base narratives:', f"{d_base['cluster'].nunique():,}")
    model_base, params_base, ses_base, pvals_base, conf_base = fit_engagement_model(d_base, outcome)
    for mech in MECH_COLS_ANALYSIS:
        results.append({'metric': metric, 'model': 'Base', 'mechanism': mech, 'mechanism_label': MECH_LABELS[mech], 'n': len(d_base), 'n_narratives': d_base['cluster'].nunique(), 'coef_sd': float(params_base[mech]), 'se': float(ses_base[mech]), 'ci_low': float(conf_base.loc[mech, 'ci_low']), 'ci_high': float(conf_base.loc[mech, 'ci_high']), 'p': float(pvals_base[mech])})
    author_required = [outcome, 'platform', 'month', 'cluster', author_control] + MECH_COLS_ANALYSIS
    d_author = model_df[author_required].replace([np.inf, -np.inf], np.nan).dropna().copy()
    print('Author-adjusted N:', f'{len(d_author):,}')
    print('Author-adjusted narratives:', f"{d_author['cluster'].nunique():,}")
    if len(d_author) == 0:
        print('No author-adjusted observations; skipping.')
        continue
    model_author, params_author, ses_author, pvals_author, conf_author = fit_engagement_model(d_author, outcome, extra_controls=[author_control])
    for mech in MECH_COLS_ANALYSIS:
        results.append({'metric': metric, 'model': 'Author-adjusted', 'mechanism': mech, 'mechanism_label': MECH_LABELS[mech], 'n': len(d_author), 'n_narratives': d_author['cluster'].nunique(), 'coef_sd': float(params_author[mech]), 'se': float(ses_author[mech]), 'ci_low': float(conf_author.loc[mech, 'ci_low']), 'ci_high': float(conf_author.loc[mech, 'ci_high']), 'p': float(pvals_author[mech])})
author_engagement_results = pd.DataFrame(results)
if len(author_engagement_results) == 0:
    raise ValueError('No regression results produced.')
author_engagement_results['q_fdr'] = multipletests(author_engagement_results['p'], method='fdr_bh')[1]
print('\n' + '=' * 80)
print('ENGAGEMENT RESULTS: BASE VS AUTHOR-ADJUSTED')
print('=' * 80)
display(author_engagement_results[['metric', 'model', 'mechanism_label', 'coef_sd', 'se', 'ci_low', 'ci_high', 'p', 'q_fdr', 'n', 'n_narratives']].sort_values(['metric', 'mechanism_label', 'model']).round(4))
coef_compare = author_engagement_results.pivot_table(index=['metric', 'mechanism_label'], columns='model', values='coef_sd').reset_index()
if 'Base' in coef_compare.columns and 'Author-adjusted' in coef_compare.columns:
    coef_compare['change_after_author_control'] = coef_compare['Author-adjusted'] - coef_compare['Base']
    coef_compare['abs_change'] = np.abs(coef_compare['change_after_author_control'])
print('\n' + '=' * 80)
print('COEFFICIENT CHANGE AFTER AUTHOR CONTROL')
print('=' * 80)
display(coef_compare.sort_values('abs_change', ascending=False).round(4))
author_only = author_engagement_results[author_engagement_results['model'] == 'Author-adjusted'].copy()
appendix_author_table = author_only.pivot(index='mechanism_label', columns='metric', values='coef_sd').reindex(['Boundary construction', 'Threat construction', 'Scapegoating', 'Negative evaluation', 'Dehumanization', 'Action orientation'])
print('\n' + '=' * 80)
print('APPENDIX — AUTHOR-ADJUSTED ASSOCIATIONS')
print('=' * 80)
display(appendix_author_table.round(3))
author_effect_summary = author_only.groupby('mechanism_label').agg(min_effect_sd=('coef_sd', 'min'), max_effect_sd=('coef_sd', 'max'), max_abs_effect_sd=('coef_sd', lambda x: np.abs(x).max())).reset_index().sort_values('max_abs_effect_sd', ascending=False)
print('\n' + '=' * 80)
print('MAXIMUM AUTHOR-ADJUSTED ASSOCIATION')
print('=' * 80)
display(author_effect_summary.round(4))
author_engagement_results.to_csv(OUTPUT_DIR / 'engagement_base_vs_author_adjusted.csv', index=False)
coef_compare.to_csv(OUTPUT_DIR / 'engagement_author_control_coefficient_changes.csv', index=False)
appendix_author_table.to_csv(OUTPUT_DIR / 'engagement_author_adjusted_appendix_table.csv')
author_effect_summary.to_csv(OUTPUT_DIR / 'engagement_author_adjusted_effect_summary.csv', index=False)
print('\nSaved.')


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
plot_df = decile_prevalence[decile_prevalence['metric'] == 'like_count'].copy()
MECH_ORDER = ['Boundary construction', 'Threat construction', 'Scapegoating', 'Negative evaluation', 'Dehumanization', 'Action orientation']
PLATFORM_ORDER = ['TikTok', 'Twitter/X', 'Truth Social']
fig, axes = plt.subplots(2, 3, figsize=(10.5, 6.5), sharex=True)
axes = axes.flatten()
for ax, mech in zip(axes, MECH_ORDER):
    sub = plot_df[plot_df['mechanism'] == mech]
    for platform in PLATFORM_ORDER:
        p = sub[sub['platform'] == platform].sort_values('decile')
        if len(p) == 0:
            continue
        ax.plot(p['decile'], p['prevalence_pct'], marker='o', linewidth=1.8, markersize=4, label=platform)
    ax.set_title(mech, fontsize=11, fontweight='bold')
    ax.set_xticks(range(1, 11))
    ax.grid(axis='y', alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)
fig.supxlabel('Within-platform like-count decile', fontsize=11)
fig.supylabel('Posts containing mechanism (%)', fontsize=11)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.01), ncol=3, frameon=False)
fig.suptitle('Harmful-rhetoric mechanisms across the engagement distribution', fontsize=13, fontweight='bold', y=1.06)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'mechanism_prevalence_by_like_decile.pdf', bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'mechanism_prevalence_by_like_decile.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
DEDUP_COLS = ['platform', 'doc_id', 'like_count', 'reply_count']
print('Before dedup:', f'{len(engagement_df):,}')
engagement_df = engagement_df.drop_duplicates(subset=DEDUP_COLS, keep='first').reset_index(drop=True).copy()
print('After dedup: ', f'{len(engagement_df):,}')
print('Duplicates remaining:', engagement_df.duplicated(DEDUP_COLS).sum())
assert not engagement_df.duplicated(DEDUP_COLS).any()


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from IPython.display import display
from tqdm.auto import tqdm
MECH_COLS_ANALYSIS = ['mech_boundary_construction', 'mech_threat_construction', 'mech_scapegoating', 'mech_negative_evaluation', 'mech_dehumanization', 'mech_action_orientation']
MECH_LABELS = {'mech_boundary_construction': 'Boundary construction', 'mech_threat_construction': 'Threat construction', 'mech_scapegoating': 'Scapegoating', 'mech_negative_evaluation': 'Negative evaluation', 'mech_dehumanization': 'Dehumanization', 'mech_action_orientation': 'Action orientation'}
ENGAGEMENT_METRICS = ['like_count', 'reply_count', 'repost_count', 'view_count']
JOIN_KEYS = ['_shard', 'doc_id', '_doc_occurrence']
print('=' * 80)
print('BUILDING ENGAGEMENT REGRESSION DATA')
print('=' * 80)
print('engagement_df rows:', f'{len(engagement_df):,}')
meta_cols = JOIN_KEYS + ['timestamp', 'cluster']
meta = annotated[meta_cols].drop_duplicates(subset=JOIN_KEYS, keep='first').copy()
print('metadata rows:', f'{len(meta):,}')
print('duplicate metadata join keys:', meta.duplicated(JOIN_KEYS).sum())
model_df = engagement_df.merge(meta, on=JOIN_KEYS, how='left', validate='one_to_one')
print('model_df rows:', f'{len(model_df):,}')
assert len(model_df) == len(engagement_df)
model_df['timestamp'] = pd.to_datetime(model_df['timestamp'], errors='coerce', utc=True)
model_df['month'] = model_df['timestamp'].dt.strftime('%Y-%m')
for mech in MECH_COLS_ANALYSIS:
    model_df[mech] = pd.to_numeric(model_df[mech], errors='coerce').astype('float64')
for metric in ENGAGEMENT_METRICS:
    z_col = f'z_{metric}'
    if z_col in model_df.columns:
        model_df[z_col] = pd.to_numeric(model_df[z_col], errors='coerce').astype('float64')
model_df['platform'] = model_df['platform'].astype(str)
model_df['month'] = model_df['month'].astype(str)
model_df['cluster'] = model_df['cluster'].astype(str)
print('\n' + '=' * 80)
print('ENGAGEMENT REGRESSION ROBUSTNESS')
print('=' * 80)
print(f'Rows available: {len(model_df):,}')
print('\nPlatforms:')
print(model_df['platform'].value_counts())
print('\nMonths:')
print(model_df['month'].value_counts().sort_index())
results = []
for metric in tqdm(ENGAGEMENT_METRICS, desc='Engagement models'):
    outcome = f'z_{metric}'
    if outcome not in model_df.columns:
        print(f'\nSkipping {metric}: {outcome} not found')
        continue
    required = [outcome, 'platform', 'month', 'cluster'] + MECH_COLS_ANALYSIS
    d = model_df[required].replace({'nan': np.nan, 'NaT': np.nan, 'None': np.nan}).dropna().copy()
    print('\n' + '-' * 80)
    print(metric)
    print(f'N = {len(d):,}')
    print('N narratives =', f"{d['cluster'].nunique():,}")
    print('-' * 80)
    if len(d) == 0:
        print('No complete rows; skipping.')
        continue
    X_mech = d[MECH_COLS_ANALYSIS].astype(np.float64).copy()
    X_platform = pd.get_dummies(d['platform'], prefix='platform', drop_first=True, dtype=float)
    X_month = pd.get_dummies(d['month'], prefix='month', drop_first=True, dtype=float)
    X = pd.concat([X_mech, X_platform, X_month], axis=1)
    X = sm.add_constant(X, has_constant='add')
    X = X.astype(np.float64)
    X_np = X.to_numpy(dtype=np.float64)
    y = d[outcome].astype(np.float64).to_numpy()
    groups = d['cluster'].astype(str).to_numpy()
    assert X_np.dtype == np.float64
    assert y.dtype == np.float64
    assert np.isfinite(X_np).all(), f'Non-finite values found in X for {metric}'
    assert np.isfinite(y).all(), f'Non-finite values found in y for {metric}'
    assert len(y) == X_np.shape[0]
    assert len(groups) == len(y)
    model = sm.OLS(y, X_np).fit(cov_type='cluster', cov_kwds={'groups': groups})
    param_names = X.columns.tolist()
    params = pd.Series(model.params, index=param_names)
    ses = pd.Series(model.bse, index=param_names)
    pvals = pd.Series(model.pvalues, index=param_names)
    conf = pd.DataFrame(model.conf_int(), index=param_names, columns=['ci_low', 'ci_high'])
    for mech in MECH_COLS_ANALYSIS:
        results.append({'metric': metric, 'mechanism': mech, 'mechanism_label': MECH_LABELS[mech], 'n': len(d), 'n_narratives': d['cluster'].nunique(), 'coef_sd': float(params[mech]), 'se': float(ses[mech]), 'ci_low': float(conf.loc[mech, 'ci_low']), 'ci_high': float(conf.loc[mech, 'ci_high']), 'p': float(pvals[mech])})
engagement_regression_results = pd.DataFrame(results)
if len(engagement_regression_results) == 0:
    raise ValueError('No engagement regression results were produced.')
engagement_regression_results['q_fdr'] = multipletests(engagement_regression_results['p'], method='fdr_bh')[1]
print('\n' + '=' * 80)
print('ADJUSTED ASSOCIATIONS WITH ENGAGEMENT')
print('=' * 80)
display(engagement_regression_results[['metric', 'mechanism_label', 'coef_sd', 'se', 'ci_low', 'ci_high', 'p', 'q_fdr', 'n', 'n_narratives']].sort_values(['metric', 'coef_sd'], ascending=[True, False]).round(4))
effect_summary = engagement_regression_results.groupby('mechanism_label').agg(min_effect_sd=('coef_sd', 'min'), max_effect_sd=('coef_sd', 'max'), max_abs_effect_sd=('coef_sd', lambda x: np.abs(x).max())).reset_index().sort_values('max_abs_effect_sd', ascending=False)
print('\n' + '=' * 80)
print('MAXIMUM ADJUSTED ENGAGEMENT ASSOCIATION')
print('=' * 80)
display(effect_summary.round(4))
appendix_regression_table = engagement_regression_results.pivot(index='mechanism_label', columns='metric', values='coef_sd').reindex(['Boundary construction', 'Threat construction', 'Scapegoating', 'Negative evaluation', 'Dehumanization', 'Action orientation'])
print('\n' + '=' * 80)
print('APPENDIX — STANDARDIZED ADJUSTED ASSOCIATIONS')
print('=' * 80)
display(appendix_regression_table.round(3))
engagement_regression_results.to_csv(OUTPUT_DIR / 'adjusted_engagement_mechanism_associations.csv', index=False)
effect_summary.to_csv(OUTPUT_DIR / 'adjusted_engagement_effect_summary.csv', index=False)
appendix_regression_table.to_csv(OUTPUT_DIR / 'adjusted_engagement_mechanism_coefficients.csv')
print('\nSaved engagement regression outputs.')


## Descriptive Statistics

In [ ]:
prevalence = X_bn.mean().rename('prevalence').to_frame()
prevalence['label'] = prevalence.index.map(FULL_LABEL)
display(prevalence.sort_values('prevalence', ascending=False).round(4))
corr = X_bn.corr()
display(corr.round(3))

def cond_prob(df, A, B):
    sub = df.loc[df[A] == 1]
    return np.nan if len(sub) == 0 else float(sub[B].mean())

def jaccard(df, A, B):
    both = ((df[A] == 1) & (df[B] == 1)).sum()
    either = ((df[A] == 1) | (df[B] == 1)).sum()
    return np.nan if either == 0 else both / either
pair_rows = []
for A, B in combinations(MECH_COLS, 2):
    pair_rows.append({'A': A, 'B': B, 'A_label': SHORT[A], 'B_label': SHORT[B], 'P_B_given_A': cond_prob(X_bn, A, B), 'P_A_given_B': cond_prob(X_bn, B, A), 'jaccard': jaccard(X_bn, A, B), 'phi': float(X_bn[[A, B]].corr().iloc[0, 1])})
pairwise_association = pd.DataFrame(pair_rows).sort_values('jaccard', ascending=False).reset_index(drop=True)
display(pairwise_association.round(3))


## Theory Structures

In [ ]:
def canon(a, b):
    return tuple(sorted((a, b)))
B = 'mech_boundary_construction'
T = 'mech_threat_construction'
D = 'mech_dehumanization'
A = 'mech_action_orientation'
THEORY_PAIRS = {'Moral-exclusion': {canon(B, D), canon(D, A)}, 'Threat-mediated': {canon(B, T), canon(T, A)}, 'Integrated': {canon(B, D), canon(D, A), canon(B, T), canon(T, A)}}
ALL_PAIRS = [canon(a, b) for a, b in combinations(MECH_COLS, 2)]
INDEPENDENT_PAIRS = set()
SATURATED_PAIRS = set(ALL_PAIRS)
outer_cv = KFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
OUTER_SPLITS = list(outer_cv.split(X_bn))
for theory, pairs in THEORY_PAIRS.items():
    print(f'{theory:20s}: {len(pairs)} edges')
print('All possible edges:', len(ALL_PAIRS))


## Structural Scoring

In [ ]:
def fit_intercept_probability(y):
    return np.clip(np.mean(y), 1e-08, 1 - 1e-08)

def fit_logistic(X_train, y_train):
    clf = LogisticRegression(penalty='l2', C=1000000.0, solver='lbfgs', max_iter=1000, tol=1e-06)
    clf.fit(X_train, y_train)
    return clf

def evaluate_parent_set(train_df, test_df, outcome, predictors):
    y_train = train_df[outcome].astype(int).to_numpy()
    y_test = test_df[outcome].astype(int).to_numpy()
    if len(predictors) == 0 or np.unique(y_train).size < 2:
        prob = np.repeat(fit_intercept_probability(y_train), len(y_test))
    else:
        X_train = train_df[list(predictors)].astype(np.float32).to_numpy()
        X_test = test_df[list(predictors)].astype(np.float32).to_numpy()
        clf = fit_logistic(X_train, y_train)
        prob = clf.predict_proba(X_test)[:, 1]
    prob = np.clip(prob, 1e-08, 1 - 1e-08)
    ll = y_test * np.log(prob) + (1 - y_test) * np.log(1 - prob)
    auc = roc_auc_score(y_test, prob) if np.unique(y_test).size == 2 else np.nan
    return {'ll_sum': float(ll.sum()), 'll_mean': float(ll.mean()), 'log_loss': float(log_loss(y_test, prob, labels=[0, 1])), 'auc': float(auc) if np.isfinite(auc) else np.nan, 'n_test': len(y_test)}

def graph_neighbors(pairs):
    nbrs = {m: [] for m in MECH_COLS}
    for a, b in pairs:
        nbrs[a].append(b)
        nbrs[b].append(a)
    return {m: sorted(v) for m, v in nbrs.items()}

def evaluate_graph(train_df, test_df, pairs):
    nbrs = graph_neighbors(pairs)
    rows = []
    for target in MECH_COLS:
        result = evaluate_parent_set(train_df, test_df, target, nbrs[target])
        rows.append(result)
    node = pd.DataFrame(rows)
    total_ll = node['ll_sum'].sum()
    total_n = node['n_test'].sum()
    return {'mean_conditional_ll': total_ll / total_n, 'mean_log_loss': -total_ll / total_n, 'mean_auc': node['auc'].mean(), 'n_pairs': len(pairs)}


In [ ]:
def stratified_structure_sample(train_df, n=STRUCTURE_SAMPLE_N, random_state=RANDOM_STATE):
    if len(train_df) <= n:
        return train_df.copy().reset_index(drop=True)
    work = train_df.copy()
    bits = work[MECH_COLS].to_numpy(dtype=np.int8)
    powers = 1 << np.arange(len(MECH_COLS), dtype=np.int64)
    work['_pattern'] = bits @ powers
    counts = work['_pattern'].value_counts()
    targets = (counts / counts.sum() * n).round().astype(int).clip(lower=1)
    rng = np.random.default_rng(random_state)
    parts = []
    for pattern, target_n in targets.items():
        group = work.loc[work['_pattern'] == pattern]
        parts.append(group.sample(n=min(int(target_n), len(group)), replace=False, random_state=int(rng.integers(0, 2 ** 31 - 1))))
    out = pd.concat(parts, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=random_state)
    return out.drop(columns='_pattern').reset_index(drop=True)

def directed_unique_weights(df, n_folds, random_state):
    rows = []
    for target in tqdm(MECH_COLS):
        predictors = [m for m in MECH_COLS if m != target]
        y = df[target].astype(int).to_numpy()
        cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
        values = {source: [] for source in predictors}
        for tr, va in tqdm(cv.split(df, y)):
            train = df.iloc[tr]
            valid = df.iloc[va]
            full = evaluate_parent_set(train, valid, target, predictors)
            for source in tqdm(predictors):
                reduced_predictors = [m for m in predictors if m != source]
                reduced = evaluate_parent_set(train, valid, target, reduced_predictors)
                values[source].append(reduced['log_loss'] - full['log_loss'])
        for source, deltas in values.items():
            deltas = np.asarray(deltas)
            rows.append({'source': source, 'target': target, 'delta_log_loss': deltas.mean(), 'sd_delta_log_loss': deltas.std(ddof=1), 'positive_folds': int((deltas > 0).sum())})
    return pd.DataFrame(rows)

def pair_weights_from_directed(directed):
    rows = []
    for a, b in combinations(MECH_COLS, 2):
        ab = directed.loc[(directed['source'] == a) & (directed['target'] == b), 'delta_log_loss'].iloc[0]
        ba = directed.loc[(directed['source'] == b) & (directed['target'] == a), 'delta_log_loss'].iloc[0]
        pair = canon(a, b)
        rows.append({'A': pair[0], 'B': pair[1], 'A_to_B': ab if pair[0] == a else ba, 'B_to_A': ba if pair[0] == a else ab, 'pair_weight': max(0.0, (ab + ba) / 2)})
    out = pd.DataFrame(rows).sort_values('pair_weight', ascending=False).reset_index(drop=True)
    total = out['pair_weight'].sum()
    out['weight_share'] = out['pair_weight'] / total if total > 0 else 0
    out['cumulative_share'] = out['weight_share'].cumsum()
    return out

def select_backbone(pair_weights, threshold=BACKBONE_CUM_WEIGHT):
    selected = set()
    for row in pair_weights.loc[pair_weights['pair_weight'] > 0].itertuples():
        selected.add(canon(row.A, row.B))
        if row.cumulative_share >= threshold:
            break
    return selected
from pgmpy.estimators import HillClimbSearch
try:
    from pgmpy.estimators import BIC
    BIC_CLASS = BIC
except ImportError:
    from pgmpy.estimators import BicScore
    BIC_CLASS = BicScore

def learn_bn_skeleton(df):
    model = HillClimbSearch(df).estimate(scoring_method=BIC_CLASS(df), show_progress=False)
    return {canon(u, v) for u, v in list(model.edges())}


## Held-Out Structural Comparison

In [ ]:
comparison_rows = []
backbone_fold_pairs = {}
bn_fold_pairs = {}
for fold, (train_idx, test_idx) in tqdm(enumerate(OUTER_SPLITS, start=1), total=OUTER_FOLDS, desc='Outer structural CV'):
    train_df = X_bn.iloc[train_idx].reset_index(drop=True)
    test_df = X_bn.iloc[test_idx].reset_index(drop=True)
    structure_train = stratified_structure_sample(train_df, n=STRUCTURE_SAMPLE_N, random_state=RANDOM_STATE + fold)
    directed = directed_unique_weights(structure_train, n_folds=EDGE_INNER_FOLDS, random_state=RANDOM_STATE + fold)
    weights = pair_weights_from_directed(directed)
    backbone = select_backbone(weights)
    bn = learn_bn_skeleton(structure_train)
    backbone_fold_pairs[fold] = backbone
    bn_fold_pairs[fold] = bn
    models = {'independent': INDEPENDENT_PAIRS, 'moral_exclusion_theory': THEORY_PAIRS['Moral-exclusion'], 'threat_theory': THEORY_PAIRS['Threat-mediated'], 'integrated_theory': THEORY_PAIRS['Integrated'], 'empirical_backbone_95': backbone, 'bn_skeleton': bn, 'saturated': SATURATED_PAIRS}
    for name, pairs in models.items():
        result = evaluate_graph(train_df, test_df, pairs)
        comparison_rows.append({'fold': fold, 'model': name, **result})
fair_structure_comparison = pd.DataFrame(comparison_rows)
fair_structure_summary = fair_structure_comparison.groupby('model').agg(mean_conditional_ll=('mean_conditional_ll', 'mean'), sd_conditional_ll=('mean_conditional_ll', 'std'), mean_log_loss=('mean_log_loss', 'mean'), mean_auc=('mean_auc', 'mean'), mean_pairs=('n_pairs', 'mean')).sort_values('mean_conditional_ll', ascending=False)
display(fair_structure_summary.round(6))
wide = fair_structure_comparison.pivot(index='fold', columns='model', values='mean_conditional_ll')
denom = wide['saturated'] - wide['independent']
recovery_rows = []
for name in wide.columns:
    if name in {'independent', 'saturated'}:
        continue
    recovery_rows.append({'model': name, 'gain_recovered_pct': 100 * ((wide[name] - wide['independent']) / denom).mean()})
recovery_table = pd.DataFrame(recovery_rows).sort_values('gain_recovered_pct', ascending=False)
display(recovery_table.round(2))


## Equal-Edge Theory Comparison

In [ ]:
node_cache = {}
for fold_i, (train_idx, test_idx) in enumerate(tqdm(OUTER_SPLITS, desc='Caching node models')):
    train = X_bn.iloc[train_idx].reset_index(drop=True)
    test = X_bn.iloc[test_idx].reset_index(drop=True)
    for target in MECH_COLS:
        available = [m for m in MECH_COLS if m != target]
        for k in range(len(available) + 1):
            for subset in combinations(available, k):
                result = evaluate_parent_set(train, test, target, list(subset))
                node_cache[fold_i, target, tuple(sorted(subset))] = result['ll_mean']

def graph_cll(pairs):
    nbrs = graph_neighbors(pairs)
    fold_scores = []
    for fold_i in range(len(OUTER_SPLITS)):
        node_scores = []
        for target in MECH_COLS:
            node_scores.append(node_cache[fold_i, target, tuple(sorted(nbrs[target]))])
        fold_scores.append(np.mean(node_scores))
    return float(np.mean(fold_scores))
cll_ind = graph_cll(INDEPENDENT_PAIRS)
cll_sat = graph_cll(SATURATED_PAIRS)
print('Reference models')
print('----------------')
print(f'Independent CLL: {cll_ind:.8f}')
print(f'Saturated   CLL: {cll_sat:.8f}')

def saturated_gain(cll):
    return (cll - cll_ind) / (cll_sat - cll_ind)
budget_results = {}
for k in sorted({len(x) for x in THEORY_PAIRS.values()}):
    structures = list(combinations(ALL_PAIRS, k))
    print(f'\nScoring all {len(structures):,} {k}-edge structures...')
    rows = []
    for edges in tqdm(structures, desc=f'{k}-edge structures', leave=False):
        edges = frozenset(edges)
        cll = graph_cll(edges)
        rows.append({'n_edges': k, 'edges': edges, 'cll': cll, 'saturated_gain': saturated_gain(cll)})
    df = pd.DataFrame(rows).sort_values('cll', ascending=False).reset_index(drop=True)
    df['rank'] = np.arange(len(df)) + 1
    budget_results[k] = df
    print(f"Best {k}-edge saturated gain: {df.iloc[0]['saturated_gain']:.3%}")
summary_rows = []
for theory, edges in THEORY_PAIRS.items():
    k = len(edges)
    df = budget_results[k]
    theory_cll = graph_cll(edges)
    theory_gain = saturated_gain(theory_cll)
    best_cll = df['cll'].max()
    best_gain = df['saturated_gain'].max()
    rank = 1 + int((df['cll'] > theory_cll).sum())
    percentile = 100 * np.mean(df['cll'] <= theory_cll)
    fraction_best = (theory_cll - cll_ind) / (best_cll - cll_ind)
    summary_rows.append({'theory': theory, 'edges': k, 'CLL': theory_cll, 'saturated_gain_pct': 100 * theory_gain, 'same_size_rank': rank, 'same_size_total': len(df), 'same_size_percentile': percentile, 'best_same_size_gain_pct': 100 * best_gain, 'fraction_best_same_size_pct': 100 * fraction_best})
theory_budget_summary = pd.DataFrame(summary_rows).sort_values(['edges', 'same_size_rank']).reset_index(drop=True)
display(theory_budget_summary.round(3))


## Empirical Structure Benchmarking

In [ ]:
def graph_cll_on_fold(pairs, fold_i):
    nbrs = graph_neighbors(pairs)
    node_scores = []
    for target in MECH_COLS:
        node_scores.append(node_cache[fold_i, target, tuple(sorted(nbrs[target]))])
    return float(np.mean(node_scores))

def benchmark_same_size_on_fold(target_pairs, fold_i):
    target_pairs = frozenset(target_pairs)
    k = len(target_pairs)
    target_cll = graph_cll_on_fold(target_pairs, fold_i)
    ind_cll = graph_cll_on_fold(INDEPENDENT_PAIRS, fold_i)
    scores = []
    for edges in combinations(ALL_PAIRS, k):
        scores.append(graph_cll_on_fold(frozenset(edges), fold_i))
    scores = np.asarray(scores, dtype=float)
    best_cll = float(scores.max())
    rank = 1 + int(np.sum(scores > target_cll))
    percentile = 100 * np.mean(scores <= target_cll)
    denom = best_cll - ind_cll
    fraction_best = (target_cll - ind_cll) / denom if denom > 0 else np.nan
    return {'edges': k, 'CLL': target_cll, 'same_size_rank': rank, 'same_size_total': len(scores), 'same_size_percentile': percentile, 'best_same_size_CLL': best_cll, 'fraction_best_same_size_pct': 100 * fraction_best if np.isfinite(fraction_best) else np.nan}
empirical_budget_rows = []
for fold in tqdm(range(1, OUTER_FOLDS + 1), desc='Empirical same-size benchmarks'):
    fold_i = fold - 1
    for structure_name, pair_dict in [('empirical_backbone_95', backbone_fold_pairs), ('bn_skeleton', bn_fold_pairs)]:
        result = benchmark_same_size_on_fold(pair_dict[fold], fold_i)
        empirical_budget_rows.append({'fold': fold, 'structure': structure_name, **result})
empirical_same_size_folds = pd.DataFrame(empirical_budget_rows)
display(empirical_same_size_folds.sort_values(['structure', 'fold']).round(3))
empirical_same_size_summary = empirical_same_size_folds.groupby('structure').agg(mean_edges=('edges', 'mean'), mean_same_size_percentile=('same_size_percentile', 'mean'), min_same_size_percentile=('same_size_percentile', 'min'), mean_fraction_best_same_size_pct=('fraction_best_same_size_pct', 'mean'), min_fraction_best_same_size_pct=('fraction_best_same_size_pct', 'min'))
display(empirical_same_size_summary.round(2))


In [ ]:
directed_edge_weights = directed_unique_weights(X_bn, n_folds=EDGE_FULL_FOLDS, random_state=RANDOM_STATE)
pair_weights_full = pair_weights_from_directed(directed_edge_weights)
BACKBONE_PAIRS = select_backbone(pair_weights_full, threshold=BACKBONE_CUM_WEIGHT)
stability_rows = []
for pair in tqdm(ALL_PAIRS):
    backbone_hits = sum((pair in backbone_fold_pairs[fold] for fold in backbone_fold_pairs))
    bn_hits = sum((pair in bn_fold_pairs[fold] for fold in bn_fold_pairs))
    stability_rows.append({'A': pair[0], 'B': pair[1], 'A_label': SHORT[pair[0]], 'B_label': SHORT[pair[1]], 'backbone_fold_count': backbone_hits, 'backbone_fold_frequency': backbone_hits / OUTER_FOLDS, 'bn_fold_count': bn_hits, 'bn_fold_frequency': bn_hits / OUTER_FOLDS})
structure_fold_stability = pd.DataFrame(stability_rows)
BN_CONSENSUS_PAIRS = {canon(row.A, row.B) for row in structure_fold_stability.itertuples() if row.bn_fold_frequency >= BN_CONSENSUS_MIN_FREQ}
pair_weights_full['in_95_backbone'] = pair_weights_full.apply(lambda row: canon(row['A'], row['B']) in BACKBONE_PAIRS, axis=1)
pair_weights_full['in_consensus_bn'] = pair_weights_full.apply(lambda row: canon(row['A'], row['B']) in BN_CONSENSUS_PAIRS, axis=1)
print('Final 95% empirical backbone')
print('--------------------------------')
for a, b in sorted(BACKBONE_PAIRS):
    print(f'{SHORT[a]} -- {SHORT[b]}')
print(f'\nBackbone edges: {len(BACKBONE_PAIRS)}')
print('\nConsensus BN skeleton')
print('--------------------------------')
for a, b in sorted(BN_CONSENSUS_PAIRS):
    print(f'{SHORT[a]} -- {SHORT[b]}')
print(f'\nConsensus BN edges: {len(BN_CONSENSUS_PAIRS)}')
print('\nBackbone edges contained in consensus BN:', f'{len(BACKBONE_PAIRS & BN_CONSENSUS_PAIRS)} / {len(BACKBONE_PAIRS)}')
display(pair_weights_full.round(6))


In [ ]:
def benchmark_same_size_cv(structure_name, target_pairs):
    target_pairs = frozenset(target_pairs)
    k = len(target_pairs)
    target_cll = graph_cll(target_pairs)
    scores = []
    for edges in tqdm(combinations(ALL_PAIRS, k), total=math.comb(len(ALL_PAIRS), k), desc=f'{structure_name}: {k}-edge graphs', leave=False):
        scores.append(graph_cll(frozenset(edges)))
    scores = np.asarray(scores, dtype=float)
    best_cll = float(scores.max())
    rank = 1 + int(np.sum(scores > target_cll))
    percentile = 100 * np.mean(scores <= target_cll)
    target_gain = saturated_gain(target_cll)
    best_gain = saturated_gain(best_cll)
    fraction_best = (target_cll - cll_ind) / (best_cll - cll_ind) if best_cll > cll_ind else np.nan
    return {'structure': structure_name, 'edges': k, 'CLL': target_cll, 'saturated_gain_pct': 100 * target_gain, 'same_size_rank': rank, 'same_size_total': len(scores), 'same_size_percentile': percentile, 'best_same_size_gain_pct': 100 * best_gain, 'fraction_best_same_size_pct': 100 * fraction_best if np.isfinite(fraction_best) else np.nan}
pooled_empirical_budget_summary = pd.DataFrame([benchmark_same_size_cv('Final empirical backbone 95%', BACKBONE_PAIRS), benchmark_same_size_cv('Final consensus BN skeleton', BN_CONSENSUS_PAIRS)])
display(pooled_empirical_budget_summary.round(3))


## Backbone Stability And Bayesian Networks

In [ ]:
stability_rows = []
for pair in ALL_PAIRS:
    backbone_hits = sum((pair in backbone_fold_pairs[fold] for fold in backbone_fold_pairs))
    bn_hits = sum((pair in bn_fold_pairs[fold] for fold in bn_fold_pairs))
    stability_rows.append({'A': pair[0], 'B': pair[1], 'A_label': SHORT[pair[0]], 'B_label': SHORT[pair[1]], 'backbone_fold_frequency': backbone_hits / OUTER_FOLDS, 'bn_fold_frequency': bn_hits / OUTER_FOLDS})
structure_fold_stability = pd.DataFrame(stability_rows)
BN_CONSENSUS_PAIRS = {canon(row.A, row.B) for row in structure_fold_stability.itertuples() if row.bn_fold_frequency >= BN_CONSENSUS_MIN_FREQ}
directed_edge_weights = directed_unique_weights(X_bn, n_folds=EDGE_FULL_FOLDS, random_state=RANDOM_STATE)
pair_weights_full = pair_weights_from_directed(directed_edge_weights)
BACKBONE_PAIRS = select_backbone(pair_weights_full)
pair_weights_full['in_95_backbone'] = pair_weights_full.apply(lambda r: canon(r['A'], r['B']) in BACKBONE_PAIRS, axis=1)
pair_weights_full['in_consensus_bn'] = pair_weights_full.apply(lambda r: canon(r['A'], r['B']) in BN_CONSENSUS_PAIRS, axis=1)
display(pair_weights_full.round(6))
print('\n95% backbone:')
for a, b in sorted(BACKBONE_PAIRS):
    print(' ', SHORT[a], '—', SHORT[b])
print('\nConsensus BN:')
for a, b in sorted(BN_CONSENSUS_PAIRS):
    print(' ', SHORT[a], '—', SHORT[b])
print('\nBackbone recovered by consensus BN:', f'{len(BACKBONE_PAIRS & BN_CONSENSUS_PAIRS)} / {len(BACKBONE_PAIRS)}')


## Conditional Associations

In [ ]:
conditional_rows = []
for target in tqdm(MECH_COLS, desc='Conditional GLMs'):
    predictors = [m for m in MECH_COLS if m != target]
    X = sm.add_constant(X_bn[predictors].astype(float), has_constant='add')
    y = X_bn[target].astype(int)
    result = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    ci = result.conf_int()
    for source in predictors:
        weight = directed_edge_weights.loc[(directed_edge_weights['source'] == source) & (directed_edge_weights['target'] == target)].iloc[0]
        beta = result.params[source]
        conditional_rows.append({'source': source, 'target': target, 'source_label': SHORT[source], 'target_label': SHORT[target], 'beta': beta, 'odds_ratio': np.exp(beta), 'or_ci_low': np.exp(ci.loc[source, 0]), 'or_ci_high': np.exp(ci.loc[source, 1]), 'p_value': result.pvalues[source], 'delta_log_loss': weight['delta_log_loss'], 'positive_folds': weight['positive_folds']})
conditional_effects = pd.DataFrame(conditional_rows)
conditional_effects['predictive_rank_for_target'] = conditional_effects.groupby('target')['delta_log_loss'].rank(method='min', ascending=False)
display(conditional_effects.sort_values('delta_log_loss', ascending=False).round(5))


## Platform Replication

In [ ]:
platform_skeletons = {}
platform_ns = {}
for platform in tqdm(sorted(annotated['platform'].dropna().unique()), desc='Platform BNs'):
    sub = annotated.loc[annotated['platform'] == platform, MECH_COLS].dropna().astype(int).reset_index(drop=True)
    platform_ns[platform] = len(sub)
    if len(sub) >= 100:
        platform_skeletons[platform] = learn_bn_skeleton(sub)
platform_rows = []
for pair in ALL_PAIRS:
    row = {'A': pair[0], 'B': pair[1], 'A_label': SHORT[pair[0]], 'B_label': SHORT[pair[1]], 'in_95_backbone': pair in BACKBONE_PAIRS, 'in_consensus_bn': pair in BN_CONSENSUS_PAIRS}
    recovered = 0
    for platform, skeleton in platform_skeletons.items():
        present = pair in skeleton
        row[str(platform)] = present
        recovered += int(present)
    row['n_platforms'] = recovered
    row['platform_fraction'] = recovered / len(platform_skeletons) if platform_skeletons else np.nan
    platform_rows.append(row)
platform_edge_replication = pd.DataFrame(platform_rows)
display(platform_edge_replication.sort_values(['in_95_backbone', 'platform_fraction'], ascending=[False, False]))


## Temporal First Appearance

In [ ]:
temporal = annotated.copy()
temporal['timestamp'] = pd.to_datetime(temporal['timestamp'], utc=True, errors='coerce')
temporal = temporal.dropna(subset=['timestamp', 'cluster']).copy()
cluster_sizes = temporal.groupby('cluster').size()
eligible = cluster_sizes.loc[cluster_sizes >= TEMPORAL_MIN_POSTS].index
temporal = temporal.loc[temporal['cluster'].isin(eligible)].copy()
print('Eligible narratives:', temporal['cluster'].nunique())
first_rows = []
for cluster, g in tqdm(temporal.groupby('cluster', sort=False), desc='First appearances'):
    g = g.sort_values('timestamp')
    n = len(g)
    for mech in MECH_COLS:
        arr = g[mech].astype(int).to_numpy()
        k = int(arr.sum())
        if k == 0:
            continue
        first_zero = int(np.flatnonzero(arr)[0])
        observed = first_zero / (n - 1) if n > 1 else 0.0
        expected_rank_1 = (n + 1) / (k + 1)
        expected = (expected_rank_1 - 1) / (n - 1) if n > 1 else 0.0
        first_rows.append({'cluster': cluster, 'mechanism': mech, 'n_posts': n, 'k_occurrences': k, 'observed_first_norm': observed, 'expected_first_norm': expected, 'observed_minus_expected': observed - expected})
first_appearance = pd.DataFrame(first_rows)
analytic_temporal_summary = first_appearance.groupby('mechanism').agg(narratives=('cluster', 'nunique'), observed_first=('observed_first_norm', 'mean'), expected_from_base_rate=('expected_first_norm', 'mean'), observed_minus_expected=('observed_minus_expected', 'mean')).sort_values('observed_minus_expected')
display(analytic_temporal_summary.round(4))


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
observed_means = first_appearance.groupby('mechanism')['observed_first_norm'].mean()
nk = {mech: first_appearance.loc[first_appearance['mechanism'] == mech, ['n_posts', 'k_occurrences']].to_numpy(dtype=int) for mech in MECH_COLS}
null_means = {mech: np.empty(TEMPORAL_NULL_DRAWS) for mech in MECH_COLS}
for draw in tqdm(range(TEMPORAL_NULL_DRAWS), desc='Temporal null'):
    for mech in MECH_COLS:
        mins = []
        for n, k in nk[mech]:
            positions = rng.choice(n, size=k, replace=False)
            first = int(positions.min())
            mins.append(first / (n - 1) if n > 1 else 0.0)
        null_means[mech][draw] = np.mean(mins)
null_rows = []
for mech in MECH_COLS:
    sims = null_means[mech]
    obs = float(observed_means[mech])
    center = float(sims.mean())
    p_two = (1 + np.sum(np.abs(sims - center) >= abs(obs - center))) / (len(sims) + 1)
    null_rows.append({'mechanism': mech, 'label': FULL_LABEL[mech], 'observed_mean_first': obs, 'null_mean': center, 'null_ci_low': np.quantile(sims, 0.025), 'null_ci_high': np.quantile(sims, 0.975), 'observed_minus_null': obs - center, 'empirical_p_two_sided': p_two})
temporal_null_results = pd.DataFrame(null_rows).sort_values('observed_minus_null')
display(temporal_null_results.round(4))


## Adjusted Temporal Ordering

In [ ]:
temporal_reg = first_appearance.copy()
temporal_reg['log_k_occurrences'] = np.log1p(temporal_reg['k_occurrences'])
temporal_reg['log_n_posts'] = np.log1p(temporal_reg['n_posts'])
REFERENCE_MECH = MECH_COLS[0]
formula = f"observed_first_norm ~ C(mechanism, Treatment(reference='{REFERENCE_MECH}')) + log_k_occurrences + log_n_posts"
timing_model = smf.ols(formula, data=temporal_reg).fit(cov_type='cluster', cov_kwds={'groups': temporal_reg['cluster']})
grid = pd.DataFrame({'mechanism': MECH_COLS, 'log_k_occurrences': temporal_reg['log_k_occurrences'].mean(), 'log_n_posts': temporal_reg['log_n_posts'].mean()})
grid['adjusted_first'] = timing_model.predict(grid)
grid['label'] = grid['mechanism'].map(FULL_LABEL)
adjusted_timing = grid.sort_values('adjusted_first').reset_index(drop=True)
display(adjusted_timing)


In [ ]:
params = timing_model.params
cov = timing_model.cov_params()
param_names = list(params.index)

def mechanism_parameter_name(mech):
    if mech == REFERENCE_MECH:
        return None
    return f"C(mechanism, Treatment(reference='{REFERENCE_MECH}'))[T.{mech}]"
contrast_rows = []
for a, b in combinations(MECH_COLS, 2):
    c = np.zeros(len(params))
    pa = mechanism_parameter_name(a)
    pb = mechanism_parameter_name(b)
    if pa is not None:
        c[param_names.index(pa)] += 1
    if pb is not None:
        c[param_names.index(pb)] -= 1
    estimate = float(c @ params.to_numpy())
    variance = float(c @ cov.to_numpy() @ c)
    se = math.sqrt(max(variance, 0))
    z = estimate / se if se > 0 else np.nan
    p = 2 * norm.sf(abs(z)) if np.isfinite(z) else np.nan
    contrast_rows.append({'A': a, 'B': b, 'A_label': FULL_LABEL[a], 'B_label': FULL_LABEL[b], 'adjusted_diff_A_minus_B': estimate, 'se': se, 'z': z, 'p_value': p})
timing_pairwise = pd.DataFrame(contrast_rows)
valid = timing_pairwise['p_value'].notna()
timing_pairwise.loc[valid, 'fdr_q'] = multipletests(timing_pairwise.loc[valid, 'p_value'], method='fdr_bh')[1]
display(timing_pairwise.sort_values('fdr_q').round(5))


In [ ]:
ordered_mechs = adjusted_timing['mechanism'].tolist()

def q_for_pair(a, b):
    pair = canon(a, b)
    row = timing_pairwise.loc[timing_pairwise.apply(lambda r: canon(r['A'], r['B']) == pair, axis=1)]
    return float(row.iloc[0]['fdr_q'])

def contiguous_partitions(items):
    n = len(items)
    for mask in range(1 << n - 1):
        groups = []
        start = 0
        for i in range(n - 1):
            if mask & 1 << i:
                groups.append(items[start:i + 1])
                start = i + 1
        groups.append(items[start:])
        yield groups

def valid_phase_partition(groups, alpha=0.05):
    for group in groups:
        for a, b in combinations(group, 2):
            if q_for_pair(a, b) < alpha:
                return False
    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            for a in groups[i]:
                for b in groups[j]:
                    if q_for_pair(a, b) >= alpha:
                        return False
    return True
valid_partitions = [p for p in contiguous_partitions(ordered_mechs) if valid_phase_partition(p)]
if valid_partitions:
    valid_partitions.sort(key=len)
    temporal_phases = valid_partitions[0]
    phase_rows = []
    print('Statistically supported phases:')
    for phase, group in enumerate(temporal_phases, start=1):
        print(f'  Phase {phase}: ' + ', '.join((FULL_LABEL[m] for m in group)))
        for mech in group:
            phase_rows.append({'mechanism': mech, 'label': FULL_LABEL[mech], 'phase': phase})
    temporal_phase_table = pd.DataFrame(phase_rows)
else:
    temporal_phases = []
    temporal_phase_table = pd.DataFrame(columns=['mechanism', 'label', 'phase'])
    print('No exact phase partition satisfies the FDR-corrected within/between-phase criteria.')
display(temporal_phase_table)


In [ ]:
from itertools import combinations
import numpy as np
import pandas as pd
import math
from sklearn.cluster import KMeans, AgglomerativeClustering
timing = adjusted_timing[['mechanism', 'label', 'adjusted_first']].sort_values('adjusted_first').reset_index(drop=True).copy()
ordered_mechs = timing['mechanism'].tolist()
ordered_values = timing['adjusted_first'].to_numpy(dtype=float)
print('Adjusted temporal ordering')
print('--------------------------')
display(timing.round(5))

def canon_pair(a, b):
    return tuple(sorted((a, b)))

def q_for_pair(a, b):
    pair = canon_pair(a, b)
    row = timing_pairwise.loc[timing_pairwise.apply(lambda r: canon_pair(r['A'], r['B']) == pair, axis=1)]
    if len(row) != 1:
        raise ValueError(f'Expected exactly one contrast for {a}, {b}; got {len(row)}')
    return float(row.iloc[0]['fdr_q'])

def groups_to_string(groups):
    return ' | '.join((', '.join((FULL_LABEL[m] for m in group)) for group in groups))

def boundaries_from_groups(groups):
    boundaries = []
    for i in range(len(groups) - 1):
        boundaries.append((groups[i][-1], groups[i + 1][0]))
    return boundaries

def make_phase_table(method, groups):
    rows = []
    for phase_i, group in enumerate(groups, start=1):
        for mech in group:
            rows.append({'method': method, 'mechanism': mech, 'label': FULL_LABEL[mech], 'phase': phase_i})
    return pd.DataFrame(rows)

def contiguous_groups_from_cuts(items, cuts):
    groups = []
    start = 0
    for cut in cuts:
        groups.append(items[start:cut])
        start = cut
    groups.append(items[start:])
    return groups
ALPHA = 0.05
adjacent_rows = []
for earlier, later in zip(ordered_mechs[:-1], ordered_mechs[1:]):
    q = q_for_pair(earlier, later)
    adjacent_rows.append({'earlier': earlier, 'later': later, 'earlier_label': FULL_LABEL[earlier], 'later_label': FULL_LABEL[later], 'fdr_q': q, 'significant': q < ALPHA})
adjacent_phase_tests = pd.DataFrame(adjacent_rows)
adjacent_groups = [[ordered_mechs[0]]]
for row in adjacent_phase_tests.itertuples():
    if row.significant:
        adjacent_groups.append([row.later])
    else:
        adjacent_groups[-1].append(row.later)
print('\n' + '=' * 80)
print('METHOD 1 — ADJACENT SIGNIFICANCE BREAKS')
print('=' * 80)
display(adjacent_phase_tests.round(5))
print(groups_to_string(adjacent_groups))
gaps = np.diff(ordered_values)

def largest_gap_partition(k):
    if k < 1 or k > len(ordered_mechs):
        raise ValueError(k)
    if k == 1:
        return [ordered_mechs.copy()]
    gap_idx = np.argsort(gaps)[-(k - 1):]
    cuts = sorted((gap_idx + 1).tolist())
    return contiguous_groups_from_cuts(ordered_mechs, cuts)
largest_gap_results = {}
for k in [2, 3]:
    groups = largest_gap_partition(k)
    largest_gap_results[k] = groups
print('\n' + '=' * 80)
print('METHOD 2 — LARGEST TEMPORAL GAPS')
print('=' * 80)
for k, groups in largest_gap_results.items():
    print(f'{k} phases:', groups_to_string(groups))

def segment_sse(values, start, stop):
    segment = values[start:stop]
    if len(segment) == 0:
        return np.inf
    return float(np.sum((segment - segment.mean()) ** 2))

def all_contiguous_k_partitions(n, k):
    if k == 1:
        yield []
        return
    for cuts in combinations(range(1, n), k - 1):
        yield list(cuts)

def optimal_contiguous_partition(values, items, k):
    best = None
    for cuts in all_contiguous_k_partitions(len(items), k):
        endpoints = [0] + cuts + [len(items)]
        sse = 0.0
        for start, stop in zip(endpoints[:-1], endpoints[1:]):
            sse += segment_sse(values, start, stop)
        if best is None or sse < best['sse']:
            best = {'cuts': cuts, 'sse': sse, 'groups': contiguous_groups_from_cuts(items, cuts)}
    return best
optimal_results = {}
for k in [1, 2, 3, 4]:
    optimal_results[k] = optimal_contiguous_partition(ordered_values, ordered_mechs, k)
print('\n' + '=' * 80)
print('METHOD 3 — OPTIMAL CONTIGUOUS PARTITION')
print('=' * 80)
for k in [2, 3]:
    result = optimal_results[k]
    print(f"{k} phases | SSE={result['sse']:.8f}:", groups_to_string(result['groups']))
n = len(ordered_values)
bic_rows = []
for k, result in optimal_results.items():
    sse = result['sse']
    sigma2 = max(sse / n, 1e-12)
    n_parameters = k + 1
    bic = n * np.log(sigma2) + n_parameters * np.log(n)
    aic = n * np.log(sigma2) + 2 * n_parameters
    bic_rows.append({'phases': k, 'SSE': sse, 'AIC': aic, 'BIC': bic, 'grouping': groups_to_string(result['groups'])})
phase_model_selection = pd.DataFrame(bic_rows).sort_values('BIC').reset_index(drop=True)
print('\n' + '=' * 80)
print('METHOD 4 — MODEL SELECTION')
print('=' * 80)
display(phase_model_selection.round(5))
best_bic_k = int(phase_model_selection.iloc[0]['phases'])
best_bic_groups = optimal_results[best_bic_k]['groups']
print('Best BIC solution:', best_bic_k, 'phases')
print(groups_to_string(best_bic_groups))
bic_2_vs_3 = phase_model_selection.loc[phase_model_selection['phases'].isin([2, 3])].sort_values('BIC').reset_index(drop=True)
print('\nBIC restricted to 2 vs 3 phases:')
display(bic_2_vs_3.round(5))
kmeans_results = {}
X_time = ordered_values.reshape(-1, 1)
for k in [2, 3]:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=100)
    raw_labels = km.fit_predict(X_time)
    cluster_means = {c: ordered_values[raw_labels == c].mean() for c in np.unique(raw_labels)}
    phase_order = {cluster: phase_i for phase_i, cluster in enumerate(sorted(cluster_means, key=cluster_means.get), start=1)}
    phase_labels = np.array([phase_order[x] for x in raw_labels])
    groups = []
    for phase_i in range(1, k + 1):
        group = [ordered_mechs[i] for i in range(len(ordered_mechs)) if phase_labels[i] == phase_i]
        groups.append(group)
    kmeans_results[k] = {'groups': groups, 'inertia': km.inertia_}
print('\n' + '=' * 80)
print('METHOD 5 — 1D K-MEANS')
print('=' * 80)
for k, result in kmeans_results.items():
    print(f"{k} phases | inertia={result['inertia']:.8f}:", groups_to_string(result['groups']))
hierarchical_results = {}
for k in [2, 3]:
    hc = AgglomerativeClustering(n_clusters=k, linkage='ward')
    raw_labels = hc.fit_predict(X_time)
    cluster_means = {c: ordered_values[raw_labels == c].mean() for c in np.unique(raw_labels)}
    phase_order = {cluster: phase_i for phase_i, cluster in enumerate(sorted(cluster_means, key=cluster_means.get), start=1)}
    phase_labels = np.array([phase_order[x] for x in raw_labels])
    groups = []
    for phase_i in range(1, k + 1):
        group = [ordered_mechs[i] for i in range(len(ordered_mechs)) if phase_labels[i] == phase_i]
        groups.append(group)
    hierarchical_results[k] = groups
print('\n' + '=' * 80)
print('METHOD 6 — HIERARCHICAL WARD CLUSTERING')
print('=' * 80)
for k, groups in hierarchical_results.items():
    print(f'{k} phases:', groups_to_string(groups))
comparison_rows = []
comparison_rows.append({'method': 'Adjacent FDR breaks', 'requested_k': len(adjacent_groups), 'n_phases': len(adjacent_groups), 'grouping': groups_to_string(adjacent_groups), 'boundaries': str([(FULL_LABEL[a], FULL_LABEL[b]) for a, b in boundaries_from_groups(adjacent_groups)])})
for k in [2, 3]:
    groups = largest_gap_results[k]
    comparison_rows.append({'method': 'Largest gaps', 'requested_k': k, 'n_phases': len(groups), 'grouping': groups_to_string(groups), 'boundaries': str([(FULL_LABEL[a], FULL_LABEL[b]) for a, b in boundaries_from_groups(groups)])})
    groups = optimal_results[k]['groups']
    comparison_rows.append({'method': 'Optimal contiguous SSE', 'requested_k': k, 'n_phases': len(groups), 'grouping': groups_to_string(groups), 'boundaries': str([(FULL_LABEL[a], FULL_LABEL[b]) for a, b in boundaries_from_groups(groups)])})
    groups = kmeans_results[k]['groups']
    comparison_rows.append({'method': '1D K-means', 'requested_k': k, 'n_phases': len(groups), 'grouping': groups_to_string(groups), 'boundaries': str([(FULL_LABEL[a], FULL_LABEL[b]) for a, b in boundaries_from_groups(groups)])})
    groups = hierarchical_results[k]
    comparison_rows.append({'method': 'Ward hierarchical', 'requested_k': k, 'n_phases': len(groups), 'grouping': groups_to_string(groups), 'boundaries': str([(FULL_LABEL[a], FULL_LABEL[b]) for a, b in boundaries_from_groups(groups)])})
phase_method_comparison = pd.DataFrame(comparison_rows)
print('\n' + '=' * 80)
print('ALL PHASE METHODS')
print('=' * 80)
display(phase_method_comparison)
boundary_counter = {}
for row in phase_method_comparison.itertuples():
    if row.requested_k not in [2, 3]:
        continue
    pass
method_group_sets = []
for k in [2, 3]:
    method_group_sets.extend([(f'Largest gaps k={k}', largest_gap_results[k]), (f'Optimal contiguous k={k}', optimal_results[k]['groups']), (f'K-means k={k}', kmeans_results[k]['groups']), (f'Ward k={k}', hierarchical_results[k])])
boundary_rows = []
for i in range(len(ordered_mechs) - 1):
    left = ordered_mechs[i]
    right = ordered_mechs[i + 1]
    count = 0
    methods = []
    for method_name, groups in method_group_sets:
        boundaries = set(boundaries_from_groups(groups))
        if (left, right) in boundaries:
            count += 1
            methods.append(method_name)
    boundary_rows.append({'after': FULL_LABEL[left], 'before': FULL_LABEL[right], 'boundary_count': count, 'total_solutions': len(method_group_sets), 'boundary_fraction': count / len(method_group_sets), 'methods': ', '.join(methods)})
phase_boundary_consensus = pd.DataFrame(boundary_rows).sort_values('boundary_fraction', ascending=False)
print('\n' + '=' * 80)
print('BOUNDARY CONSENSUS ACROSS METHODS')
print('=' * 80)
display(phase_boundary_consensus.round(3))


## Temporal Bootstrap And Figures

In [ ]:
N_TIME_BOOT = 1000
BOOT_SEED = 42
rng = np.random.default_rng(BOOT_SEED)
clusters = temporal_reg['cluster'].dropna().unique()
bootstrap_timing_rows = []

def fit_adjusted_timing_bootstrap(boot_df):
    reference = REFERENCE_MECH
    formula = f"observed_first_norm ~ C(mechanism, Treatment(reference='{reference}')) + log_k_occurrences + log_n_posts"
    model = smf.ols(formula, data=boot_df).fit()
    grid = pd.DataFrame({'mechanism': MECH_COLS, 'log_k_occurrences': boot_df['log_k_occurrences'].mean(), 'log_n_posts': boot_df['log_n_posts'].mean()})
    grid['adjusted_first'] = model.predict(grid)
    return grid[['mechanism', 'adjusted_first']]
for b in tqdm(range(N_TIME_BOOT), desc='Bootstrap adjusted timing'):
    sampled_clusters = rng.choice(clusters, size=len(clusters), replace=True)
    boot_parts = []
    for draw_i, cluster in enumerate(sampled_clusters):
        g = temporal_reg.loc[temporal_reg['cluster'] == cluster].copy()
        g['cluster_boot'] = draw_i
        boot_parts.append(g)
    boot_df = pd.concat(boot_parts, ignore_index=True)
    try:
        timing_est = fit_adjusted_timing_bootstrap(boot_df)
    except Exception:
        continue
    timing_est['bootstrap'] = b
    bootstrap_timing_rows.append(timing_est)
bootstrap_timing = pd.concat(bootstrap_timing_rows, ignore_index=True)
bootstrap_timing['label'] = bootstrap_timing['mechanism'].map(FULL_LABEL)
print('Successful bootstrap samples:', bootstrap_timing['bootstrap'].nunique(), '/', N_TIME_BOOT)
bootstrap_timing_summary = bootstrap_timing.groupby(['mechanism', 'label']).agg(bootstrap_mean=('adjusted_first', 'mean'), bootstrap_median=('adjusted_first', 'median'), ci_low=('adjusted_first', lambda x: np.quantile(x, 0.025)), ci_high=('adjusted_first', lambda x: np.quantile(x, 0.975)), sd=('adjusted_first', 'std')).reset_index().sort_values('bootstrap_mean')
display(bootstrap_timing_summary.round(5))
rank_rows = []
for b, g in bootstrap_timing.groupby('bootstrap'):
    g = g.sort_values('adjusted_first').reset_index(drop=True)
    for rank, row in enumerate(g.itertuples(), start=1):
        rank_rows.append({'bootstrap': b, 'mechanism': row.mechanism, 'label': FULL_LABEL[row.mechanism], 'rank': rank, 'adjusted_first': row.adjusted_first})
bootstrap_ranks = pd.DataFrame(rank_rows)
bootstrap_rank_summary = bootstrap_ranks.groupby(['mechanism', 'label']).agg(mean_rank=('rank', 'mean'), median_rank=('rank', 'median'), rank_2_5=('rank', lambda x: np.quantile(x, 0.025)), rank_97_5=('rank', lambda x: np.quantile(x, 0.975)), pct_rank_1=('rank', lambda x: 100 * np.mean(x == 1)), pct_rank_6=('rank', lambda x: 100 * np.mean(x == 6))).reset_index().sort_values('mean_rank')
print('\nBootstrap rank distributions')
display(bootstrap_rank_summary.round(2))
wide_boot = bootstrap_timing.pivot(index='bootstrap', columns='mechanism', values='adjusted_first')
pairwise_boot_rows = []
for a, b in combinations(MECH_COLS, 2):
    valid = wide_boot[[a, b]].dropna()
    diff = valid[b] - valid[a]
    pairwise_boot_rows.append({'A': a, 'B': b, 'A_label': FULL_LABEL[a], 'B_label': FULL_LABEL[b], 'mean_B_minus_A': diff.mean(), 'ci_low': np.quantile(diff, 0.025), 'ci_high': np.quantile(diff, 0.975), 'P_B_later_than_A': np.mean(diff > 0), 'P_A_later_than_B': np.mean(diff < 0)})
bootstrap_pairwise_timing = pd.DataFrame(pairwise_boot_rows).sort_values('mean_B_minus_A')
print('\nPairwise bootstrap timing separation')
display(bootstrap_pairwise_timing.round(4))
PROPOSED_PHASES = {1: ['mech_negative_evaluation', 'mech_boundary_construction', 'mech_action_orientation'], 2: ['mech_dehumanization', 'mech_threat_construction'], 3: ['mech_scapegoating']}
phase_boot_rows = []
for b, g in bootstrap_timing.groupby('bootstrap'):
    vals = dict(zip(g['mechanism'], g['adjusted_first']))
    phase_means = {}
    for phase, mechs in PROPOSED_PHASES.items():
        phase_means[phase] = np.mean([vals[m] for m in mechs if m in vals])
    phase_boot_rows.append({'bootstrap': b, 'phase1_mean': phase_means[1], 'phase2_mean': phase_means[2], 'phase3_mean': phase_means[3], 'phase1_before_phase2': phase_means[1] < phase_means[2], 'phase2_before_phase3': phase_means[2] < phase_means[3], 'correct_3_phase_order': phase_means[1] < phase_means[2] < phase_means[3]})
phase_order_bootstrap = pd.DataFrame(phase_boot_rows)
print('\nProposed 3-phase bootstrap validation')
print('-------------------------------------')
print('P(Phase 1 earlier than Phase 2):', f"{phase_order_bootstrap['phase1_before_phase2'].mean():.3f}")
print('P(Phase 2 earlier than Phase 3):', f"{phase_order_bootstrap['phase2_before_phase3'].mean():.3f}")
print('P(full Phase 1 < Phase 2 < Phase 3 ordering):', f"{phase_order_bootstrap['correct_3_phase_order'].mean():.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
order = bootstrap_timing_summary['mechanism'].tolist()
data = [bootstrap_timing.loc[bootstrap_timing['mechanism'] == mech, 'adjusted_first'].to_numpy() for mech in order]
ax.boxplot(data, vert=False, labels=[FULL_LABEL[m] for m in order], showfliers=False)
ax.set_xlabel('Prevalence-adjusted first appearance')
ax.set_ylabel('')
ax.set_title('Bootstrap distributions of adjusted temporal timing')
plt.tight_layout()
plt.show()


In [ ]:
bootstrap_ranks_long = bootstrap_ranks[['bootstrap', 'mechanism', 'label', 'rank']].sort_values(['bootstrap', 'mechanism']).reset_index(drop=True)
display(bootstrap_ranks_long.head(20))
bootstrap_rank_array = bootstrap_ranks.pivot(index='bootstrap', columns='mechanism', values='rank').sort_index()
bootstrap_rank_array = bootstrap_rank_array.rename(columns=FULL_LABEL)
display(bootstrap_rank_array.head())
print('Rank-array shape:', bootstrap_rank_array.shape)
rank_distribution_rows = []
for mech in MECH_COLS:
    ranks = bootstrap_ranks.loc[bootstrap_ranks['mechanism'] == mech, 'rank'].astype(int)
    n = len(ranks)
    row = {'mechanism': mech, 'label': FULL_LABEL[mech], 'n_bootstraps': n}
    for rank in range(1, len(MECH_COLS) + 1):
        count = int((ranks == rank).sum())
        row[f'count_rank_{rank}'] = count
        row[f'pct_rank_{rank}'] = 100 * count / n if n > 0 else np.nan
        row[f'prob_rank_{rank}'] = count / n if n > 0 else np.nan
    rank_distribution_rows.append(row)
bootstrap_rank_distribution = pd.DataFrame(rank_distribution_rows)
mean_ranks = bootstrap_ranks.groupby('mechanism')['rank'].mean()
bootstrap_rank_distribution['mean_rank'] = bootstrap_rank_distribution['mechanism'].map(mean_ranks)
bootstrap_rank_distribution = bootstrap_rank_distribution.sort_values('mean_rank').reset_index(drop=True)
print('\nFull rank probability distributions')
display(bootstrap_rank_distribution.round(4))
tidy_rows = []
for mech in MECH_COLS:
    ranks = bootstrap_ranks.loc[bootstrap_ranks['mechanism'] == mech, 'rank'].astype(int)
    n = len(ranks)
    for rank in range(1, len(MECH_COLS) + 1):
        count = int((ranks == rank).sum())
        tidy_rows.append({'mechanism': mech, 'label': FULL_LABEL[mech], 'rank': rank, 'count': count, 'probability': count / n if n > 0 else np.nan, 'percent': 100 * count / n if n > 0 else np.nan})
bootstrap_rank_distribution_long = pd.DataFrame(tidy_rows).sort_values(['mechanism', 'rank'])
display(bootstrap_rank_distribution_long.head(20))
bootstrap_ranks_long.to_csv(OUTPUT_DIR / 'bootstrap_ranks_long.csv', index=False)
bootstrap_rank_array.to_csv(OUTPUT_DIR / 'bootstrap_rank_array.csv', index=True)
bootstrap_rank_distribution.to_csv(OUTPUT_DIR / 'bootstrap_rank_distribution.csv', index=False)
bootstrap_rank_distribution_long.to_csv(OUTPUT_DIR / 'bootstrap_rank_distribution_long.csv', index=False)
print('\nSaved:')
print(OUTPUT_DIR / 'bootstrap_ranks_long.csv')
print(OUTPUT_DIR / 'bootstrap_rank_array.csv')
print(OUTPUT_DIR / 'bootstrap_rank_distribution.csv')
print(OUTPUT_DIR / 'bootstrap_rank_distribution_long.csv')


In [ ]:
N_RAW_BOOT = 1000
BOOT_SEED = 42
rng = np.random.default_rng(BOOT_SEED)
raw_temporal = first_appearance[['cluster', 'mechanism', 'observed_first_norm']].copy()
clusters = raw_temporal['cluster'].dropna().unique()
raw_bootstrap_rows = []
for b in tqdm(range(N_RAW_BOOT), desc='Bootstrap raw temporal ranks'):
    sampled_clusters = rng.choice(clusters, size=len(clusters), replace=True)
    boot_parts = []
    for draw_i, cluster in enumerate(sampled_clusters):
        g = raw_temporal.loc[raw_temporal['cluster'] == cluster].copy()
        g['cluster_boot'] = draw_i
        boot_parts.append(g)
    boot = pd.concat(boot_parts, ignore_index=True)
    estimates = boot.groupby('mechanism')['observed_first_norm'].mean()
    if not all((mech in estimates.index for mech in MECH_COLS)):
        continue
    ordered = estimates.sort_values().index.tolist()
    rank_map = {mech: rank for rank, mech in enumerate(ordered, start=1)}
    for mech in MECH_COLS:
        raw_bootstrap_rows.append({'bootstrap': b, 'mechanism': mech, 'label': FULL_LABEL[mech], 'raw_first': float(estimates[mech]), 'rank': rank_map[mech]})
raw_bootstrap_ranks = pd.DataFrame(raw_bootstrap_rows)
print('Successful raw bootstrap samples:', raw_bootstrap_ranks['bootstrap'].nunique(), '/', N_RAW_BOOT)
raw_bootstrap_rank_array = raw_bootstrap_ranks.pivot(index='bootstrap', columns='mechanism', values='rank').sort_index()
raw_bootstrap_rank_array = raw_bootstrap_rank_array.rename(columns=FULL_LABEL)
display(raw_bootstrap_rank_array.head())
print('Raw rank-array shape:', raw_bootstrap_rank_array.shape)
raw_rank_distribution_rows = []
for mech in MECH_COLS:
    ranks = raw_bootstrap_ranks.loc[raw_bootstrap_ranks['mechanism'] == mech, 'rank'].astype(int)
    n = len(ranks)
    for rank in range(1, len(MECH_COLS) + 1):
        count = int((ranks == rank).sum())
        raw_rank_distribution_rows.append({'mechanism': mech, 'label': FULL_LABEL[mech], 'rank': rank, 'count': count, 'probability': count / n if n > 0 else np.nan, 'percent': 100 * count / n if n > 0 else np.nan})
raw_bootstrap_rank_distribution_long = pd.DataFrame(raw_rank_distribution_rows)
display(raw_bootstrap_rank_distribution_long.head(20))
raw_bootstrap_timing_summary = raw_bootstrap_ranks.groupby(['mechanism', 'label']).agg(bootstrap_mean_raw_first=('raw_first', 'mean'), bootstrap_median_raw_first=('raw_first', 'median'), raw_first_ci_low=('raw_first', lambda x: np.quantile(x, 0.025)), raw_first_ci_high=('raw_first', lambda x: np.quantile(x, 0.975)), mean_rank=('rank', 'mean'), median_rank=('rank', 'median')).reset_index().sort_values('mean_rank')
display(raw_bootstrap_timing_summary.round(4))
raw_bootstrap_ranks.to_csv(OUTPUT_DIR / 'raw_bootstrap_ranks_long.csv', index=False)
raw_bootstrap_rank_array.to_csv(OUTPUT_DIR / 'raw_bootstrap_rank_array.csv', index=True)
raw_bootstrap_rank_distribution_long.to_csv(OUTPUT_DIR / 'raw_bootstrap_rank_distribution_long.csv', index=False)
raw_bootstrap_timing_summary.to_csv(OUTPUT_DIR / 'raw_bootstrap_timing_summary.csv', index=False)
print('\nSaved:')
print(OUTPUT_DIR / 'raw_bootstrap_rank_array.csv')
print(OUTPUT_DIR / 'raw_bootstrap_ranks_long.csv')
print(OUTPUT_DIR / 'raw_bootstrap_rank_distribution_long.csv')
print(OUTPUT_DIR / 'raw_bootstrap_timing_summary.csv')


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['cmr10', 'CMU Serif', 'DejaVu Serif']
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['axes.formatter.use_mathtext'] = True
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['text.color'] = '#000000'
bold_font = FontProperties(family='cmb10')
BLUE = '#0075F2'
GRAY = '#999999'
RED = '#B23A48'
frontier = [(0, 0.0), (2, 74.03), (4, 88.58), (7, 97.76)]
structures = [('Independent', 0, 0.0, True), ('Moral-exclusion', 2, 14.3, False), ('Threat-mediated', 2, 74.0, True), ('Integrated', 4, 84.1, False), ('Empirical backbone', 7, 97.8, True)]
fig, ax = plt.subplots(figsize=(5.3, 4.8))
fx, fy = zip(*frontier)
ax.plot(fx, fy, color='#c9c9c9', linewidth=1.6, linestyle=(0, (4, 2)), zorder=1, label='Best possible structure at this edge count')
label_offsets = {'Independent': (0.22, 1.0, 'left'), 'Moral-exclusion': (0.22, 0.0, 'left'), 'Threat-mediated': (0.22, 1.0, 'left'), 'Integrated': (0.22, 2.0, 'left'), 'Empirical backbone': (-0.22, 1.5, 'right')}
for name, n, pct, on_frontier in structures:
    color = BLUE if on_frontier else GRAY
    ax.scatter([n], [pct], s=120, color=color, edgecolor='white', linewidth=1.3, zorder=4)
    ox, oy, ha = label_offsets[name]
    ax.text(n + ox, pct + oy, name, fontsize=10.2, fontproperties=bold_font, color=color, ha=ha, va='center')
ax.plot([2, 2], [14.3, 74.03], color=RED, linewidth=1.1, linestyle=(0, (2, 2)), alpha=0.8, zorder=2)
ax.text(2.33, 43, '81% worse than\nbest possible', fontsize=9, color=RED, ha='left', va='center', linespacing=1.25)
ax.plot([4, 4], [84.1, 88.58], color=RED, linewidth=1.0, linestyle=(0, (2, 2)), alpha=0.65, zorder=2)
ax.set_xlabel('Number of edges', fontsize=10.5, fontproperties=bold_font, labelpad=6)
ax.set_ylabel('Share of available predictive\ngain recovered (%)', fontsize=10, fontproperties=bold_font, labelpad=6)
ax.set_xlim(-0.25, 7.35)
ax.set_ylim(-2, 102)
ax.set_xticks([0, 2, 4, 7])
ax.set_yticks([0, 20, 40, 60, 80, 100])
ax.tick_params(axis='both', labelsize=9, width=0.8, length=3)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.spines['left'].set_linewidth(0.9)
ax.spines['bottom'].set_linewidth(0.9)
ax.legend(loc='lower right', bbox_to_anchor=(0.99, 0.015), fontsize=8.2, frameon=False, handlelength=2.4)
plt.subplots_adjust(left=0.18, right=0.98, bottom=0.14, top=0.98)
plt.savefig('./efficient_frontier_v4.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['cmr10', 'CMU Serif', 'DejaVu Serif']
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['axes.formatter.use_mathtext'] = True
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['text.color'] = '#000000'
bold_font = FontProperties(family='cmb10')
BLUE = '#0075F2'
GRAY = '#999999'
RED = '#B23A48'
frontier = [(0, 0.0), (2, 74.03), (4, 88.58), (7, 97.76)]
structures = [('Independent', 0, 0.0, True), ('Moral-exclusion', 2, 14.3, False), ('Threat-mediated', 2, 74.0, True), ('Integrated', 4, 84.1, False), ('Empirical backbone', 7, 97.8, True)]
fig, ax = plt.subplots(figsize=(6.4, 4.7))
fx, fy = zip(*frontier)
ax.plot(fx, fy, color='#cccccc', linewidth=1.9, linestyle=(0, (4, 2)), zorder=1, label='Best possible structure at this edge count')
label_offsets = {'Independent': (0.34, 1.0, 'left'), 'Moral-exclusion': (0.34, -0.5, 'left'), 'Threat-mediated': (0.34, 0.5, 'left'), 'Integrated': (0.34, 2.8, 'left'), 'Empirical backbone': (-0.3, 2.3, 'right')}
for name, n, pct, on_frontier in structures:
    color = BLUE if on_frontier else GRAY
    ax.scatter([n], [pct], s=210, color=color, edgecolor='white', linewidth=1.8, zorder=4)
    ox, oy, ha = label_offsets[name]
    ax.text(n + ox, pct + oy, name, fontsize=11.2, fontproperties=bold_font, color=color, ha=ha, va='center', zorder=5)

def add_gap_callout(n, pct, text_side='right', y_nudge=0, show_text=True):
    frontier_pct = dict(frontier)[n]
    ax.plot([n, n], [pct, frontier_pct], color=RED, linewidth=1.4, linestyle=(0, (2, 2)), alpha=0.8, zorder=2)
    if not show_text:
        return
    worse_by = 100 - pct / frontier_pct * 100
    ha = 'left' if text_side == 'right' else 'right'
    x_text = n + 0.42 if text_side == 'right' else n - 0.42
    y_text = (pct + frontier_pct) / 2 + y_nudge
    ax.annotate(f'{worse_by:.0f}% worse than\nbest possible', xy=(x_text, y_text), ha=ha, va='center', fontsize=10.1, color=RED, linespacing=1.25)
add_gap_callout(2, 14.3, text_side='right')
add_gap_callout(4, 84.1, show_text=False)
ax.set_xlabel('Number of edges', fontsize=11.5, fontproperties=bold_font, labelpad=7)
ax.set_ylabel('Share of available predictive\ngain recovered (%)', fontsize=11.0, fontproperties=bold_font, labelpad=8)
ax.set_xlim(-0.45, 7.75)
ax.set_ylim(-5, 105)
ax.set_xticks([0, 2, 4, 7])
ax.set_yticks([0, 20, 40, 60, 80, 100])
ax.tick_params(axis='both', labelsize=9.8, width=0.9, length=3.5)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.spines['left'].set_linewidth(0.9)
ax.spines['bottom'].set_linewidth(0.9)
ax.legend(loc='lower right', fontsize=9.2, frameon=False, handlelength=2.8)
plt.tight_layout()
plt.savefig('./efficient_frontier_v5.pdf', bbox_inches='tight', pad_inches=0.03)
plt.show()


## Theory-Predicted Temporal Sequences

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
THEORY_SEQUENCES = {'Threat-mediated': ['mech_boundary_construction', 'mech_threat_construction', 'mech_action_orientation'], 'Moral-exclusion': ['mech_boundary_construction', 'mech_dehumanization', 'mech_action_orientation']}
N_BOOT = 2000
N_PERM = 5000
RANDOM_STATE = 42
print('Building narrative × mechanism first-appearance table...')
temporal_wide = first_appearance.pivot_table(index='cluster', columns='mechanism', values='observed_first_norm', aggfunc='first')
print(f'Done. {len(temporal_wide):,} narratives × {temporal_wide.shape[1]} mechanisms.')

def evaluate_sequence(wide_df, sequence, n_boot=2000, n_perm=5000, seed=42, theory_name='Theory'):
    local_rng = np.random.default_rng(seed)
    print(f"\n{'=' * 70}")
    print(f'Evaluating: {theory_name}')
    print('Sequence:', ' → '.join(sequence))
    print(f"{'=' * 70}")
    dat = wide_df[sequence].dropna().copy()
    n = len(dat)
    print(f'Eligible narratives: {n:,}')
    if n == 0:
        raise ValueError(f'No narratives contain all mechanisms in {sequence}')
    vals = dat.to_numpy()
    strict_full = np.all(np.diff(vals, axis=1) > 0, axis=1)
    weak_full = np.all(np.diff(vals, axis=1) >= 0, axis=1)
    no_ties = np.array([len(np.unique(row)) == len(row) for row in vals])
    strict_rate = strict_full.mean()
    weak_rate = weak_full.mean()
    strict_non_tied_rate = strict_full[no_ties].mean() if no_ties.sum() > 0 else np.nan
    print(f'Strict full sequence: {strict_full.sum():,}/{n:,} ({100 * strict_rate:.2f}%)')
    print(f'Weak full sequence: {100 * weak_rate:.2f}%')
    print(f'Non-tied narratives: {no_ties.sum():,}/{n:,}')
    step_rows = []
    for i in range(len(sequence) - 1):
        A = sequence[i]
        B = sequence[i + 1]
        a = dat[A].to_numpy()
        b = dat[B].to_numpy()
        non_tied = a != b
        step_rows.append({'source': A, 'target': B, 'n': len(dat), 'n_non_tied': int(non_tied.sum()), 'pct_source_before_target_all': 100 * np.mean(a < b), 'pct_source_before_target_non_tied': 100 * np.mean(a[non_tied] < b[non_tied]) if non_tied.sum() > 0 else np.nan, 'pct_tied': 100 * np.mean(a == b), 'mean_difference_target_minus_source': np.mean(b - a), 'median_difference_target_minus_source': np.median(b - a)})
    boot_rates = []
    for _ in tqdm(range(n_boot), desc=f'{theory_name}: bootstrap', leave=True):
        idx = local_rng.integers(0, n, size=n)
        boot_rates.append(strict_full[idx].mean())
    boot_rates = np.asarray(boot_rates)
    ci_low, ci_high = np.quantile(boot_rates, [0.025, 0.975])
    print(f'Bootstrap 95% CI: [{100 * ci_low:.2f}, {100 * ci_high:.2f}]%')
    perm_rates = np.empty(n_perm)
    for p in tqdm(range(n_perm), desc=f'{theory_name}: permutations', leave=True):
        perm_success = np.zeros(n, dtype=bool)
        for j, row in enumerate(vals):
            permuted = local_rng.permutation(row)
            perm_success[j] = np.all(np.diff(permuted) > 0)
        perm_rates[p] = perm_success.mean()
    null_mean = perm_rates.mean()
    perm_p = (1 + np.sum(perm_rates >= strict_rate)) / (n_perm + 1)
    print(f'Permutation null mean: {100 * null_mean:.2f}%')
    print(f'One-sided permutation p: {perm_p:.6f}')
    summary = {'narratives_all_mechanisms': n, 'strict_sequence_count': int(strict_full.sum()), 'strict_sequence_pct': 100 * strict_rate, 'weak_sequence_pct': 100 * weak_rate, 'non_tied_narratives': int(no_ties.sum()), 'strict_sequence_pct_non_tied': 100 * strict_non_tied_rate, 'bootstrap_ci_low_pct': 100 * ci_low, 'bootstrap_ci_high_pct': 100 * ci_high, 'permutation_null_pct': 100 * null_mean, 'permutation_p_one_sided': perm_p}
    return (summary, pd.DataFrame(step_rows), boot_rates, perm_rates)
theory_sequence_rows = []
theory_step_tables = []
sequence_bootstraps = {}
sequence_nulls = {}
for theory_i, (theory, sequence) in enumerate(tqdm(THEORY_SEQUENCES.items(), total=len(THEORY_SEQUENCES), desc='Theory sequences')):
    summary, steps, boots, nulls = evaluate_sequence(temporal_wide, sequence, n_boot=N_BOOT, n_perm=N_PERM, seed=RANDOM_STATE + theory_i, theory_name=theory)
    summary['theory'] = theory
    summary['sequence'] = ' → '.join((s.replace('mech_', '') for s in sequence))
    theory_sequence_rows.append(summary)
    steps['theory'] = theory
    theory_step_tables.append(steps)
    sequence_bootstraps[theory] = boots
    sequence_nulls[theory] = nulls
theory_sequence_results = pd.DataFrame(theory_sequence_rows)[['theory', 'sequence', 'narratives_all_mechanisms', 'strict_sequence_count', 'strict_sequence_pct', 'bootstrap_ci_low_pct', 'bootstrap_ci_high_pct', 'permutation_null_pct', 'permutation_p_one_sided', 'weak_sequence_pct', 'non_tied_narratives', 'strict_sequence_pct_non_tied']]
theory_sequence_steps = pd.concat(theory_step_tables, ignore_index=True)
print('\nTHEORY-LEVEL TEMPORAL SEQUENCE TESTS')
print('------------------------------------')
display(theory_sequence_results.round(3))
print('\nINDIVIDUAL STEPS WITHIN EACH THEORY')
print('-----------------------------------')
display(theory_sequence_steps.round(3))
theory_sequence_results.to_csv(OUTPUT_DIR / 'theory_temporal_sequence_tests.csv', index=False)
theory_sequence_steps.to_csv(OUTPUT_DIR / 'theory_temporal_sequence_steps.csv', index=False)
print('\nSaved:')
print(OUTPUT_DIR / 'theory_temporal_sequence_tests.csv')
print(OUTPUT_DIR / 'theory_temporal_sequence_steps.csv')


## Corpus And Appendix Statistics

In [ ]:
import pandas as pd
from pathlib import Path
print('=' * 80)
print('FINAL ANNOTATED CORPUS')
print('=' * 80)
print(f'Final fully annotated rows: {len(annotated):,}')
print(f"Unique doc_id values:       {annotated['doc_id'].nunique():,}")
print('\nRows by platform:')
display(annotated['platform'].value_counts().rename('rows').to_frame())
first_shard = sorted(annotated['_shard'].unique())[0]
first_path = source_path(first_shard)
schema_cols = pd.read_parquet(first_path).columns.tolist()
print('\nAvailable source columns:')
print(schema_cols)
USER_CANDIDATES = ['user_id', 'author_id', 'userid', 'user', 'author', 'account_id', 'username', 'author_username', 'screen_name']
user_col = next((c for c in USER_CANDIDATES if c in schema_cols), None)
if user_col is None:
    raise ValueError(f'Could not automatically identify the user column.\nAvailable columns are:\n{schema_cols}')
print(f'\nDetected user column: {user_col}')
user_frames = []
for shard in sorted(annotated['_shard'].unique()):
    path = source_path(shard)
    src = pd.read_parquet(path, columns=['id', user_col]).copy()
    src['_shard'] = shard
    src['doc_id'] = src['id'].astype(str).str.strip()
    src['_doc_occurrence'] = src.groupby(['_shard', 'doc_id'], sort=False).cumcount().astype(int)
    user_frames.append(src[['_shard', 'doc_id', '_doc_occurrence', user_col]])
source_users = pd.concat(user_frames, ignore_index=True)
paper_df = annotated.merge(source_users, on=['_shard', 'doc_id', '_doc_occurrence'], how='left', validate='one_to_one')
platform_map = {'tiktok': 'TikTok', 'twitter': 'Twitter/X', 'x': 'Twitter/X', 'truth social': 'Truth Social', 'truth_social': 'Truth Social', 'truthsocial': 'Truth Social'}
paper_df['paper_platform'] = paper_df['platform'].astype(str).str.strip().str.lower().map(platform_map).fillna(paper_df['platform'].astype(str))
platform_order = ['TikTok', 'Twitter/X', 'Truth Social']
platform_counts = paper_df.groupby('paper_platform').agg(Posts=('doc_id', 'size'), Users=(user_col, 'nunique')).reindex(platform_order)
total_posts = len(paper_df)
total_users = paper_df[['paper_platform', user_col]].dropna().drop_duplicates().shape[0]
platform_counts['Post_pct'] = 100 * platform_counts['Posts'] / total_posts
platform_counts['User_pct'] = 100 * platform_counts['Users'] / total_users
print('\n' + '=' * 80)
print('PAPER TABLE COUNTS')
print('=' * 80)
display(platform_counts[['Posts', 'Post_pct', 'Users', 'User_pct']].round(2))
print(f'\nTOTAL ANNOTATED POSTS/ROWS: {total_posts:,}')
print(f'TOTAL UNIQUE USERS:         {total_users:,}')
sanity = paper_df.groupby('paper_platform').agg(rows=('doc_id', 'size'), unique_doc_ids=('doc_id', 'nunique')).reindex(platform_order)
print('\nRows vs unique doc IDs:')
display(sanity)
print('\nOverall unique doc IDs:', f"{paper_df['doc_id'].nunique():,}")
print('\n' + '=' * 80)
print('LATEX')
print('=' * 80)
for platform in platform_order:
    r = platform_counts.loc[platform]
    print(f"{platform:<12} & {int(r['Posts']):,} ({r['Post_pct']:.1f}\\%) & {int(r['Users']):,} ({r['User_pct']:.1f}\\%) \\\\")
print('\\midrule')
print(f'\\textbf{{Total}} & \\textbf{{{total_posts:,}}} & \\textbf{{{total_users:,}}} \\\\')


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from IPython.display import display
B = 'mech_boundary_construction'
T = 'mech_threat_construction'
d = annotated[[B, T]].dropna().copy()
d[B] = d[B].astype(int)
d[T] = d[T].astype(int)
N = len(d)
print('=' * 80)
print('BOUNDARY vs THREAT — LABEL DISTINCTNESS CHECK')
print('=' * 80)
print(f'N = {N:,}')
ct = pd.crosstab(d[B], d[T], rownames=['Boundary'], colnames=['Threat'])
ct = ct.reindex(index=[0, 1], columns=[0, 1], fill_value=0)
print('\n2x2 contingency table:')
display(ct)
n00 = int(ct.loc[0, 0])
n01 = int(ct.loc[0, 1])
n10 = int(ct.loc[1, 0])
n11 = int(ct.loc[1, 1])
summary = pd.DataFrame({'Category': ['Neither', 'Boundary only', 'Threat only', 'Both'], 'N': [n00, n10, n01, n11]})
summary['Percent'] = 100 * summary['N'] / N
print('\nOverlap categories:')
display(summary.style.format({'N': '{:,}', 'Percent': '{:.3f}'}))
boundary_positive = n10 + n11
threat_positive = n01 + n11
union = n10 + n01 + n11
boundary_prev = boundary_positive / N
threat_prev = threat_positive / N
p_threat_given_boundary = n11 / boundary_positive if boundary_positive > 0 else np.nan
p_boundary_given_threat = n11 / threat_positive if threat_positive > 0 else np.nan
jaccard = n11 / union if union > 0 else np.nan
a = float(n11)
b = float(n10)
c = float(n01)
d0 = float(n00)
denom = np.sqrt((a + b) * (c + d0) * (a + c) * (b + d0))
phi = (a * d0 - b * c) / denom if denom > 0 else np.nan
overall_disagreement = (n10 + n01) / N
discordant_within_union = (n10 + n01) / union if union > 0 else np.nan
both_within_union = n11 / union if union > 0 else np.nan
if min(n00, n01, n10, n11) == 0:
    aa = n11 + 0.5
    bb = n10 + 0.5
    cc = n01 + 0.5
    dd = n00 + 0.5
else:
    aa = float(n11)
    bb = float(n10)
    cc = float(n01)
    dd = float(n00)
raw_or = aa * dd / (bb * cc)
chi2, p_chi2, dof, expected = chi2_contingency(ct)
metrics = pd.DataFrame({'Metric': ['Boundary prevalence', 'Threat prevalence', 'P(Threat | Boundary)', 'P(Boundary | Threat)', 'Jaccard similarity', 'Phi coefficient', 'Overall label disagreement', 'Discordant among Boundary-or-Threat posts', 'Both among Boundary-or-Threat posts', 'Raw pairwise odds ratio'], 'Value': [boundary_prev, threat_prev, p_threat_given_boundary, p_boundary_given_threat, jaccard, phi, overall_disagreement, discordant_within_union, both_within_union, raw_or]})
print('\n' + '=' * 80)
print('OVERLAP / DISTINCTNESS METRICS')
print('=' * 80)
display(metrics.round(4))
print('\n' + '=' * 80)
print('REVIEWER-FACING DISTINCTNESS NUMBERS')
print('=' * 80)
print(f'Boundary-positive posts: {boundary_positive:,} ({100 * boundary_prev:.2f}% of all posts)')
print(f'Threat-positive posts:   {threat_positive:,} ({100 * threat_prev:.2f}% of all posts)')
print()
print(f'Boundary only: {n10:,} ({100 * n10 / N:.2f}% of all posts)')
print(f'Threat only:   {n01:,} ({100 * n01 / N:.2f}% of all posts)')
print(f'Both:          {n11:,} ({100 * n11 / N:.2f}% of all posts)')
print()
print(f'Among posts containing Boundary OR Threat, {100 * discordant_within_union:.2f}% contain only one of the two mechanisms.')
print(f'Among posts containing Boundary OR Threat, {100 * both_within_union:.2f}% contain both.')
print()
print(f'P(Threat | Boundary) = {100 * p_threat_given_boundary:.2f}%')
print(f'P(Boundary | Threat) = {100 * p_boundary_given_threat:.2f}%')
print()
print(f'Jaccard similarity = {jaccard:.3f}')
print(f'Phi coefficient    = {phi:.3f}')
print(f'Raw odds ratio     = {raw_or:.3f}')
print(f'Chi-square p-value = {p_chi2:.3e}')
TEXT_CANDIDATES = ['status', 'text', 'claim']
text_col = next((c for c in TEXT_CANDIDATES if c in annotated.columns), None)
if text_col is not None:
    cols_to_show = [c for c in ['doc_id', 'platform', text_col, B, T] if c in annotated.columns]
    boundary_only_mask = (annotated[B] == 1) & (annotated[T] == 0)
    threat_only_mask = (annotated[B] == 0) & (annotated[T] == 1)
    boundary_only_examples = annotated.loc[boundary_only_mask, cols_to_show].sample(n=min(20, int(boundary_only_mask.sum())), random_state=42)
    threat_only_examples = annotated.loc[threat_only_mask, cols_to_show].sample(n=min(20, int(threat_only_mask.sum())), random_state=42)
    print('\n' + '=' * 80)
    print('RANDOM BOUNDARY-ONLY EXAMPLES')
    print('=' * 80)
    display(boundary_only_examples)
    print('\n' + '=' * 80)
    print('RANDOM THREAT-ONLY EXAMPLES')
    print('=' * 80)
    display(threat_only_examples)
else:
    print('\nNo text column found directly in `annotated`; quantitative distinctness metrics computed successfully.')


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
from IPython.display import display
MECHANISMS = ['mech_boundary_construction', 'mech_threat_construction', 'mech_scapegoating', 'mech_negative_evaluation', 'mech_dehumanization', 'mech_action_orientation']
PLATFORM_ORDER = ['twitter', 'tiktok', 'truth_social']
PLATFORM_LABELS = {'twitter': 'Twitter/X', 'tiktok': 'TikTok', 'truth_social': 'Truth Social'}
MECH_LABELS = {'mech_boundary_construction': 'Boundary construction', 'mech_threat_construction': 'Threat construction', 'mech_scapegoating': 'Scapegoating', 'mech_negative_evaluation': 'Negative evaluation', 'mech_dehumanization': 'Dehumanization', 'mech_action_orientation': 'Action orientation'}
missing = [m for m in MECHANISMS if m not in annotated.columns]
if missing:
    raise ValueError(f'Missing mechanism columns: {missing}')
d = annotated[['platform'] + MECHANISMS].copy()
d = d[d['platform'].isin(PLATFORM_ORDER)].copy()
for m in MECHANISMS:
    d[m] = pd.to_numeric(d[m], errors='coerce')
print('=' * 80)
print('PLATFORM HETEROGENEITY ANALYSIS')
print('=' * 80)
print(f'Total rows: {len(d):,}')
print('\nRows by platform:')
display(d['platform'].value_counts().reindex(PLATFORM_ORDER).rename(index=PLATFORM_LABELS).to_frame('N'))
prevalence_rows = []
for mech in MECHANISMS:
    for platform in PLATFORM_ORDER:
        x = d.loc[d['platform'] == platform, mech].dropna()
        n = len(x)
        positives = int(x.sum())
        rate = positives / n if n else np.nan
        prevalence_rows.append({'mechanism': mech, 'mechanism_label': MECH_LABELS[mech], 'platform': platform, 'platform_label': PLATFORM_LABELS[platform], 'n': n, 'positive': positives, 'prevalence': rate, 'prevalence_pct': 100 * rate})
platform_prevalence = pd.DataFrame(prevalence_rows)
prevalence_wide = platform_prevalence.pivot(index='mechanism_label', columns='platform_label', values='prevalence_pct').reindex(index=[MECH_LABELS[m] for m in MECHANISMS])
prevalence_wide = prevalence_wide[['TikTok', 'Twitter/X', 'Truth Social']]
print('\n' + '=' * 80)
print('PREVALENCE (%) BY PLATFORM')
print('=' * 80)
display(prevalence_wide.round(2))
heterogeneity_rows = []
for mech in CANONICAL_MECH_COLS:
    contingency = []
    rates = []
    for platform in PLATFORM_ORDER:
        x = d.loc[d['platform'] == platform, mech].dropna()
        pos = int(x.sum())
        neg = int(len(x) - pos)
        contingency.append([neg, pos])
        rates.append(pos / len(x) if len(x) else np.nan)
    contingency = np.asarray(contingency, dtype=float)
    chi2, p, dof, expected = chi2_contingency(contingency)
    n_total = contingency.sum()
    cramers_v = np.sqrt(chi2 / n_total)
    rate_spread_pp = 100 * (np.nanmax(rates) - np.nanmin(rates))
    heterogeneity_rows.append({'mechanism': mech, 'mechanism_label': MECH_LABELS[mech], 'chi2': chi2, 'df': dof, 'p': p, 'cramers_v': cramers_v, 'max_min_spread_pp': rate_spread_pp})
platform_heterogeneity = pd.DataFrame(heterogeneity_rows)
platform_heterogeneity['q_fdr'] = multipletests(platform_heterogeneity['p'], method='fdr_bh')[1]
print('\n' + '=' * 80)
print('OMNIBUS PLATFORM HETEROGENEITY')
print('=' * 80)
display(platform_heterogeneity[['mechanism_label', 'chi2', 'df', 'p', 'q_fdr', 'cramers_v', 'max_min_spread_pp']].round(4))
platform_pairs = [('tiktok', 'twitter'), ('tiktok', 'truth_social'), ('twitter', 'truth_social')]
pairwise_rows = []
for mech in MECHANISMS:
    for p1, p2 in platform_pairs:
        x1 = d.loc[d['platform'] == p1, mech].dropna()
        x2 = d.loc[d['platform'] == p2, mech].dropna()
        pos1 = int(x1.sum())
        pos2 = int(x2.sum())
        n1 = len(x1)
        n2 = len(x2)
        rate1 = pos1 / n1
        rate2 = pos2 / n2
        stat, p = proportions_ztest(count=[pos1, pos2], nobs=[n1, n2])
        pairwise_rows.append({'mechanism': mech, 'mechanism_label': MECH_LABELS[mech], 'platform_1': PLATFORM_LABELS[p1], 'platform_2': PLATFORM_LABELS[p2], 'rate_1_pct': 100 * rate1, 'rate_2_pct': 100 * rate2, 'difference_pp': 100 * (rate1 - rate2), 'z': stat, 'p': p})
platform_pairwise = pd.DataFrame(pairwise_rows)
platform_pairwise['q_fdr'] = multipletests(platform_pairwise['p'], method='fdr_bh')[1]
print('\n' + '=' * 80)
print('PAIRWISE PLATFORM DIFFERENCES')
print('=' * 80)
display(platform_pairwise[['mechanism_label', 'platform_1', 'platform_2', 'rate_1_pct', 'rate_2_pct', 'difference_pp', 'p', 'q_fdr']].round(4))
appendix_table = prevalence_wide.reset_index().rename(columns={'mechanism_label': 'Mechanism'})
effect_map = platform_heterogeneity.set_index('mechanism_label')[['cramers_v', 'max_min_spread_pp', 'q_fdr']]
appendix_table = appendix_table.set_index('Mechanism').join(effect_map).reset_index()
print('\n' + '=' * 80)
print('APPENDIX-READY TABLE')
print('=' * 80)
display(appendix_table.round({'TikTok': 2, 'Twitter/X': 2, 'Truth Social': 2, 'cramers_v': 3, 'max_min_spread_pp': 2, 'q_fdr': 4}))
print('\n' + '=' * 80)
print('LATEX ROWS')
print('=' * 80)
for _, r in appendix_table.iterrows():
    q = r['q_fdr']
    if q < 0.001:
        q_str = '$<.001$'
    else:
        q_str = f'{q:.3f}'
    print(f"{r['Mechanism']} & {r['TikTok']:.2f}\\% & {r['Twitter/X']:.2f}\\% & {r['Truth Social']:.2f}\\% & {r['cramers_v']:.3f} & {q_str} \\\\")
platform_prevalence.to_csv(OUTPUT_DIR / 'platform_mechanism_prevalence.csv', index=False)
platform_heterogeneity.to_csv(OUTPUT_DIR / 'platform_mechanism_heterogeneity.csv', index=False)
platform_pairwise.to_csv(OUTPUT_DIR / 'platform_mechanism_pairwise.csv', index=False)
appendix_table.to_csv(OUTPUT_DIR / 'platform_mechanism_appendix_table.csv', index=False)
print('\nSaved platform heterogeneity outputs.')


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
df = annotated.copy()

def first_existing(candidates, columns):
    for c in candidates:
        if c in columns:
            return c
    return None
NARRATIVE_COL = first_existing(['cluster', 'cluster_id', 'narrative', 'narrative_id', 'narrative_cluster'], df.columns)
PLATFORM_COL = first_existing(['platform', 'source_platform'], df.columns)
TEXT_COL = first_existing(['claim', 'text', 'status'], df.columns)
TIMESTAMP_COL = first_existing(['timestamp', 'created_at', 'date'], df.columns)
print('Narrative column:', NARRATIVE_COL)
print('Platform column:', PLATFORM_COL)
print('Text/claim column:', TEXT_COL)
print('Timestamp column:', TIMESTAMP_COL)
assert NARRATIVE_COL is not None
assert PLATFORM_COL is not None
x = df.dropna(subset=[NARRATIVE_COL, PLATFORM_COL]).copy()
x[PLATFORM_COL] = x[PLATFORM_COL].astype(str).str.lower().replace({'x': 'twitter', 'twitter/x': 'twitter', 'truth social': 'truth_social', 'truthsocial': 'truth_social'})
narrative_summary = x.groupby(NARRATIVE_COL).agg(n_posts=(NARRATIVE_COL, 'size'), n_platforms=(PLATFORM_COL, 'nunique')).reset_index()
platform_sets = x.groupby(NARRATIVE_COL)[PLATFORM_COL].agg(lambda s: tuple(sorted(set(s)))).rename('platforms').reset_index()
narrative_summary = narrative_summary.merge(platform_sets, on=NARRATIVE_COL, how='left')
N = len(narrative_summary)
N_MULTI = (narrative_summary['n_platforms'] >= 2).sum()
N_ALL3 = (narrative_summary['n_platforms'] >= 3).sum()
PCT_MULTI = 100 * N_MULTI / N
q1 = narrative_summary['n_posts'].quantile(0.25)
median = narrative_summary['n_posts'].median()
q3 = narrative_summary['n_posts'].quantile(0.75)
print('\n' + '=' * 70)
print('NARRATIVE SUMMARY')
print('=' * 70)
print(f'Total narratives: {N:,}')
print(f'Multi-platform narratives: {N_MULTI:,} ({PCT_MULTI:.1f}%)')
print(f'All-three-platform narratives: {N_ALL3:,} ({100 * N_ALL3 / N:.1f}%)')
print(f'Median posts per narrative: {median:.0f} (IQR {q1:.0f}–{q3:.0f})')
print('\nNarratives by number of platforms:')
display(narrative_summary['n_platforms'].value_counts().sort_index().rename_axis('n_platforms').to_frame('narratives'))
print('\nMost common platform combinations:')
display(narrative_summary['platforms'].value_counts().head(20).rename_axis('platforms').to_frame('narratives'))


In [ ]:
from pathlib import Path
import pandas as pd
OUT = OUTPUT_DIR
temporal = pd.read_csv(OUT / 'temporal_first_appearance.csv')
print('Temporal columns:')
print(temporal.columns.tolist())
TEMP_NARRATIVE_COL = first_existing([NARRATIVE_COL, 'cluster', 'cluster_id', 'narrative', 'narrative_id', 'narrative_cluster'], temporal.columns)
print('\nTemporal narrative ID column:', TEMP_NARRATIVE_COL)
assert TEMP_NARRATIVE_COL is not None
used_ids = set(temporal[TEMP_NARRATIVE_COL].dropna().unique())
used = x[x[NARRATIVE_COL].isin(used_ids)].copy()
used_summary = used.groupby(NARRATIVE_COL).agg(n_posts=(NARRATIVE_COL, 'size'), n_platforms=(PLATFORM_COL, 'nunique')).reset_index()
print('\n' + '=' * 70)
print('TEMPORAL-ANALYSIS NARRATIVES')
print('=' * 70)
print(f'N narratives: {used_summary[NARRATIVE_COL].nunique():,}')
print(f'Cross-platform: {(used_summary.n_platforms >= 2).sum():,} ({100 * (used_summary.n_platforms >= 2).mean():.1f}%)')
print(f'All 3 platforms: {(used_summary.n_platforms == 3).sum():,} ({100 * (used_summary.n_platforms == 3).mean():.1f}%)')
print('Posts/narrative median:', f'{used_summary.n_posts.median():.0f}')
print('Posts/narrative IQR:', f'{used_summary.n_posts.quantile(0.25):.0f}–{used_summary.n_posts.quantile(0.75):.0f}')


## Left-Censoring Robustness

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.display import display
CLUSTER_COL = 'cluster'
TIME_COL = 'timestamp'
MECH_COLS_TEMPORAL = ['mech_boundary_construction', 'mech_threat_construction', 'mech_scapegoating', 'mech_negative_evaluation', 'mech_dehumanization', 'mech_action_orientation']
MECH_LABELS_TEMPORAL = {'mech_boundary_construction': 'Boundary construction', 'mech_threat_construction': 'Threat construction', 'mech_scapegoating': 'Scapegoating', 'mech_negative_evaluation': 'Negative evaluation', 'mech_dehumanization': 'Dehumanization', 'mech_action_orientation': 'Action orientation'}
MECH_ORDER_DISPLAY = ['Negative evaluation', 'Boundary construction', 'Action orientation', 'Dehumanization', 'Threat construction', 'Scapegoating']
COLD_WINDOWS = [0, 7, 14, 21, 30]
COLD_WINDOWS = list(range(0, 31))
MIN_NARRATIVE_POSTS = 10
EARLY_WINDOW_DAYS = 7
MIN_EARLY_POSTS = 5
MIN_EARLY_SHARE = 0.25
MAX_PEAK_DAY = 7
LATE_START_DAY = 14
MAX_LATE_TO_EARLY_RATIO = 0.5
needed = [CLUSTER_COL, TIME_COL] + MECH_COLS_TEMPORAL
missing = [c for c in needed if c not in annotated.columns]
if missing:
    raise KeyError(f'annotated missing required columns: {missing}')
temp_df = annotated[needed].copy()
temp_df[TIME_COL] = pd.to_datetime(temp_df[TIME_COL], errors='coerce', utc=True)
temp_df = temp_df.dropna(subset=[CLUSTER_COL, TIME_COL]).sort_values([CLUSTER_COL, TIME_COL]).reset_index(drop=True)
for mech in MECH_COLS_TEMPORAL:
    temp_df[mech] = pd.to_numeric(temp_df[mech], errors='coerce').fillna(0).astype(np.int8)
DATA_START = temp_df[TIME_COL].min()
DATA_END = temp_df[TIME_COL].max()
print('=' * 80)
print('OBSERVATION WINDOW')
print('=' * 80)
print('Start:', DATA_START)
print('End:  ', DATA_END)
print('Length:', f'{(DATA_END - DATA_START).days:,} days')
narrative_summary = temp_df.groupby(CLUSTER_COL).agg(first_post=(TIME_COL, 'min'), last_post=(TIME_COL, 'max'), n_posts=(TIME_COL, 'size')).reset_index()
narrative_summary = narrative_summary[narrative_summary['n_posts'] >= MIN_NARRATIVE_POSTS].copy()
narrative_summary['days_from_dataset_start'] = (narrative_summary['first_post'] - DATA_START).dt.total_seconds() / 86400
narrative_summary['duration_days'] = (narrative_summary['last_post'] - narrative_summary['first_post']).dt.total_seconds() / 86400
print('\n' + '=' * 80)
print('NARRATIVE UNIVERSE')
print('=' * 80)
print('Narratives >= 10 posts:', f'{len(narrative_summary):,}')
for window in COLD_WINDOWS:
    narrative_summary[f'cold_{window}d'] = narrative_summary['days_from_dataset_start'] >= window
print('\n' + '=' * 80)
print('COLD-START NARRATIVE COUNTS')
print('=' * 80)
cold_counts = []
for window in COLD_WINDOWS:
    n = int(narrative_summary[f'cold_{window}d'].sum())
    cold_counts.append({'minimum_pre_observation_days': window, 'n_narratives': n, 'pct_of_narratives': 100 * n / len(narrative_summary)})
cold_counts = pd.DataFrame(cold_counts)
display(cold_counts.round(2))
eligible_clusters = set(narrative_summary[CLUSTER_COL])
activity_df = temp_df[temp_df[CLUSTER_COL].isin(eligible_clusters)].copy()
first_post_lookup = narrative_summary.set_index(CLUSTER_COL)['first_post']
activity_df['narrative_first_post'] = activity_df[CLUSTER_COL].map(first_post_lookup)
activity_df['day_since_onset'] = ((activity_df[TIME_COL] - activity_df['narrative_first_post']).dt.total_seconds() / 86400).apply(np.floor).astype(int)
daily_counts = activity_df.groupby([CLUSTER_COL, 'day_since_onset']).size().rename('posts').reset_index()
burst_rows = []
for cluster, g in tqdm(daily_counts.groupby(CLUSTER_COL), desc='Characterizing narrative onset'):
    g = g.sort_values('day_since_onset')
    total_posts = int(g['posts'].sum())
    early = g[g['day_since_onset'] < EARLY_WINDOW_DAYS]
    early_posts = int(early['posts'].sum())
    early_share = early_posts / total_posts if total_posts > 0 else np.nan
    early_mean_daily = early_posts / EARLY_WINDOW_DAYS
    peak_idx = g['posts'].idxmax()
    peak_day = int(g.loc[peak_idx, 'day_since_onset'])
    peak_posts = int(g.loc[peak_idx, 'posts'])
    max_day = int(g['day_since_onset'].max())
    late = g[g['day_since_onset'] >= LATE_START_DAY]
    late_posts = int(late['posts'].sum())
    if max_day >= LATE_START_DAY:
        n_late_days = max_day - LATE_START_DAY + 1
        late_mean_daily = late_posts / n_late_days
    else:
        n_late_days = 0
        late_mean_daily = np.nan
    if np.isfinite(late_mean_daily) and early_mean_daily > 0:
        late_to_early_ratio = late_mean_daily / early_mean_daily
    else:
        late_to_early_ratio = np.nan
    burst_rows.append({CLUSTER_COL: cluster, 'early_posts': early_posts, 'early_share': early_share, 'early_mean_daily': early_mean_daily, 'peak_day': peak_day, 'peak_posts': peak_posts, 'late_posts': late_posts, 'late_mean_daily': late_mean_daily, 'late_to_early_ratio': late_to_early_ratio, 'max_day': max_day})
burst_features = pd.DataFrame(burst_rows)
narrative_summary = narrative_summary.merge(burst_features, on=CLUSTER_COL, how='left', validate='one_to_one')
narrative_summary['burst_onset'] = narrative_summary['cold_30d'] & (narrative_summary['early_posts'] >= MIN_EARLY_POSTS) & (narrative_summary['early_share'] >= MIN_EARLY_SHARE) & (narrative_summary['peak_day'] <= MAX_PEAK_DAY) & (narrative_summary['max_day'] >= LATE_START_DAY) & (narrative_summary['late_to_early_ratio'] <= MAX_LATE_TO_EARLY_RATIO)
print('\n' + '=' * 80)
print('BURST-ONSET NARRATIVES')
print('=' * 80)
print('N:', f"{narrative_summary['burst_onset'].sum():,}")
print('Percent of eligible narratives:', f"{100 * narrative_summary['burst_onset'].mean():.2f}%")
display(narrative_summary.loc[narrative_summary['burst_onset'], [CLUSTER_COL, 'n_posts', 'days_from_dataset_start', 'duration_days', 'early_posts', 'early_share', 'peak_day', 'late_to_early_ratio']].describe().round(3))
temporal_posts = activity_df.sort_values([CLUSTER_COL, TIME_COL]).copy()
temporal_posts['_post_idx'] = temporal_posts.groupby(CLUSTER_COL).cumcount()
temporal_posts['_n_posts'] = temporal_posts.groupby(CLUSTER_COL)[TIME_COL].transform('size')
temporal_posts['_norm_position'] = np.where(temporal_posts['_n_posts'] > 1, temporal_posts['_post_idx'] / (temporal_posts['_n_posts'] - 1), 0.0)
first_appearance_rows = []
for mech in tqdm(MECH_COLS_TEMPORAL, desc='First appearances'):
    pos = temporal_posts[temporal_posts[mech] == 1]
    first = pos.sort_values([CLUSTER_COL, TIME_COL]).groupby(CLUSTER_COL, as_index=False).first()
    tmp = first[[CLUSTER_COL, '_norm_position']].copy()
    tmp['mechanism'] = MECH_LABELS_TEMPORAL[mech]
    tmp = tmp.rename(columns={'_norm_position': 'first_norm'})
    first_appearance_rows.append(tmp)
first_appearance = pd.concat(first_appearance_rows, ignore_index=True)
subset_cols = [CLUSTER_COL] + [f'cold_{w}d' for w in COLD_WINDOWS] + ['burst_onset']
first_appearance = first_appearance.merge(narrative_summary[subset_cols], on=CLUSTER_COL, how='left', validate='many_to_one')
timing_results = []
for window in COLD_WINDOWS:
    subset_name = 'Full sample' if window == 0 else f'{window}-day cold start'
    col = f'cold_{window}d'
    d = first_appearance[first_appearance[col]]
    summary = d.groupby('mechanism')['first_norm'].agg(mean_first='mean', median_first='median', n='size').reset_index()
    summary['subset'] = subset_name
    summary['window_days'] = window
    timing_results.append(summary)
d_burst = first_appearance[first_appearance['burst_onset']]
burst_summary = d_burst.groupby('mechanism')['first_norm'].agg(mean_first='mean', median_first='median', n='size').reset_index()
burst_summary['subset'] = 'Burst onset'
burst_summary['window_days'] = 31
timing_results.append(burst_summary)
timing_results = pd.concat(timing_results, ignore_index=True)
timing_results['rank'] = timing_results.groupby('subset')['mean_first'].rank(method='min', ascending=True).astype(int)
print('\n' + '=' * 80)
print('TEMPORAL ORDERING BY LEFT-CENSORING REQUIREMENT')
print('=' * 80)
display(timing_results[['subset', 'mechanism', 'n', 'mean_first', 'median_first', 'rank']].sort_values(['window_days', 'rank']).round(4))
full_order = timing_results[timing_results['subset'] == 'Full sample'].sort_values('mean_first')['mechanism'].tolist()
rank_stability = []
for subset, g in timing_results.groupby('subset'):
    order = g.sort_values('mean_first')['mechanism'].tolist()
    full_rank = {mech: i + 1 for i, mech in enumerate(full_order)}
    this_rank = {mech: i + 1 for i, mech in enumerate(order)}
    common = [m for m in full_order if m in this_rank]
    if len(common) >= 2:
        x = np.array([full_rank[m] for m in common], dtype=float)
        y = np.array([this_rank[m] for m in common], dtype=float)
        spearman_rank = pd.Series(x).corr(pd.Series(y), method='spearman')
    else:
        spearman_rank = np.nan
    exact_order = order == full_order
    rank_stability.append({'subset': subset, 'ordering': ' < '.join(order), 'same_exact_order': exact_order, 'spearman_rank_vs_full': spearman_rank})
rank_stability = pd.DataFrame(rank_stability)
print('\n' + '=' * 80)
print('ORDER STABILITY')
print('=' * 80)
display(rank_stability)
plot_df = timing_results[timing_results['subset'] != 'Burst onset'].copy()
fig, ax = plt.subplots(figsize=(9, 5.5))
for mech in MECH_ORDER_DISPLAY:
    m = plot_df[plot_df['mechanism'] == mech].sort_values('window_days')
    ax.plot(m['window_days'], m['mean_first'], marker='o', linewidth=1.8, markersize=5, label=mech)
ax.set_xlabel('Required observation period before first narrative appearance (days)')
ax.set_ylabel('Mean normalized first-appearance position')
ax.set_xticks(COLD_WINDOWS)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()
compare_subsets = ['Full sample', '30-day cold start', 'Burst onset']
compare = timing_results[timing_results['subset'].isin(compare_subsets)].copy()
x = np.arange(len(MECH_ORDER_DISPLAY))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5.5))
for i, subset in enumerate(compare_subsets):
    g = compare[compare['subset'] == subset].set_index('mechanism').reindex(MECH_ORDER_DISPLAY)
    ax.bar(x + (i - 1) * width, g['mean_first'], width=width, label=subset)
ax.set_xticks(x)
ax.set_xticklabels(MECH_ORDER_DISPLAY, rotation=25, ha='right')
ax.set_ylabel('Mean normalized first-appearance position')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
try:
    timing_results.to_csv(OUTPUT_DIR / 'left_censoring_temporal_robustness.csv', index=False)
    narrative_summary.to_csv(OUTPUT_DIR / 'narrative_cold_start_burst_classification.csv', index=False)
    rank_stability.to_csv(OUTPUT_DIR / 'left_censoring_order_stability.csv', index=False)
    print('\nSaved robustness outputs.')
except NameError:
    print('\nOUTPUT_DIR not defined; results remain in memory.')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
print('\n' + '=' * 80)
print('TEMPORAL ORDERING BY LEFT-CENSORING REQUIREMENT')
print('=' * 80)
timing_display = timing_results.sort_values(['window_days', 'rank'])[['subset', 'mechanism', 'n', 'mean_first', 'median_first', 'rank']]
display(timing_display.round(4))
full_order = timing_results[timing_results['subset'] == 'Full sample'].sort_values('mean_first')['mechanism'].tolist()
print('\nFull-sample ordering:')
print(' < '.join(full_order))
rank_stability = []
for subset, g in timing_results.groupby('subset', sort=False):
    g = g.sort_values('mean_first')
    order = g['mechanism'].tolist()
    full_rank = {mech: i + 1 for i, mech in enumerate(full_order)}
    this_rank = {mech: i + 1 for i, mech in enumerate(order)}
    common = [mech for mech in full_order if mech in this_rank]
    if len(common) >= 2:
        x = pd.Series([full_rank[m] for m in common], dtype=float)
        y = pd.Series([this_rank[m] for m in common], dtype=float)
        rho = x.corr(y, method='spearman')
    else:
        rho = np.nan
    full_means = timing_results[timing_results['subset'] == 'Full sample'].set_index('mechanism')['mean_first']
    this_means = g.set_index('mechanism')['mean_first']
    common_means = full_means.index.intersection(this_means.index)
    mean_abs_shift = (this_means.loc[common_means] - full_means.loc[common_means]).abs().mean()
    max_abs_shift = (this_means.loc[common_means] - full_means.loc[common_means]).abs().max()
    rank_stability.append({'subset': subset, 'ordering': ' < '.join(order), 'same_exact_order': order == full_order, 'spearman_rank_vs_full': rho, 'mean_abs_timing_shift': mean_abs_shift, 'max_abs_timing_shift': max_abs_shift})
rank_stability = pd.DataFrame(rank_stability)
subset_order = {'Full sample': 0, '7-day cold start': 7, '14-day cold start': 14, '21-day cold start': 21, '30-day cold start': 30, 'Burst onset': 31}
rank_stability['_order'] = rank_stability['subset'].map(subset_order)
rank_stability = rank_stability.sort_values('_order').drop(columns='_order').reset_index(drop=True)
print('\n' + '=' * 80)
print('ORDER STABILITY')
print('=' * 80)
display(rank_stability.round(4))
timing_wide = timing_results.pivot(index='mechanism', columns='subset', values='mean_first')
desired_cols = ['Full sample', '7-day cold start', '14-day cold start', '21-day cold start', '30-day cold start', 'Burst onset']
desired_cols = [c for c in desired_cols if c in timing_wide.columns]
timing_wide = timing_wide[desired_cols].reindex(full_order)
print('\n' + '=' * 80)
print('MEAN NORMALIZED FIRST APPEARANCE')
print('=' * 80)
display(timing_wide.round(4))
cold_plot = timing_results[timing_results['subset'] != 'Burst onset'].copy()
fig, ax = plt.subplots(figsize=(9, 5.5))
for mech in full_order:
    g = cold_plot[cold_plot['mechanism'] == mech].sort_values('window_days')
    ax.plot(g['window_days'], g['mean_first'], marker='o', linewidth=1.8, markersize=5, label=mech)
ax.set_xlabel('Required observation before first narrative appearance (days)')
ax.set_ylabel('Mean normalized first-appearance position')
ax.set_xticks([0, 7, 14, 21, 30])
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()
compare_subsets = ['Full sample', '30-day cold start', 'Burst onset']
compare_subsets = [s for s in compare_subsets if s in timing_results['subset'].unique()]
compare = timing_results[timing_results['subset'].isin(compare_subsets)].copy()
x = np.arange(len(full_order))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5.5))
offsets = np.arange(len(compare_subsets)) - (len(compare_subsets) - 1) / 2
for offset, subset in zip(offsets, compare_subsets):
    g = compare[compare['subset'] == subset].set_index('mechanism').reindex(full_order)
    ax.bar(x + offset * width, g['mean_first'], width=width, label=subset)
ax.set_xticks(x)
ax.set_xticklabels(full_order, rotation=25, ha='right')
ax.set_ylabel('Mean normalized first-appearance position')
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
try:
    timing_results.to_csv(OUTPUT_DIR / 'left_censoring_temporal_robustness.csv', index=False)
    rank_stability.to_csv(OUTPUT_DIR / 'left_censoring_order_stability.csv', index=False)
    timing_wide.to_csv(OUTPUT_DIR / 'left_censoring_mean_first_appearance.csv')
    print('\nSaved robustness outputs.')
except NameError:
    print('\nOUTPUT_DIR not defined; results remain in memory.')


In [ ]:
overleaf_cold_start = timing_results[timing_results['window_days'].between(0, 30)].pivot(index='window_days', columns='mechanism', values='mean_first').rename(columns={'Boundary construction': 'boundary_construction', 'Action orientation': 'action_orientation', 'Threat construction': 'threat_construction', 'Negative evaluation': 'negative_evaluation', 'Dehumanization': 'dehumanization', 'Scapegoating': 'scapegoating'}).reset_index().sort_values('window_days')
display(overleaf_cold_start.head())
overleaf_cold_start.to_csv(OUTPUT_DIR / 'cold_start_temporal_plot.csv', index=False)
print('Saved cold_start_temporal_plot.csv')


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
CLUSTER_COL = 'cluster'
TIME_COL = 'timestamp'
MECH_COLS = ['mech_boundary_construction', 'mech_threat_construction', 'mech_scapegoating', 'mech_negative_evaluation', 'mech_dehumanization', 'mech_action_orientation']
MECH_LABELS = {'mech_boundary_construction': 'Boundary construction', 'mech_threat_construction': 'Threat construction', 'mech_scapegoating': 'Scapegoating', 'mech_negative_evaluation': 'Negative evaluation', 'mech_dehumanization': 'Dehumanization', 'mech_action_orientation': 'Action orientation'}
COLD_WINDOWS = list(range(0, 31))
MIN_NARRATIVE_POSTS = 10
cols_needed = [CLUSTER_COL, TIME_COL] + MECH_COLS
missing = [c for c in cols_needed if c not in annotated.columns]
if missing:
    raise KeyError(f'annotated missing required columns: {missing}')
df = annotated[cols_needed].copy()
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce', utc=True)
df = df.dropna(subset=[CLUSTER_COL, TIME_COL]).sort_values([CLUSTER_COL, TIME_COL]).reset_index(drop=True)
for mech in MECH_COLS:
    df[mech] = pd.to_numeric(df[mech], errors='coerce').fillna(0).astype(np.int8)
DATA_START = df[TIME_COL].min()
print('=' * 80)
print('DATA WINDOW')
print('=' * 80)
print('Start:', DATA_START)
print('End:  ', df[TIME_COL].max())
narratives = df.groupby(CLUSTER_COL).agg(first_post=(TIME_COL, 'min'), last_post=(TIME_COL, 'max'), n_posts=(TIME_COL, 'size')).reset_index()
narratives = narratives[narratives['n_posts'] >= MIN_NARRATIVE_POSTS].copy()
narratives['days_prior_observation'] = (narratives['first_post'] - DATA_START).dt.total_seconds() / 86400
eligible_clusters = set(narratives[CLUSTER_COL])
df = df[df[CLUSTER_COL].isin(eligible_clusters)].copy()
print('\nEligible narratives:', f'{len(narratives):,}')
df = df.sort_values([CLUSTER_COL, TIME_COL]).copy()
df['_post_idx'] = df.groupby(CLUSTER_COL).cumcount()
df['_n_posts'] = df.groupby(CLUSTER_COL)[TIME_COL].transform('size')
df['_norm_position'] = df['_post_idx'] / (df['_n_posts'] - 1)
rows = []
for mech in tqdm(MECH_COLS, desc='Building first-appearance table'):
    pos = df[df[mech] == 1].copy()
    occurrence_counts = pos.groupby(CLUSTER_COL).size().rename('k_occurrences')
    first = pos.sort_values([CLUSTER_COL, TIME_COL]).groupby(CLUSTER_COL).first().reset_index()
    first['k_occurrences'] = first[CLUSTER_COL].map(occurrence_counts)
    tmp = first[[CLUSTER_COL, '_norm_position', '_n_posts', 'k_occurrences']].copy()
    tmp = tmp.rename(columns={'_norm_position': 'observed_first_norm', '_n_posts': 'n_posts'})
    tmp['mechanism'] = MECH_LABELS[mech]
    rows.append(tmp)
first_adjustment_df = pd.concat(rows, ignore_index=True)
first_adjustment_df = first_adjustment_df.merge(narratives[[CLUSTER_COL, 'days_prior_observation']], on=CLUSTER_COL, how='left', validate='many_to_one')
first_adjustment_df['log_k_occurrences'] = np.log1p(first_adjustment_df['k_occurrences'])
first_adjustment_df['log_n_posts'] = np.log1p(first_adjustment_df['n_posts'])
print('\nFirst-appearance observations:')
print(f'{len(first_adjustment_df):,}')
MECH_ORDER_MODEL = ['Boundary construction', 'Action orientation', 'Threat construction', 'Negative evaluation', 'Dehumanization', 'Scapegoating']

def fit_adjusted_timing(d):
    d = d.copy()
    d['mechanism'] = pd.Categorical(d['mechanism'], categories=MECH_ORDER_MODEL, ordered=False)
    mech_dummies = pd.get_dummies(d['mechanism'], prefix='mech', drop_first=True, dtype=float)
    X = pd.concat([mech_dummies, d[['log_k_occurrences', 'log_n_posts']].astype(float)], axis=1)
    X = sm.add_constant(X, has_constant='add').astype(float)
    y = d['observed_first_norm'].astype(float).to_numpy()
    groups = d[CLUSTER_COL].astype(str).to_numpy()
    model = sm.OLS(y, X.to_numpy(dtype=float)).fit(cov_type='cluster', cov_kwds={'groups': groups})
    params = pd.Series(model.params, index=X.columns)
    mean_log_k = d['log_k_occurrences'].mean()
    mean_log_n = d['log_n_posts'].mean()
    adjusted_rows = []
    for mechanism in MECH_ORDER_MODEL:
        row = pd.Series(0.0, index=X.columns)
        row['const'] = 1.0
        row['log_k_occurrences'] = mean_log_k
        row['log_n_posts'] = mean_log_n
        if mechanism != MECH_ORDER_MODEL[0]:
            dummy_name = f'mech_{mechanism}'
            if dummy_name in row.index:
                row[dummy_name] = 1.0
        pred = float(np.dot(row.to_numpy(dtype=float), params.to_numpy(dtype=float)))
        adjusted_rows.append({'mechanism': mechanism, 'adjusted_first': pred})
    result = pd.DataFrame(adjusted_rows)
    result['adjusted_rank'] = result['adjusted_first'].rank(method='min', ascending=True).astype(int)
    return result
all_results = []
for window in tqdm(COLD_WINDOWS, desc='Adjusted cold-start models'):
    d = first_adjustment_df[first_adjustment_df['days_prior_observation'] >= window].copy()
    adjusted = fit_adjusted_timing(d)
    adjusted['window_days'] = window
    adjusted['n_narratives'] = d[CLUSTER_COL].nunique()
    adjusted['n_mechanism_observations'] = len(d)
    all_results.append(adjusted)
adjusted_cold_start = pd.concat(all_results, ignore_index=True)
print('\n' + '=' * 80)
print('ADJUSTED TEMPORAL TIMING BY COLD-START THRESHOLD')
print('=' * 80)
display(adjusted_cold_start[['window_days', 'mechanism', 'adjusted_first', 'adjusted_rank', 'n_narratives']].sort_values(['window_days', 'adjusted_rank']).round(4))
full = adjusted_cold_start[adjusted_cold_start['window_days'] == 0].sort_values('adjusted_rank')
full_order = full['mechanism'].tolist()
stability_rows = []
for window, g in adjusted_cold_start.groupby('window_days'):
    g = g.sort_values('adjusted_rank')
    order = g['mechanism'].tolist()
    full_rank = {m: i + 1 for i, m in enumerate(full_order)}
    current_rank = {m: i + 1 for i, m in enumerate(order)}
    x = pd.Series([full_rank[m] for m in full_order], dtype=float)
    y = pd.Series([current_rank[m] for m in full_order], dtype=float)
    rho = x.corr(y, method='spearman')
    stability_rows.append({'window_days': window, 'same_exact_order': order == full_order, 'spearman_rho': rho, 'ordering': ' < '.join(order)})
adjusted_rank_stability = pd.DataFrame(stability_rows)
print('\n' + '=' * 80)
print('ADJUSTED RANK STABILITY')
print('=' * 80)
display(adjusted_rank_stability)
print('\nFull adjusted ordering:')
print(' < '.join(full_order))
print('\nNumber of thresholds with exact same ordering:', f"{adjusted_rank_stability['same_exact_order'].sum()}/{len(adjusted_rank_stability)}")
print('Minimum Spearman rho:', round(adjusted_rank_stability['spearman_rho'].min(), 4))
fig, ax = plt.subplots(figsize=(9, 5.5))
for mechanism in full_order:
    g = adjusted_cold_start[adjusted_cold_start['mechanism'] == mechanism].sort_values('window_days')
    ax.plot(g['window_days'], g['adjusted_first'], marker='o', markersize=3, linewidth=1.8, label=mechanism)
ax.set_xlim(-1.5, 30.5)
ax.set_xticks([0, 5, 10, 15, 20, 25, 30])
ax.set_xlabel('Required prior observation (days)')
ax.set_ylabel('Adjusted normalized first appearance')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(figsize=(9, 5.5))
for mechanism in full_order:
    g = adjusted_cold_start[adjusted_cold_start['mechanism'] == mechanism].sort_values('window_days')
    ax.plot(g['window_days'], g['adjusted_rank'], marker='o', markersize=3, linewidth=1.8, label=mechanism)
ax.set_xlim(-1.5, 30.5)
ax.set_ylim(6.25, 0.75)
ax.set_yticks(range(1, 7))
ax.set_yticklabels(['Rank 1', 'Rank 2', 'Rank 3', 'Rank 4', 'Rank 5', 'Rank 6'])
ax.set_xticks([0, 5, 10, 15, 20, 25, 30])
ax.set_xlabel('Required prior observation (days)')
ax.set_ylabel('Adjusted temporal rank')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()
overleaf_adjusted_position = adjusted_cold_start.pivot(index='window_days', columns='mechanism', values='adjusted_first').rename(columns={'Boundary construction': 'boundary_construction', 'Action orientation': 'action_orientation', 'Threat construction': 'threat_construction', 'Negative evaluation': 'negative_evaluation', 'Dehumanization': 'dehumanization', 'Scapegoating': 'scapegoating'}).reset_index().sort_values('window_days')
overleaf_adjusted_rank = adjusted_cold_start.pivot(index='window_days', columns='mechanism', values='adjusted_rank').rename(columns={'Boundary construction': 'boundary_construction', 'Action orientation': 'action_orientation', 'Threat construction': 'threat_construction', 'Negative evaluation': 'negative_evaluation', 'Dehumanization': 'dehumanization', 'Scapegoating': 'scapegoating'}).reset_index().sort_values('window_days')
print('\n' + '=' * 80)
print('OVERLEAF ADJUSTED POSITION CSV')
print('=' * 80)
display(overleaf_adjusted_position.head())
print('\n' + '=' * 80)
print('OVERLEAF ADJUSTED RANK CSV')
print('=' * 80)
display(overleaf_adjusted_rank.head())
adjusted_cold_start.to_csv(OUTPUT_DIR / 'adjusted_cold_start_temporal_robustness.csv', index=False)
adjusted_rank_stability.to_csv(OUTPUT_DIR / 'adjusted_cold_start_rank_stability.csv', index=False)
overleaf_adjusted_position.to_csv(OUTPUT_DIR / 'adjusted_cold_start_temporal_plot.csv', index=False)
overleaf_adjusted_rank.to_csv(OUTPUT_DIR / 'adjusted_cold_start_rank_plot.csv', index=False)
print('\nSaved.')


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
EARLY = ['Negative evaluation', 'Boundary construction', 'Action orientation']
INTERMEDIATE = ['Dehumanization', 'Threat construction']
LATE = ['Scapegoating']
tier_rows = []
for window, g in adjusted_cold_start.groupby('window_days'):
    vals = g.set_index('mechanism')['adjusted_first']
    early_vals = vals.reindex(EARLY)
    early_min = early_vals.min()
    early_max = early_vals.max()
    intermediate_vals = vals.reindex(INTERMEDIATE)
    intermediate_min = intermediate_vals.min()
    intermediate_max = intermediate_vals.max()
    late_val = vals.reindex(LATE).iloc[0]
    early_intermediate_gap = intermediate_min - early_max
    intermediate_late_gap = late_val - intermediate_max
    tier_rows.append({'window_days': window, 'early_min': early_min, 'early_max': early_max, 'intermediate_min': intermediate_min, 'intermediate_max': intermediate_max, 'late': late_val, 'early_intermediate_gap': early_intermediate_gap, 'intermediate_late_gap': intermediate_late_gap, 'early_before_intermediate': early_intermediate_gap > 0, 'intermediate_before_late': intermediate_late_gap > 0, 'full_three_tier_structure': early_intermediate_gap > 0 and intermediate_late_gap > 0})
tier_stability = pd.DataFrame(tier_rows)
print('=' * 80)
print('TEMPORAL TIER STABILITY ACROSS COLD-START THRESHOLDS')
print('=' * 80)
display(tier_stability.round(4))
n_thresholds = len(tier_stability)
n_full = int(tier_stability['full_three_tier_structure'].sum())
n_early_mid = int(tier_stability['early_before_intermediate'].sum())
n_mid_late = int(tier_stability['intermediate_before_late'].sum())
print('\n' + '=' * 80)
print('SUMMARY')
print('=' * 80)
print('Early < Intermediate:', f'{n_early_mid}/{n_thresholds}', f'({100 * n_early_mid / n_thresholds:.1f}%)')
print('Intermediate < Late:', f'{n_mid_late}/{n_thresholds}', f'({100 * n_mid_late / n_thresholds:.1f}%)')
print('Full 3-tier structure:', f'{n_full}/{n_thresholds}', f'({100 * n_full / n_thresholds:.1f}%)')
print('\nMinimum Early → Intermediate gap:', f"{tier_stability['early_intermediate_gap'].min():.4f}")
print('Mean Early → Intermediate gap:', f"{tier_stability['early_intermediate_gap'].mean():.4f}")
print('\nMinimum Intermediate → Late gap:', f"{tier_stability['intermediate_late_gap'].min():.4f}")
print('Mean Intermediate → Late gap:', f"{tier_stability['intermediate_late_gap'].mean():.4f}")
weak_early_mid = tier_stability.sort_values('early_intermediate_gap').head(5)
weak_mid_late = tier_stability.sort_values('intermediate_late_gap').head(5)
print('\n' + '=' * 80)
print('SMALLEST EARLY / INTERMEDIATE GAPS')
print('=' * 80)
display(weak_early_mid[['window_days', 'early_max', 'intermediate_min', 'early_intermediate_gap']].round(4))
print('\n' + '=' * 80)
print('SMALLEST INTERMEDIATE / LATE GAPS')
print('=' * 80)
display(weak_mid_late[['window_days', 'intermediate_max', 'late', 'intermediate_late_gap']].round(4))
try:
    tier_stability.to_csv(OUTPUT_DIR / 'adjusted_cold_start_tier_stability.csv', index=False)
    print('\nSaved.')
except NameError:
    pass


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from tqdm.auto import tqdm
from IPython.display import display
CLUSTER_COL = 'cluster'
MECH_ORDER = ['Negative evaluation', 'Boundary construction', 'Action orientation', 'Dehumanization', 'Threat construction', 'Scapegoating']
REFERENCE_MECH = 'Boundary construction'
EARLY = ['Negative evaluation', 'Boundary construction', 'Action orientation']
INTERMEDIATE = ['Dehumanization', 'Threat construction']
LATE = ['Scapegoating']
EQUIV_MARGIN = 0.02
EQUIV_MARGINS = [0.01, 0.02, 0.03]
N_BOOT = 500
RANDOM_SEED = 42
required_cols = [CLUSTER_COL, 'mechanism', 'observed_first_norm', 'log_k_occurrences', 'log_n_posts', 'days_prior_observation']
missing = [c for c in required_cols if c not in first_adjustment_df.columns]
if missing:
    raise KeyError(f'first_adjustment_df missing: {missing}')
d_all = first_adjustment_df[required_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
d_all['mechanism'] = pd.Categorical(d_all['mechanism'], categories=[REFERENCE_MECH] + [m for m in MECH_ORDER if m != REFERENCE_MECH])
d_all['prior_obs_30'] = d_all['days_prior_observation'] / 30.0
print('=' * 80)
print('STATISTICAL ROBUSTNESS OF COLD-START TIMING')
print('=' * 80)
print('Mechanism observations:', f'{len(d_all):,}')
print('Narratives:', f'{d_all[CLUSTER_COL].nunique():,}')
mech_dummies = pd.get_dummies(d_all['mechanism'], prefix='mech', drop_first=True, dtype=float)
X_parts = [mech_dummies, d_all[['log_k_occurrences', 'log_n_posts', 'prior_obs_30']].astype(float)]
interaction_names = []
for col in mech_dummies.columns:
    interaction_col = f'{col}:prior_obs_30'
    interaction_names.append(interaction_col)
    X_parts.append(pd.DataFrame({interaction_col: mech_dummies[col] * d_all['prior_obs_30']}))
X = pd.concat(X_parts, axis=1)
X = sm.add_constant(X, has_constant='add').astype(float)
y = d_all['observed_first_norm'].astype(float).to_numpy()
groups = d_all[CLUSTER_COL].astype(str).to_numpy()
interaction_model = sm.OLS(y, X.to_numpy(dtype=float)).fit(cov_type='cluster', cov_kwds={'groups': groups})
param_names = X.columns.tolist()
params = pd.Series(interaction_model.params, index=param_names)
ses = pd.Series(interaction_model.bse, index=param_names)
pvals = pd.Series(interaction_model.pvalues, index=param_names)
interaction_idx = [param_names.index(name) for name in interaction_names]
R = np.zeros((len(interaction_idx), len(param_names)))
for row_i, param_i in enumerate(interaction_idx):
    R[row_i, param_i] = 1.0
wald = interaction_model.wald_test(R, scalar=True)
print('\n' + '=' * 80)
print('GLOBAL MECHANISM x PRIOR-OBSERVATION TEST')
print('=' * 80)
print('Wald statistic:', round(float(wald.statistic), 4))
print('df:', len(interaction_names))
print('p-value:', f'{float(wald.pvalue):.6g}')
interaction_results = []
for name in interaction_names:
    mechanism = name.replace('mech_', '').replace(':prior_obs_30', '')
    interaction_results.append({'mechanism_vs_boundary': mechanism, 'delta_relative_position_per_30d': float(params[name]), 'se': float(ses[name]), 'p': float(pvals[name])})
interaction_results = pd.DataFrame(interaction_results)
print('\nDifferential change in timing relative to Boundary construction over 30 days:')
display(interaction_results.round(4))

def adjusted_positions(data, cluster_weights=None):
    d = data.copy()
    if cluster_weights is None:
        d['_weight'] = 1.0
    else:
        d['_weight'] = d[CLUSTER_COL].map(cluster_weights).fillna(0).astype(float)
        d = d[d['_weight'] > 0].copy()
    categories = [REFERENCE_MECH] + [m for m in MECH_ORDER if m != REFERENCE_MECH]
    d['mechanism'] = pd.Categorical(d['mechanism'], categories=categories)
    mech_d = pd.get_dummies(d['mechanism'], prefix='mech', drop_first=True, dtype=float)
    X = pd.concat([mech_d, d[['log_k_occurrences', 'log_n_posts']].astype(float)], axis=1)
    X = sm.add_constant(X, has_constant='add').astype(float)
    y = d['observed_first_norm'].astype(float).to_numpy()
    weights = d['_weight'].astype(float).to_numpy()
    fit = sm.WLS(y, X.to_numpy(dtype=float), weights=weights).fit()
    coef = pd.Series(fit.params, index=X.columns)
    mean_log_k = np.average(d['log_k_occurrences'], weights=weights)
    mean_log_n = np.average(d['log_n_posts'], weights=weights)
    adjusted = {}
    for mech in MECH_ORDER:
        row = pd.Series(0.0, index=X.columns)
        row['const'] = 1.0
        row['log_k_occurrences'] = mean_log_k
        row['log_n_posts'] = mean_log_n
        if mech != REFERENCE_MECH:
            dummy = f'mech_{mech}'
            if dummy in row.index:
                row[dummy] = 1.0
        adjusted[mech] = float(np.dot(row.to_numpy(dtype=float), coef.to_numpy(dtype=float)))
    return pd.Series(adjusted)
d0 = d_all.copy()
d30 = d_all[d_all['days_prior_observation'] >= 30].copy()
pos0 = adjusted_positions(d0)
pos30 = adjusted_positions(d30)
rel0 = pos0 - pos0.mean()
rel30 = pos30 - pos30.mean()
observed_change = rel30 - rel0
relative_change_table = pd.DataFrame({'full_adjusted_position': pos0, 'day30_adjusted_position': pos30, 'full_relative_position': rel0, 'day30_relative_position': rel30, 'relative_change_30_minus_0': observed_change})
print('\n' + '=' * 80)
print('FULL SAMPLE VS 30-DAY RELATIVE TIMING')
print('=' * 80)
display(relative_change_table.round(4))
print('\nMaximum absolute relative change:', round(observed_change.abs().max(), 4))
print('Mean absolute relative change:', round(observed_change.abs().mean(), 4))

def tier_gaps(position_series):
    early_max = position_series[EARLY].max()
    intermediate_min = position_series[INTERMEDIATE].min()
    intermediate_max = position_series[INTERMEDIATE].max()
    late = position_series[LATE[0]]
    return pd.Series({'early_intermediate_gap': intermediate_min - early_max, 'intermediate_late_gap': late - intermediate_max})
gaps0 = tier_gaps(pos0)
gaps30 = tier_gaps(pos30)
gap_change = gaps30 - gaps0
print('\n' + '=' * 80)
print('TIER GAPS: FULL SAMPLE VS 30 DAYS')
print('=' * 80)
gap_table = pd.DataFrame({'full_sample': gaps0, 'day30': gaps30, 'change': gap_change})
display(gap_table.round(4))
rng = np.random.default_rng(RANDOM_SEED)
clusters = d_all[CLUSTER_COL].drop_duplicates().to_numpy()
n_clusters = len(clusters)
boot_relative_changes = []
boot_gap_changes = []
for b in tqdm(range(N_BOOT), desc='Narrative bootstrap'):
    sampled = rng.choice(clusters, size=n_clusters, replace=True)
    unique_ids, counts = np.unique(sampled, return_counts=True)
    weights = pd.Series(counts.astype(float), index=unique_ids)
    b0 = adjusted_positions(d0, cluster_weights=weights)
    b30 = adjusted_positions(d30, cluster_weights=weights)
    b0_rel = b0 - b0.mean()
    b30_rel = b30 - b30.mean()
    delta = b30_rel - b0_rel
    delta.name = b
    boot_relative_changes.append(delta)
    b_gap0 = tier_gaps(b0)
    b_gap30 = tier_gaps(b30)
    b_gap_delta = b_gap30 - b_gap0
    b_gap_delta.name = b
    boot_gap_changes.append(b_gap_delta)
boot_relative_changes = pd.DataFrame(boot_relative_changes)
boot_gap_changes = pd.DataFrame(boot_gap_changes)
equivalence_rows = []
for mech in MECH_ORDER:
    vals = boot_relative_changes[mech].dropna()
    ci_low = np.quantile(vals, 0.05)
    ci_high = np.quantile(vals, 0.95)
    row = {'mechanism': mech, 'observed_change': observed_change[mech], 'ci90_low': ci_low, 'ci90_high': ci_high}
    for margin in EQUIV_MARGINS:
        row[f'equivalent_margin_{margin:.2f}'] = ci_low > -margin and ci_high < margin
    equivalence_rows.append(row)
equivalence_results = pd.DataFrame(equivalence_rows)
print('\n' + '=' * 80)
print('PRACTICAL EQUIVALENCE OF RELATIVE TIMING')
print('=' * 80)
display(equivalence_results.round(4))
gap_boot_rows = []
for gap in ['early_intermediate_gap', 'intermediate_late_gap']:
    vals = boot_gap_changes[gap].dropna()
    gap_boot_rows.append({'gap': gap, 'observed_change': gap_change[gap], 'ci90_low': np.quantile(vals, 0.05), 'ci90_high': np.quantile(vals, 0.95), 'ci95_low': np.quantile(vals, 0.025), 'ci95_high': np.quantile(vals, 0.975)})
tier_gap_bootstrap = pd.DataFrame(gap_boot_rows)
print('\n' + '=' * 80)
print('CHANGE IN TIER SEPARATION: 0 DAYS VS 30 DAYS')
print('=' * 80)
display(tier_gap_bootstrap.round(4))
primary_col = f'equivalent_margin_{EQUIV_MARGIN:.2f}'
n_equivalent = int(equivalence_results[primary_col].sum())
print('\n' + '=' * 80)
print('COMPACT INTERPRETATION')
print('=' * 80)
print(f'Mechanisms equivalent within ±{EQUIV_MARGIN:.2f}: {n_equivalent}/{len(MECH_ORDER)}')
print('Max observed absolute relative shift:', f'{observed_change.abs().max():.4f}')
print('Mean observed absolute relative shift:', f'{observed_change.abs().mean():.4f}')
print('Tier structure preserved across cold-start thresholds:', '31/31')
try:
    interaction_results.to_csv(OUTPUT_DIR / 'cold_start_global_interaction_results.csv', index=False)
    equivalence_results.to_csv(OUTPUT_DIR / 'cold_start_relative_timing_equivalence.csv', index=False)
    tier_gap_bootstrap.to_csv(OUTPUT_DIR / 'cold_start_tier_gap_bootstrap.csv', index=False)
    print('\nSaved statistical robustness outputs.')
except NameError:
    print('\nOUTPUT_DIR not defined; results remain in memory.')


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
from IPython.display import display
param_names = X.columns.tolist()
beta = pd.Series(interaction_model.params, index=param_names)
cov = pd.DataFrame(interaction_model.cov_params(), index=param_names, columns=param_names)

def interaction_name(mech):
    if mech == 'Boundary construction':
        return None
    return f'mech_{mech}:prior_obs_30'

def test_pair(mech_a, mech_b):
    name_a = interaction_name(mech_a)
    name_b = interaction_name(mech_b)
    c = pd.Series(0.0, index=param_names)
    if name_a is not None:
        c[name_a] += 1.0
    if name_b is not None:
        c[name_b] -= 1.0
    estimate = float(c @ beta)
    variance = float(c @ cov @ c)
    se = np.sqrt(max(variance, 0))
    z = estimate / se
    p = 2 * (1 - norm.cdf(abs(z)))
    return {'mechanism_A': mech_a, 'mechanism_B': mech_b, 'difference_in_30d_shift': estimate, 'se': se, 'z': z, 'p_raw': p}
pairs = [('Negative evaluation', 'Boundary construction'), ('Negative evaluation', 'Action orientation'), ('Boundary construction', 'Action orientation'), ('Dehumanization', 'Threat construction')]
results = pd.DataFrame([test_pair(a, b) for a, b in pairs])
reject, p_holm, _, _ = multipletests(results['p_raw'], alpha=0.05, method='holm')
results['p_holm'] = p_holm
results['significant_holm'] = reject
print('=' * 80)
print('DIRECT TESTS OF WITHIN-TIER REORDERING')
print('=' * 80)
display(results.round(4))
print('\nSignificant after Holm correction:')
sig = results[results['significant_holm']]
if len(sig) == 0:
    print('None.')
else:
    display(sig.round(4))


In [ ]:
tier_sep = tier_stability.copy()
tier_sep['min_tier_gap'] = tier_sep[['early_intermediate_gap', 'intermediate_late_gap']].min(axis=1)
tier_sep['mean_tier_gap'] = tier_sep[['early_intermediate_gap', 'intermediate_late_gap']].mean(axis=1)
print('=' * 80)
print('TIER SEPARATION BY THRESHOLD')
print('=' * 80)
display(tier_sep[['window_days', 'early_intermediate_gap', 'intermediate_late_gap', 'min_tier_gap', 'mean_tier_gap']].round(4))
for w in [0, 1, 2, 7, 14, 21, 30]:
    r = tier_sep.loc[tier_sep['window_days'] == w].iloc[0]
    print(f"{w:>2} days | early→later={r['early_intermediate_gap']:.4f} | later→latest={r['intermediate_late_gap']:.4f} | weakest gap={r['min_tier_gap']:.4f} | mean gap={r['mean_tier_gap']:.4f}")
r1 = tier_sep.loc[tier_sep['window_days'] == 1].iloc[0]
r30 = tier_sep.loc[tier_sep['window_days'] == 30].iloc[0]
for metric in ['early_intermediate_gap', 'intermediate_late_gap', 'min_tier_gap', 'mean_tier_gap']:
    pct = (r30[metric] - r1[metric]) / r1[metric] * 100
    print(f'{metric}: {r1[metric]:.4f} -> {r30[metric]:.4f} ({pct:+.1f}%)')
print('\nStrongest weakest-boundary separation:')
display(tier_sep.sort_values('min_tier_gap', ascending=False).head(10)[['window_days', 'min_tier_gap', 'early_intermediate_gap', 'intermediate_late_gap']].round(4))


In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from IPython.display import display
BOOT_DF = bootstrap_temporal
BOOT_COL = 'bootstrap'
MECH_COL = 'mechanism'
VALUE_COL = 'adjusted_first'
MECH_ORDER = ['Negative evaluation', 'Boundary construction', 'Action orientation', 'Dehumanization', 'Threat construction', 'Scapegoating']
wide = BOOT_DF.pivot(index=BOOT_COL, columns=MECH_COL, values=VALUE_COL).reindex(columns=MECH_ORDER)
print('Bootstrap replicates:', len(wide))
rows = []
for A, B in combinations(MECH_ORDER, 2):
    valid = wide[[A, B]].dropna()
    delta = valid[B] - valid[A]
    mean_delta = delta.mean()
    median_delta = delta.median()
    ci_low, ci_high = np.quantile(delta, [0.025, 0.975])
    p_A_earlier = (valid[A] < valid[B]).mean()
    p_B_earlier = (valid[B] < valid[A]).mean()
    rows.append({'A': A, 'B': B, 'delta_B_minus_A': mean_delta, 'gap_pp': 100 * mean_delta, 'ci95_low_pp': 100 * ci_low, 'ci95_high_pp': 100 * ci_high, 'P_A_earlier': p_A_earlier, 'P_B_earlier': p_B_earlier, 'CI_excludes_zero': ci_low > 0 or ci_high < 0, 'n_boot': len(valid)})
pairwise = pd.DataFrame(rows)
print('=' * 90)
print('PAIRWISE TEMPORAL SEPARATION')
print('=' * 90)
display(pairwise[['A', 'B', 'gap_pp', 'ci95_low_pp', 'ci95_high_pp', 'P_A_earlier', 'CI_excludes_zero']].round(3))
EARLY = ['Negative evaluation', 'Boundary construction', 'Action orientation']
LATER = ['Dehumanization', 'Threat construction']
LATEST = ['Scapegoating']
between_tier_pairs = []
for A in EARLY:
    for B in LATER:
        between_tier_pairs.append((A, B))
for A in LATER:
    for B in LATEST:
        between_tier_pairs.append((A, B))
between = pairwise[pairwise.apply(lambda r: (r['A'], r['B']) in between_tier_pairs, axis=1)].copy()
print('\n' + '=' * 90)
print('BETWEEN-TIER COMPARISONS')
print('=' * 90)
display(between[['A', 'B', 'gap_pp', 'ci95_low_pp', 'ci95_high_pp', 'P_A_earlier', 'CI_excludes_zero']].round(3))
within_tier_pairs = list(combinations(EARLY, 2)) + list(combinations(LATER, 2))
within = pairwise[pairwise.apply(lambda r: (r['A'], r['B']) in within_tier_pairs, axis=1)].copy()
print('\n' + '=' * 90)
print('WITHIN-TIER COMPARISONS')
print('=' * 90)
display(within[['A', 'B', 'gap_pp', 'ci95_low_pp', 'ci95_high_pp', 'P_A_earlier', 'CI_excludes_zero']].round(3))
print('\n' + '=' * 90)
print('TIER SEPARATION SUMMARY')
print('=' * 90)
print('Between-tier comparisons whose 95% CI excludes zero:', f"{between['CI_excludes_zero'].sum()}/{len(between)}")
print('Lowest P(expected earlier mechanism is earlier), between tiers:', f"{between['P_A_earlier'].min():.3f}")
print('Mean P(expected earlier mechanism is earlier), between tiers:', f"{between['P_A_earlier'].mean():.3f}")
print('\nWithin-tier comparisons whose 95% CI excludes zero:', f"{within['CI_excludes_zero'].sum()}/{len(within)}")
print('Mean absolute within-tier gap:', f"{within['gap_pp'].abs().mean():.2f} pp")
print('Mean between-tier gap:', f"{between['gap_pp'].abs().mean():.2f} pp")


In [ ]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from IPython.display import display
OUT = OUTPUT_DIR
ranks = pd.read_csv(OUT / 'bootstrap_ranks_long.csv')
pairwise_tests = pd.read_csv(OUT / 'adjusted_temporal_pairwise.csv')
MECH_ORDER = ['Negative evaluation', 'Boundary construction', 'Action orientation', 'Dehumanization', 'Threat construction', 'Scapegoating']
EARLY = ['Negative evaluation', 'Boundary construction', 'Action orientation']
LATER = ['Dehumanization', 'Threat construction']
LATEST = ['Scapegoating']
wide = ranks.pivot(index='bootstrap', columns='label', values='rank').reindex(columns=MECH_ORDER)
print('Bootstrap replicates:', len(wide))
rows = []
for A, B in combinations(MECH_ORDER, 2):
    valid = wide[[A, B]].dropna()
    p_A_earlier = (valid[A] < valid[B]).mean()
    p_B_earlier = (valid[B] < valid[A]).mean()
    mean_rank_A = valid[A].mean()
    mean_rank_B = valid[B].mean()
    rows.append({'A': A, 'B': B, 'mean_rank_A': mean_rank_A, 'mean_rank_B': mean_rank_B, 'rank_gap_B_minus_A': mean_rank_B - mean_rank_A, 'P_A_earlier': p_A_earlier, 'A_earlier_percent': 100 * p_A_earlier, 'B_earlier_percent': 100 * p_B_earlier})
boot_pairwise = pd.DataFrame(rows)
formal = pairwise_tests.copy()
formal['pair_key'] = formal.apply(lambda r: tuple(sorted([r['A_label'], r['B_label']])), axis=1)
boot_pairwise['pair_key'] = boot_pairwise.apply(lambda r: tuple(sorted([r['A'], r['B']])), axis=1)
boot_pairwise = boot_pairwise.merge(formal[['pair_key', 'A_label', 'B_label', 'adjusted_diff_A_minus_B', 'p_value', 'fdr_q']], on='pair_key', how='left')

def tier(label):
    if label in EARLY:
        return 'Early'
    if label in LATER:
        return 'Later'
    if label in LATEST:
        return 'Latest'
    return 'Unknown'
boot_pairwise['tier_A'] = boot_pairwise['A'].map(tier)
boot_pairwise['tier_B'] = boot_pairwise['B'].map(tier)
boot_pairwise['comparison'] = np.where(boot_pairwise['tier_A'] == boot_pairwise['tier_B'], 'Within tier', 'Between tiers')
print('\n' + '=' * 100)
print('PAIRWISE BOOTSTRAP ORDERING')
print('=' * 100)
display(boot_pairwise[['A', 'B', 'A_earlier_percent', 'B_earlier_percent', 'mean_rank_A', 'mean_rank_B', 'rank_gap_B_minus_A', 'fdr_q', 'comparison']].round(3))
within = boot_pairwise[boot_pairwise['comparison'] == 'Within tier'].copy()
print('\n' + '=' * 100)
print('WITHIN-TIER ORDERING')
print('=' * 100)
display(within[['A', 'B', 'A_earlier_percent', 'B_earlier_percent', 'rank_gap_B_minus_A', 'fdr_q']].round(3))
expected_pairs = []
for A in EARLY:
    for B in LATER:
        expected_pairs.append((A, B))
for A in EARLY:
    for B in LATEST:
        expected_pairs.append((A, B))
for A in LATER:
    for B in LATEST:
        expected_pairs.append((A, B))
between_rows = []
for A, B in expected_pairs:
    row = boot_pairwise[(boot_pairwise['A'] == A) & (boot_pairwise['B'] == B) | (boot_pairwise['A'] == B) & (boot_pairwise['B'] == A)].iloc[0]
    if row['A'] == A:
        pct = row['A_earlier_percent']
    else:
        pct = row['B_earlier_percent']
    between_rows.append({'Earlier tier mechanism': A, 'Later tier mechanism': B, 'P_expected_order_percent': pct, 'fdr_q': row['fdr_q']})
between = pd.DataFrame(between_rows)
print('\n' + '=' * 100)
print('BETWEEN-TIER ORDERING')
print('=' * 100)
display(between.round(3))
print('\n' + '=' * 100)
print('TIER SUMMARY')
print('=' * 100)
print('Lowest bootstrap support for expected between-tier ordering:', f"{between['P_expected_order_percent'].min():.1f}%")
print('Mean bootstrap support for expected between-tier ordering:', f"{between['P_expected_order_percent'].mean():.1f}%")
print('Median bootstrap support for expected between-tier ordering:', f"{between['P_expected_order_percent'].median():.1f}%")
print('\nBetween-tier formal pairwise comparisons significant after FDR:', f"{(between['fdr_q'] < 0.05).sum()}/{len(between)}")
print('Within-tier formal pairwise comparisons significant after FDR:', f"{(within['fdr_q'] < 0.05).sum()}/{len(within)}")
matrix = pd.DataFrame(np.nan, index=MECH_ORDER, columns=MECH_ORDER)
for A in MECH_ORDER:
    matrix.loc[A, A] = 50.0
    for B in MECH_ORDER:
        if A == B:
            continue
        matrix.loc[A, B] = 100 * (wide[A] < wide[B]).mean()
print('\n' + '=' * 100)
print('% OF BOOTSTRAPS WHERE ROW MECHANISM APPEARS EARLIER')
print('=' * 100)
display(matrix.round(1))


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display
OUT = OUTPUT_DIR
boot = pd.read_csv(OUT / 'bootstrap_ranks_long.csv')
MECHS = ['Negative evaluation', 'Boundary construction', 'Action orientation', 'Dehumanization', 'Threat construction', 'Scapegoating']
EARLY = ['Negative evaluation', 'Boundary construction', 'Action orientation']
LATER = ['Dehumanization', 'Threat construction']
LATEST = ['Scapegoating']
boot_wide = boot.pivot(index='bootstrap', columns='label', values='rank').reindex(columns=MECHS)
print('Bootstrap replicates:', len(boot_wide))
cold = adjusted_cold_start.pivot(index='window_days', columns='mechanism', values='adjusted_first').reindex(columns=MECHS)
cold_ranks = cold.rank(axis=1, method='min', ascending=True)
baseline_rank = cold_ranks.loc[0]
print('\nBaseline adjusted ordering:')
print(' < '.join(baseline_rank.sort_values().index))
pair_support = pd.DataFrame(np.nan, index=MECHS, columns=MECHS)
for A in MECHS:
    for B in MECHS:
        if A == B:
            pair_support.loc[A, B] = 0.5
        else:
            pair_support.loc[A, B] = (boot_wide[A] < boot_wide[B]).mean()
rows = []
for window in cold_ranks.index:
    current_rank = cold_ranks.loc[window]
    rhos = []
    for _, boot_rank in boot_wide.iterrows():
        rho = spearmanr(current_rank.values, boot_rank.values).statistic
        rhos.append(rho)
    rhos = np.asarray(rhos)
    supports = []
    n_majority_agree = 0
    n_pairs = 0
    for i in range(len(MECHS)):
        for j in range(i + 1, len(MECHS)):
            A = MECHS[i]
            B = MECHS[j]
            n_pairs += 1
            if current_rank[A] < current_rank[B]:
                support = pair_support.loc[A, B]
            else:
                support = pair_support.loc[B, A]
            supports.append(support)
            if support > 0.5:
                n_majority_agree += 1
    mean_pair_support = np.mean(supports)
    early_before_later = all((current_rank[A] < current_rank[B] for A in EARLY for B in LATER))
    later_before_latest = all((current_rank[A] < current_rank[B] for A in LATER for B in LATEST))
    tier_compatible = early_before_later and later_before_latest
    ordering = ' < '.join(current_rank.sort_values().index)
    rows.append({'window_days': window, 'ordering': ordering, 'spearman_vs_baseline': spearmanr(current_rank.values, baseline_rank.values).statistic, 'mean_spearman_vs_bootstraps': np.mean(rhos), 'median_spearman_vs_bootstraps': np.median(rhos), 'p_bootstrap_rho_ge_0.9': np.mean(rhos >= 0.9), 'mean_bootstrap_pair_support': mean_pair_support, 'majority_pair_agreement': n_majority_agree / n_pairs, 'early_before_later': early_before_later, 'later_before_latest': later_before_latest, 'tier_compatible': tier_compatible})
comparison = pd.DataFrame(rows)
print('\n' + '=' * 100)
print('COLD-START ORDERING VS ORIGINAL FIGURE 4 BOOTSTRAPS')
print('=' * 100)
display(comparison[['window_days', 'spearman_vs_baseline', 'mean_spearman_vs_bootstraps', 'mean_bootstrap_pair_support', 'majority_pair_agreement', 'tier_compatible', 'ordering']].round(3))
print('\n' + '=' * 100)
print('SUMMARY')
print('=' * 100)
print('Cold-start thresholds compatible with original bootstrap tiers:', f"{comparison['tier_compatible'].sum()}/{len(comparison)}")
print('\nMinimum Spearman correlation with baseline ordering:', f"{comparison['spearman_vs_baseline'].min():.3f}")
print('Mean Spearman correlation with baseline ordering:', f"{comparison['spearman_vs_baseline'].mean():.3f}")
print('\nMinimum mean bootstrap pairwise support:', f"{100 * comparison['mean_bootstrap_pair_support'].min():.1f}%")
print('Mean bootstrap pairwise support:', f"{100 * comparison['mean_bootstrap_pair_support'].mean():.1f}%")
print('Maximum mean bootstrap pairwise support:', f"{100 * comparison['mean_bootstrap_pair_support'].max():.1f}%")
print('\nLowest fraction of pairwise relations agreeing with bootstrap majority:', f"{100 * comparison['majority_pair_agreement'].min():.1f}%")
rho_support, p_support = spearmanr(comparison['window_days'], comparison['mean_bootstrap_pair_support'])
rho_similarity, p_similarity = spearmanr(comparison['window_days'], comparison['mean_spearman_vs_bootstraps'])
print('\n' + '=' * 100)
print('DOES AGREEMENT WITH FIGURE 4 IMPROVE WITH STRINGENCY?')
print('=' * 100)
print('Threshold vs bootstrap pair-support:')
print(f'Spearman rho = {rho_support:.3f}, p = {p_support:.4g}')
print('\nThreshold vs mean bootstrap rank similarity:')
print(f'Spearman rho = {rho_similarity:.3f}, p = {p_similarity:.4g}')
print('\n' + '=' * 100)
print('MOST CONSISTENT WITH ORIGINAL BOOTSTRAP ORDERING')
print('=' * 100)
display(comparison.sort_values('mean_bootstrap_pair_support', ascending=False).head(10)[['window_days', 'mean_bootstrap_pair_support', 'mean_spearman_vs_bootstraps', 'ordering']].round(3))
print('\n' + '=' * 100)
print('LEAST CONSISTENT WITH ORIGINAL BOOTSTRAP ORDERING')
print('=' * 100)
display(comparison.sort_values('mean_bootstrap_pair_support', ascending=True).head(10)[['window_days', 'mean_bootstrap_pair_support', 'mean_spearman_vs_bootstraps', 'ordering']].round(3))


## Master Exports

In [ ]:
master_rows = []
for row in pair_weights_full.itertuples():
    pair = canon(row.A, row.B)
    descriptive = pairwise_association.loc[pairwise_association.apply(lambda r: canon(r['A'], r['B']) == pair, axis=1)].iloc[0]
    stability = structure_fold_stability.loc[(structure_fold_stability['A'] == pair[0]) & (structure_fold_stability['B'] == pair[1])].iloc[0]
    platform = platform_edge_replication.loc[(platform_edge_replication['A'] == pair[0]) & (platform_edge_replication['B'] == pair[1])].iloc[0]
    master_rows.append({'A': SHORT[pair[0]], 'B': SHORT[pair[1]], 'P_B_given_A': descriptive['P_B_given_A'], 'P_A_given_B': descriptive['P_A_given_B'], 'jaccard': descriptive['jaccard'], 'phi': descriptive['phi'], 'mean_pair_delta_loss': row.pair_weight, 'weight_share': row.weight_share, 'cumulative_weight_share': row.cumulative_share, 'in_95_backbone': pair in BACKBONE_PAIRS, 'backbone_fold_frequency': stability['backbone_fold_frequency'], 'in_consensus_bn': pair in BN_CONSENSUS_PAIRS, 'bn_fold_frequency': stability['bn_fold_frequency'], 'platforms_recovered': platform['n_platforms'], 'platform_fraction': platform['platform_fraction']})
master_edge_table = pd.DataFrame(master_rows)
print('=' * 88)
print('HELD-OUT STRUCTURE COMPARISON')
display(fair_structure_summary.round(6))
print('\nFRACTION OF SATURATED GAIN RECOVERED')
display(recovery_table.round(2))
print('\nTHEORY PERFORMANCE CONTROLLING FOR EDGE BUDGET')
display(theory_budget_summary.round(3))
print('\nEMPIRICAL SAME-SIZE PERFORMANCE — FOLD-WISE')
display(empirical_same_size_summary.round(2))
print('\nEMPIRICAL SAME-SIZE PERFORMANCE — POOLED DESCRIPTIVE')
display(pooled_empirical_budget_summary.round(3))
print('\nFINAL 95% BACKBONE')
display(master_edge_table.loc[master_edge_table['in_95_backbone']].round(4))
print('\nTEMPORAL NULL')
display(temporal_null_results.round(4))
print('\nADJUSTED TEMPORAL ORDER')
display(adjusted_timing.round(4))
print('\nTEMPORAL PHASES')
display(temporal_phase_table)
exports = {'prevalence.csv': prevalence.reset_index(names='mechanism'), 'pairwise_association.csv': pairwise_association, 'fair_structure_comparison_folds.csv': fair_structure_comparison, 'fair_structure_summary.csv': fair_structure_summary.reset_index(), 'fraction_empirical_gain_recovered.csv': recovery_table, 'theory_edge_budget_comparison.csv': theory_budget_summary, 'empirical_same_size_folds.csv': empirical_same_size_folds, 'empirical_same_size_summary.csv': empirical_same_size_summary.reset_index(), 'pooled_empirical_edge_budget_comparison.csv': pooled_empirical_budget_summary, 'structure_fold_stability.csv': structure_fold_stability, 'directed_conditional_edge_weights.csv': directed_edge_weights, 'conditional_effects.csv': conditional_effects, 'pair_weights_full.csv': pair_weights_full, 'master_edge_table.csv': master_edge_table, 'platform_edge_replication.csv': platform_edge_replication, 'temporal_first_appearance.csv': first_appearance, 'temporal_null_results.csv': temporal_null_results, 'adjusted_temporal_timing.csv': adjusted_timing, 'adjusted_temporal_pairwise.csv': timing_pairwise, 'temporal_phases.csv': temporal_phase_table}
for filename, df in exports.items():
    df.to_csv(OUTPUT_DIR / filename, index=False)
for k, df in budget_results.items():
    out = df.copy()
    out['edges'] = out['edges'].apply(lambda edges: ' | '.join((f'{SHORT[a]}--{SHORT[b]}' for a, b in sorted(edges))))
    out.to_csv(OUTPUT_DIR / f'all_{k}_edge_structures.csv', index=False)
print(f'\nExported all tables to: {OUTPUT_DIR}')


## Qwen Validation

In [ ]:
qwen_df = pd.read_csv(QWEN_PATH)
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, cohen_kappa_score
MECHANISMS = ['boundary_construction', 'threat_construction', 'scapegoating', 'negative_evaluation', 'dehumanization', 'action_orientation']
rows = []
for mech in MECHANISMS:
    s = qwen_df[(qwen_df['mechanism'] == mech) & (qwen_df['prediction_valid'] == True)].copy()
    y = s['final_human_label'].astype(int).to_numpy()
    pred = s['model_prediction'].astype(int).to_numpy()
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    rows.append({'mechanism': mech, 'n': len(s), 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn, 'precision': precision_score(y, pred), 'recall': recall_score(y, pred), 'f1': f1_score(y, pred), 'kappa': cohen_kappa_score(y, pred), 'sensitivity': sensitivity, 'specificity': specificity, 'fpr': 1 - specificity, 'fnr': 1 - sensitivity})
error_rates = pd.DataFrame(rows)
display(error_rates.round(3))
print('\nMacro averages:')
print(error_rates[['precision', 'recall', 'f1', 'kappa']].mean().round(3))


## Classifier-Error Sensitivity

In [ ]:
X_sens = stratified_structure_sample(X_bn, n=STRUCTURE_SAMPLE_N, random_state=RANDOM_STATE + 999)
print('Sensitivity sample:', f'{len(X_sens):,}')
baseline_directed = directed_unique_weights(X_sens, n_folds=EDGE_FULL_FOLDS, random_state=RANDOM_STATE)
baseline_weights = pair_weights_from_directed(baseline_directed)
baseline_backbone = select_backbone(baseline_weights, threshold=BACKBONE_CUM_WEIGHT)
print('\n95% backbone on fixed 300k sample:')
print('-' * 50)
for a, b in sorted(baseline_backbone):
    print(f'{SHORT[a]} -- {SHORT[b]}')
print(f'\nNumber of edges: {len(baseline_backbone)}')
display(baseline_weights[['A', 'B', 'pair_weight', 'weight_share', 'cumulative_share']].assign(A=lambda d: d['A'].map(SHORT), B=lambda d: d['B'].map(SHORT)).round(6))


In [ ]:
benchmark_replacement = error_rates.set_index('mechanism')['precision'].rsub(1).to_dict()
print('Qwen-informed positive-label replacement fractions:\n')
for mech, frac in benchmark_replacement.items():
    print(f'{mech:25s} {frac:.3f}')


In [ ]:
def perturb_mechanism_labels(X, replacement_fraction, rng):
    out = X.copy()
    for col in MECH_COLS:
        mech = col.replace('mech_', '')
        if isinstance(replacement_fraction, dict):
            frac = replacement_fraction[mech]
        else:
            frac = replacement_fraction
        y = out[col].to_numpy(dtype=np.int8, copy=True)
        pos_idx = np.flatnonzero(y == 1)
        neg_idx = np.flatnonzero(y == 0)
        n_swap = int(round(len(pos_idx) * frac))
        if n_swap == 0:
            continue
        remove_idx = rng.choice(pos_idx, size=n_swap, replace=False)
        add_idx = rng.choice(neg_idx, size=n_swap, replace=False)
        y[remove_idx] = 0
        y[add_idx] = 1
        out[col] = y
    return out


In [ ]:
from scipy.stats import spearmanr
N_SIMS = 100
CONDITIONS = {'10%': 0.1, '25%': 0.25, 'Qwen-informed': benchmark_replacement, '50%': 0.5}
baseline_pair_weight_map = {canon(r.A, r.B): r.pair_weight for r in baseline_weights.itertuples()}
results = []
edge_rows = []
condition_names = list(CONDITIONS.keys())
for condition_idx, (condition_name, replacement) in enumerate(CONDITIONS.items()):
    print('\n' + '=' * 70)
    print('CONDITION:', condition_name)
    print('=' * 70)
    for sim in tqdm(range(N_SIMS), desc=condition_name):
        rng = np.random.default_rng(RANDOM_STATE + 100000 + sim + 10000 * condition_idx)
        X_noisy = perturb_mechanism_labels(X_sens, replacement_fraction=replacement, rng=rng)
        directed = directed_unique_weights(X_noisy, n_folds=EDGE_INNER_FOLDS, random_state=RANDOM_STATE + sim)
        weights = pair_weights_from_directed(directed)
        backbone = select_backbone(weights, threshold=BACKBONE_CUM_WEIGHT)
        intersection = baseline_backbone & backbone
        union = baseline_backbone | backbone
        jaccard = len(intersection) / len(union) if union else np.nan
        noisy_map = {canon(r.A, r.B): r.pair_weight for r in weights.itertuples()}
        common_pairs = sorted(baseline_pair_weight_map)
        baseline_vals = np.array([baseline_pair_weight_map[p] for p in common_pairs])
        noisy_vals = np.array([noisy_map[p] for p in common_pairs])
        rho = spearmanr(baseline_vals, noisy_vals).statistic
        boundary_threat = canon('mech_boundary_construction', 'mech_threat_construction')
        threat_action = canon('mech_threat_construction', 'mech_action_orientation')
        results.append({'condition': condition_name, 'simulation': sim, 'n_backbone_edges': len(backbone), 'original_edges_recovered': len(intersection), 'jaccard': jaccard, 'spearman_edge_weights': rho, 'boundary_threat_retained': int(boundary_threat in backbone), 'threat_action_retained': int(threat_action in backbone)})
        for pair in ALL_PAIRS:
            edge_rows.append({'condition': condition_name, 'simulation': sim, 'A': pair[0], 'B': pair[1], 'A_label': SHORT[pair[0]], 'B_label': SHORT[pair[1]], 'in_backbone': int(pair in backbone), 'pair_weight': noisy_map[pair], 'original_backbone': int(pair in baseline_backbone)})
sensitivity_results = pd.DataFrame(results)
sensitivity_edges = pd.DataFrame(edge_rows)


In [ ]:
summary = sensitivity_results.groupby('condition').agg(mean_backbone_edges=('n_backbone_edges', 'mean'), mean_original_edges_recovered=('original_edges_recovered', 'mean'), pct_recover_at_least_6=('original_edges_recovered', lambda x: 100 * (x >= 6).mean()), pct_recover_all_7=('original_edges_recovered', lambda x: 100 * (x == 7).mean()), mean_jaccard=('jaccard', 'mean'), mean_spearman=('spearman_edge_weights', 'mean'), boundary_threat_retained_pct=('boundary_threat_retained', lambda x: 100 * x.mean()), threat_action_retained_pct=('threat_action_retained', lambda x: 100 * x.mean()))
display(summary.round(3))


In [ ]:
edge_stability = sensitivity_edges[sensitivity_edges['original_backbone'] == 1].groupby(['condition', 'A_label', 'B_label'], as_index=False).agg(retention_rate=('in_backbone', 'mean'), mean_pair_weight=('pair_weight', 'mean'))
edge_stability['retention_pct'] = 100 * edge_stability['retention_rate']
display(edge_stability.sort_values(['condition', 'retention_rate'], ascending=[True, False]).round(3))
